# v3 on Kaggle T4: YOLOv8n vision (DCGM settled by the 2026-10-10 run)

Items 10 and 11 of `docs/PLAN.md` (`industrial-llm-platform`). One-click **Run all** on **GPU T4 x2** with internet on.
Pushed with `kaggle kernels push -p notebooks/kaggle-dcgm-vision -t 3600` (1 h hard cap); the notebook's own budget
(`BUDGET_S`) skips steps that would not finish, leaving time for the summary. Estimate: about 0.4 h (re-export, then mAP).

**Rerun scope (`RUN_SPEED = False`, `RUN_TRITON = False`: mAP only).** Triton succeeded on 2026-10-10 in
`benchmark/results/raw/kaggle-20261010-v3e/`; mAP failed there because Kaggle exports `MPLBACKEND` set to the
notebook's matplotlib_inline backend, which the vision venv lacks (now `MPLBACKEND=Agg`). The ONNX Runtime CUDA and TensorRT speed cells succeeded on 2026-10-10
(`benchmark/results/raw/kaggle-20261010-v3c/`); this session re-exports the models (Triton and mAP need them), then
runs only Triton and mAP, which failed there: mAP because the data YAML had no `train:` key (Ultralytics 8.4.174
requires it), Triton because the Python backend stub loaded Kaggle's Ubuntu libpython3.12 instead of the uv-managed
Python's (`No module named '_contextvars'`). Both fixes were reproduced and verified locally on CPU.

**A. DCGM: not run again.** The 2026-10-10 session (notebook built at 19ccae8, output in
`benchmark/results/raw/kaggle-20261010-v3a/results/dcgm/`) settled it: the basic fields work in embedded and
standalone mode, and every profiling field fails with `DCGMError_RequiresRoot: This operation is not supported when
the host engine is running as non root`. Kaggle gives no root over the host engine, so DCGM profiling (DRAM active,
SM active) cannot run there. This notebook writes `NOT_RUN-dcgm.txt` and `NOT_RUN-vllm-t4-1.5b-fp16-dcgm.txt`
pointing to that evidence and spends the session on vision. The text below on how the probe worked is kept for the
record.

How the 2026-10-10 probe worked: NVIDIA DCGM 4.7.0-2 is extracted from its pinned .deb files (no apt, no Docker; the
dcgm-exporter container cannot run here). Tested: the Python bindings in embedded mode, a standalone `nv-hostengine`,
`dcgmi discovery/profile/dmon`, each with the basic field group (clocks, clock event reasons, temperature, power,
utilisation, framebuffer) and the profiling group (GR engine, SM active, SM occupancy, tensor pipe, DRAM active, FP16
pipe). All output is kept verbatim in `results/dcgm/`. Each profiling field is then probed alone, because one
unsupported field (FP16 pipe may be one on Turing) can fail a whole watch. Only if SM active or DRAM active returns
values: vLLM 1.5B FP16 at concurrency 1 and 8 is re-run sampling the basic fields plus the profiling fields that
worked (cell `vllm-t4-1.5b-fp16-dcgm`) to see whether batch-1 decode is memory-bound (DRAM active). The probe accepts
idle zeros; `results/dcgm/rerun-check.json` then records whether SM/DRAM active were non-zero under load. Otherwise
`NOT_RUN-vllm-t4-1.5b-fp16-dcgm.txt` holds the reason.

NVIDIA proprietary but free software is used here: the DCGM proprietary package (profiling module) and TensorRT.

**B. Vision (estimate 45 min).** YOLOv8n (Ultralytics 8.4.174, **AGPL-3.0**; weights sha256-pinned) exported to ONNX
FP32 (opset 17) and TensorRT FP16 engines (one per batch size). Images/s and per-batch latency p50/p95/p99 at batch
1, 8, 32 for ONNX Runtime CUDA (FP32) and TensorRT FP16 in process, then TensorRT FP16 behind Triton via PyTriton
(Triton's Python backend; the NGC image needs Docker). mAP50-95 on the first 500 val2017 images (sha256 manifest)
for PyTorch FP32 (reference), ONNX FP32 and TensorRT FP16.

T4 is SM75: FP16 only (no bf16, no FP8). Preflight stops at once without GPU or internet (`FAILED-preflight.txt`).


In [ ]:
# Pinned configuration. Everything else is in the embedded lock files.
GIT_COMMIT = "693b44b4446c00948085c81325bbeb24b5897776"  # repo commit the embedded code came from
UV_VERSION = "0.11.32"
PYTHON_VERSION = "3.12.13"
VLLM_VERSION = "0.31.0"
MODELS = {
    "1.5b-fp16": (
        "Qwen/Qwen2.5-1.5B-Instruct",
        "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
    )
}
REPEATS = 3
MAX_MODEL_LEN = 4096
GPU_MEM_UTIL = 0.90
GSM8K_LIMIT = 250
PORT = 8000
WORK = "/kaggle/working"
RESULTS = f"{WORK}/results"
BENCH = f"{WORK}/bench"
VENVS = "/tmp/venvs"
BUDGET_S = 3000  # push -t 3600; leaves 10 min for the summary
DCGM_EVIDENCE = "benchmark/results/raw/kaggle-20261010-v3a/results/dcgm/ (notebook built at 19ccae8)"
YOLO_URL = "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt"
YOLO_SHA256 = "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36"
VISION_BATCHES = (1, 8, 32)
# ORT CUDA and TensorRT speed cells succeeded in kaggle-20261010-v3c; the rerun only needs Triton and mAP.
RUN_SPEED = False
SPEED_EVIDENCE = (
    "benchmark/results/raw/kaggle-20261010-v3c/ (notebook built at 80905ea)"
)
# Triton succeeded in kaggle-20261010-v3e; this rerun is mAP only.
RUN_TRITON = False
TRITON_EVIDENCE = (
    "benchmark/results/raw/kaggle-20261010-v3e/ (notebook built at fb5ea24)"
)
DATA = "/tmp/data/coco500"
MODELS_DIR = f"{RESULTS}/models"  # ONNX and engines, kept in the output for traceability (git-ignored)
# MPLBACKEND=Agg: Kaggle exports the notebook's matplotlib_inline backend, which the vision venv does not have.
VISION_ENV = "YOLO_OFFLINE=1 YOLO_AUTOINSTALL=False CUDA_VISIBLE_DEVICES=0 MPLBACKEND=Agg"  # no Ultralytics downloads or pip installs
PINS = {
    "vllm_version": VLLM_VERSION,
    "ultralytics": "8.4.174",
    "tensorrt": "10.16.1.11",
    "onnxruntime_gpu": "1.26.0",
    "nvidia_pytriton": "0.7.0",
    "onnx_opset": 17,
}
PREFLIGHT_URLS = (
    "https://pypi.org/simple/",
    "https://pypi.nvidia.com/tensorrt-cu12/",
    "https://github.com",
    "https://huggingface.co",
    "http://images.cocodataset.org/val2017/000000000139.jpg",
)

In [ ]:
# Embedded copies of the repo files (generated by notebooks/build_notebook.py). sha256 checked on write.
import hashlib
import json
import os
import pathlib

FILES = json.loads(
    '{"benchmark/loadgen/run.py": ["9c1bd8b99b72a49a57c86f9ff3739215d27db837744711a4b70f5b1bd256e04f", "\\"\\"\\"Closed-loop streaming load generator for any OpenAI-compatible chat endpoint (llama.cpp, vLLM).\\n\\nWrites one raw JSON per (concurrency, repeat) with full config, environment, per-request records and summary.\\nExample:\\n  python benchmark/loadgen/run.py --base-url http://127.0.0.1:8080/v1 --model qwen2.5-1.5b-instruct-q4_k_m \\\\\\n    --engine llama.cpp --engine-version b11459 --precision gguf-q4_k_m --label llamacpp-cpu-1.5b-q4km \\\\\\n    --concurrency 1 4 8 --repeats 3 --out-dir benchmark/results/raw\\n\\"\\"\\"\\n\\nimport argparse\\nimport asyncio\\nimport datetime as dt\\nimport hashlib\\nimport json\\nimport os\\nimport platform\\nimport shutil\\nimport subprocess\\nimport sys\\nimport tempfile\\nimport time\\nfrom pathlib import Path\\n\\nimport httpx\\n\\nsys.path.insert(0, str(Path(__file__).parent))\\nfrom metrics import decode_checks, summarize\\n\\nHERE = Path(__file__).parent\\n\\n\\ndef utcnow():\\n    return dt.datetime.now(dt.UTC).isoformat(timespec=\\"seconds\\")\\n\\n\\ndef sh(*cmd):\\n    try:\\n        return subprocess.run(\\n            cmd, capture_output=True, text=True, timeout=30, check=False\\n        ).stdout.strip()\\n    except (OSError, subprocess.SubprocessError):\\n        return None\\n\\n\\ndef cpu_model():\\n    if Path(\\"/proc/cpuinfo\\").exists():\\n        for line in Path(\\"/proc/cpuinfo\\").read_text().splitlines():\\n            if line.startswith(\\"model name\\"):\\n                return line.split(\\":\\", 1)[1].strip()\\n    if sys.platform == \\"win32\\":\\n        import winreg\\n\\n        key = winreg.OpenKey(\\n            winreg.HKEY_LOCAL_MACHINE, r\\"HARDWARE\\\\DESCRIPTION\\\\System\\\\CentralProcessor\\\\0\\"\\n        )\\n        return winreg.QueryValueEx(key, \\"ProcessorNameString\\")[0].strip()\\n    return platform.processor()\\n\\n\\ndef git_commit():\\n    commit = sh(\\"git\\", \\"-C\\", str(HERE), \\"rev-parse\\", \\"HEAD\\")\\n    if not commit:\\n        return None\\n    return commit + (\\n        \\"-dirty\\" if sh(\\"git\\", \\"-C\\", str(HERE), \\"status\\", \\"--porcelain\\") else \\"\\"\\n    )\\n\\n\\nasync def one_request(client, args, prompt, warmup, t0):\\n    body = {\\n        \\"model\\": args.model,\\n        \\"messages\\": [{\\"role\\": \\"user\\", \\"content\\": prompt[\\"prompt\\"]}],\\n        \\"max_tokens\\": args.max_tokens,\\n        \\"temperature\\": 0,\\n        \\"seed\\": args.seed,\\n        \\"stream\\": True,\\n        \\"stream_options\\": {\\"include_usage\\": True},\\n    }\\n    if args.ignore_eos:\\n        body[\\"ignore_eos\\"] = (\\n            True  # llama.cpp and vLLM both accept it; fixes output length\\n        )\\n    rec = {\\n        \\"prompt_id\\": prompt[\\"id\\"],\\n        \\"kind\\": prompt[\\"kind\\"],\\n        \\"warmup\\": warmup,\\n        \\"token_times\\": [],\\n        \\"error\\": None,\\n    }\\n    chunks, usage = 0, None\\n    rec[\\"start\\"] = round(time.perf_counter() - t0, 6)\\n    try:\\n        async with client.stream(\\n            \\"POST\\", f\\"{args.base_url}/chat/completions\\", json=body\\n        ) as resp:\\n            if resp.status_code != 200:\\n                rec[\\"error\\"] = (\\n                    f\\"HTTP {resp.status_code}: {(await resp.aread()).decode(errors=\'replace\')[:300]}\\"\\n                )\\n            else:\\n                async for line in resp.aiter_lines():\\n                    if not line.startswith(\\"data: \\"):\\n                        continue\\n                    data = line[6:].strip()\\n                    if data == \\"[DONE]\\":\\n                        break\\n                    msg = json.loads(data)\\n                    if msg.get(\\"usage\\"):\\n                        usage = msg[\\"usage\\"]\\n                    for ch in msg.get(\\"choices\\") or []:\\n                        if (ch.get(\\"delta\\") or {}).get(\\"content\\"):\\n                            rec[\\"token_times\\"].append(\\n                                round(time.perf_counter() - t0, 6)\\n                            )\\n                            chunks += 1\\n    except Exception as e:  # noqa: BLE001 recorded, never raised: a failed request is data\\n        rec[\\"error\\"] = f\\"{type(e).__name__}: {e}\\"\\n    rec[\\"end\\"] = round(time.perf_counter() - t0, 6)\\n    rec[\\"chunks\\"] = chunks\\n    rec[\\"prompt_tokens\\"] = (usage or {}).get(\\"prompt_tokens\\")\\n    rec[\\"output_tokens\\"] = (usage or {}).get(\\"completion_tokens\\", chunks)\\n    rec[\\"token_source\\"] = (\\n        \\"usage\\" if usage and \\"completion_tokens\\" in usage else \\"chunks\\"\\n    )\\n    if not rec[\\"error\\"] and chunks == 0:\\n        rec[\\"error\\"] = \\"no tokens streamed\\"\\n    return rec\\n\\n\\nasync def closed_loop(client, args, prompts, n, concurrency, warmup, t0):\\n    \\"\\"\\"concurrency workers, each sends its next request as soon as its previous one ends.\\"\\"\\"\\n    it = iter(range(n))\\n    out = []\\n\\n    async def worker():\\n        for i in it:\\n            out.append(\\n                await one_request(client, args, prompts[i % len(prompts)], warmup, t0)\\n            )\\n\\n    await asyncio.gather(*(worker() for _ in range(concurrency)))\\n    return out\\n\\n\\nclass GpuSampler:\\n    \\"\\"\\"nvidia-smi every 500 ms in the background, written to a file (no pipe to drain); no-op without nvidia-smi.\\"\\"\\"\\n\\n    # clocks_event_reasons.active: bitmask (0x4 SW power cap, 0x20 SW thermal, 0x40 HW thermal); needs driver >= 535.\\n    QUERY = (\\n        \\"timestamp,index,memory.used,memory.total,utilization.gpu,clocks.sm,temperature.gpu,power.draw,\\"\\n        \\"clocks.mem,clocks_event_reasons.active\\"\\n    )\\n    KEYS = (\\n        \\"timestamp\\",\\n        \\"gpu\\",\\n        \\"memory_used_mib\\",\\n        \\"memory_total_mib\\",\\n        \\"util_gpu_pct\\",\\n        \\"clock_sm_mhz\\",\\n        \\"temp_c\\",\\n        \\"power_w\\",\\n        \\"clock_mem_mhz\\",\\n        \\"clocks_event_reasons\\",\\n    )\\n\\n    def __init__(self):\\n        self.proc = None\\n        if shutil.which(\\"nvidia-smi\\"):\\n            fd, self.path = tempfile.mkstemp(suffix=\\".csv\\")\\n            os.close(fd)\\n            ids = os.environ.get(\\n                \\"CUDA_VISIBLE_DEVICES\\"\\n            )  # sample only the GPUs the server uses\\n            cmd = [\\n                \\"nvidia-smi\\",\\n                f\\"--query-gpu={self.QUERY}\\",\\n                \\"--format=csv,noheader,nounits\\",\\n                \\"-lms\\",\\n                \\"500\\",\\n            ]\\n            cmd += [\\"-i\\", ids] if ids else []\\n            self.proc = subprocess.Popen(\\n                [*cmd, \\"-f\\", self.path],\\n                stdout=subprocess.DEVNULL,\\n                stderr=subprocess.DEVNULL,\\n            )\\n\\n    def stop(self):\\n        if not self.proc:\\n            return None\\n        self.proc.terminate()\\n        try:\\n            self.proc.wait(timeout=10)\\n        except subprocess.TimeoutExpired:\\n            self.proc.kill()\\n        rows = Path(self.path).read_text().splitlines()\\n        Path(self.path).unlink()\\n        return [\\n            dict(zip(self.KEYS, [c.strip() for c in r.split(\\",\\")], strict=False))\\n            for r in rows\\n            if r.strip()\\n        ]\\n\\n\\nclass DcgmSampler:\\n    \\"\\"\\"dcgm_sampler.py in the background (--dcgm-fields), same lifecycle as GpuSampler; no-op without fields.\\n    GPU ids are taken from CUDA_VISIBLE_DEVICES, assuming CUDA and DCGM number the GPUs the same way.\\"\\"\\"\\n\\n    def __init__(self, fields, host):\\n        self.proc = None\\n        if not fields:\\n            return\\n        fd, self.path = tempfile.mkstemp(suffix=\\".csv\\")\\n        os.close(fd)\\n        fd, self.err = tempfile.mkstemp(suffix=\\".err\\")\\n        os.close(fd)\\n        ids = os.environ.get(\\"CUDA_VISIBLE_DEVICES\\")\\n        cmd = [\\n            sys.executable,\\n            str(HERE / \\"dcgm_sampler.py\\"),\\n            \\"--fields\\",\\n            fields,\\n            \\"--host\\",\\n            host,\\n        ]\\n        cmd += [\\"--out\\", self.path] + ([\\"--gpus\\", ids] if ids else [])\\n        with open(self.err, \\"w\\") as err:\\n            self.proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=err)\\n\\n    def stop(self):\\n        \\"\\"\\"(samples, error text or None); (None, None) when off.\\"\\"\\"\\n        if not self.proc:\\n            return None, None\\n        self.proc.terminate()\\n        try:\\n            self.proc.wait(timeout=10)\\n        except subprocess.TimeoutExpired:\\n            self.proc.kill()\\n        lines = Path(self.path).read_text().splitlines()\\n        keys = lines[0].split(\\",\\") if lines else []  # header written by dcgm_sampler.py\\n        rows = lines[1:]\\n        err = Path(self.err).read_text().strip() or None\\n        Path(self.path).unlink()\\n        Path(self.err).unlink()\\n        return [\\n            dict(zip(keys, r.split(\\",\\"), strict=False)) for r in rows if r.strip()\\n        ], err\\n\\n\\nasync def docker_stats():\\n    if not shutil.which(\\"docker\\"):\\n        return None\\n    try:\\n        p = await asyncio.create_subprocess_exec(\\n            \\"docker\\",\\n            \\"stats\\",\\n            \\"--no-stream\\",\\n            \\"--format\\",\\n            \\"{{json .}}\\",\\n            stdout=asyncio.subprocess.PIPE,\\n            stderr=asyncio.subprocess.DEVNULL,\\n        )\\n        out, _ = await asyncio.wait_for(p.communicate(), 30)\\n        return [json.loads(line) for line in out.decode().splitlines() if line.strip()]\\n    except (TimeoutError, OSError, json.JSONDecodeError):\\n        return None\\n\\n\\nasync def scrape(client, url):\\n    if not url:\\n        return None\\n    try:\\n        r = await client.get(url)\\n        return [\\n            line for line in r.text.splitlines() if line and not line.startswith(\\"#\\")\\n        ]\\n    except httpx.HTTPError as e:\\n        return f\\"scrape failed: {e}\\"\\n\\n\\nasync def run_cell(args, prompts, concurrency, repeat, env):\\n    n = args.requests or max(64, 4 * concurrency)\\n    n_warm = args.warmup if args.warmup is not None else max(2, concurrency)\\n    limits = httpx.Limits(\\n        max_connections=concurrency + 4, max_keepalive_connections=concurrency + 4\\n    )\\n    async with httpx.AsyncClient(\\n        timeout=httpx.Timeout(args.timeout), limits=limits\\n    ) as client:\\n        t0 = time.perf_counter()\\n        started = utcnow()\\n        # Warmup uses prompts from the end of the list so they do not prime the first measured prompts.\\n        warm = await closed_loop(\\n            client, args, prompts[::-1], n_warm, concurrency, True, t0\\n        )\\n        metrics_before = await scrape(client, args.metrics_url)\\n        gpu = GpuSampler()\\n        dcgm = DcgmSampler(args.dcgm_fields, args.dcgm_host)\\n        stats_task = asyncio.create_task(docker_stats())\\n        measured = await closed_loop(client, args, prompts, n, concurrency, False, t0)\\n        stats = await stats_task\\n        gpu_samples = gpu.stop()\\n        dcgm_samples, dcgm_error = dcgm.stop()\\n        metrics_after = await scrape(client, args.metrics_url)\\n    records = warm + measured\\n    return {\\n        \\"schema\\": 1,\\n        \\"smoke\\": args.smoke,\\n        \\"label\\": args.label,\\n        \\"engine\\": args.engine,\\n        \\"engine_version\\": args.engine_version,\\n        \\"model\\": args.model,\\n        \\"model_revision\\": args.model_revision,\\n        \\"precision\\": args.precision,\\n        \\"concurrency\\": concurrency,\\n        \\"repeat\\": repeat,\\n        \\"started_utc\\": started,\\n        \\"finished_utc\\": utcnow(),\\n        \\"config\\": {\\n            \\"base_url\\": args.base_url,\\n            \\"requests\\": n,\\n            \\"warmup_requests\\": n_warm,\\n            \\"max_tokens\\": args.max_tokens,\\n            \\"temperature\\": 0,\\n            \\"seed\\": args.seed,\\n            \\"ignore_eos\\": args.ignore_eos,\\n            \\"prompts_file\\": str(Path(args.prompts).name),\\n            \\"prompts_count\\": len(prompts),\\n            \\"prompts_sha256\\": args.prompts_sha256,\\n            \\"timeout_s\\": args.timeout,\\n            \\"notes\\": args.notes,\\n        },\\n        \\"env\\": env,\\n        \\"server_args\\": args.server_args,\\n        \\"gpu_samples\\": gpu_samples,\\n        \\"dcgm_samples\\": dcgm_samples,\\n        \\"dcgm_error\\": dcgm_error,\\n        \\"docker_stats\\": stats,\\n        \\"server_metrics_before\\": metrics_before,\\n        \\"server_metrics_after\\": metrics_after,\\n        \\"summary\\": summarize(records)\\n        | {\\"decode_checks\\": decode_checks(records, args.max_tokens, args.ignore_eos)},\\n        \\"requests\\": records,\\n    }\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\n        \\"--base-url\\",\\n        required=True,\\n        help=\\"OpenAI API root, e.g. http://127.0.0.1:8080/v1\\",\\n    )\\n    p.add_argument(\\"--model\\", required=True)\\n    p.add_argument(\\"--model-revision\\", default=None, help=\\"HF commit or file sha256\\")\\n    p.add_argument(\\"--engine\\", required=True)\\n    p.add_argument(\\"--engine-version\\", required=True)\\n    p.add_argument(\\"--precision\\", required=True)\\n    p.add_argument(\\"--label\\", required=True, help=\\"cell name prefix for output files\\")\\n    p.add_argument(\\n        \\"--hardware\\",\\n        default=None,\\n        help=\\"free-text hardware label, e.g. \'1x T4 (Kaggle)\'\\",\\n    )\\n    p.add_argument(\\"--concurrency\\", type=int, nargs=\\"+\\", default=[1])\\n    p.add_argument(\\"--repeats\\", type=int, default=3)\\n    p.add_argument(\\n        \\"--requests\\",\\n        type=int,\\n        default=0,\\n        help=\\"measured requests per run (0 = max(64, 4*concurrency))\\",\\n    )\\n    p.add_argument(\\n        \\"--warmup\\",\\n        type=int,\\n        default=None,\\n        help=\\"warmup requests per run (default max(2, concurrency))\\",\\n    )\\n    p.add_argument(\\"--max-tokens\\", type=int, default=128)\\n    p.add_argument(\\"--seed\\", type=int, default=1234)\\n    p.add_argument(\\"--no-ignore-eos\\", dest=\\"ignore_eos\\", action=\\"store_false\\")\\n    p.add_argument(\\"--timeout\\", type=float, default=600)\\n    p.add_argument(\\"--prompts\\", default=str(HERE / \\"prompts.jsonl\\"))\\n    p.add_argument(\\n        \\"--metrics-url\\",\\n        default=None,\\n        help=\\"Prometheus /metrics to snapshot before and after\\",\\n    )\\n    p.add_argument(\\n        \\"--server-args\\", default=None, help=\\"server command line, stored verbatim\\"\\n    )\\n    p.add_argument(\\n        \\"--git-commit\\", default=None, help=\\"override when not run from a git checkout\\"\\n    )\\n    p.add_argument(\\n        \\"--dcgm-fields\\",\\n        default=None,\\n        help=\\"comma-separated DCGM field ids to sample during the measured window (dcgm_sampler.py)\\",\\n    )\\n    p.add_argument(\\n        \\"--dcgm-host\\", default=\\"127.0.0.1\\", help=\\"nv-hostengine address, or \'embedded\'\\"\\n    )\\n    p.add_argument(\\"--notes\\", default=None)\\n    p.add_argument(\\n        \\"--smoke\\",\\n        action=\\"store_true\\",\\n        help=\\"mark as smoke run (excluded from RESULTS.md)\\",\\n    )\\n    p.add_argument(\\"--out-dir\\", required=True)\\n    args = p.parse_args()\\n\\n    raw_prompts = Path(args.prompts).read_bytes()\\n    args.prompts_sha256 = hashlib.sha256(raw_prompts).hexdigest()\\n    prompts = [\\n        json.loads(line)\\n        for line in raw_prompts.decode(\\"utf-8\\").splitlines()\\n        if line.strip()\\n    ]\\n    env = {\\n        \\"hardware\\": args.hardware,\\n        \\"cpu_model\\": cpu_model(),\\n        \\"cpu_count\\": os.cpu_count(),\\n        \\"platform\\": platform.platform(),\\n        \\"python\\": platform.python_version(),\\n        \\"httpx\\": httpx.__version__,\\n        \\"git_commit\\": args.git_commit or git_commit(),\\n        \\"nvidia_smi\\": sh(\\n            \\"nvidia-smi\\",\\n            \\"--query-gpu=name,driver_version,memory.total\\",\\n            \\"--format=csv,noheader\\",\\n        )\\n        if shutil.which(\\"nvidia-smi\\")\\n        else None,\\n    }\\n    with httpx.Client(timeout=30) as c:\\n        env[\\"server_models\\"] = c.get(f\\"{args.base_url}/models\\").json()\\n    out_dir = Path(args.out_dir)\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    for conc in args.concurrency:\\n        for rep in range(1, args.repeats + 1):\\n            res = asyncio.run(run_cell(args, prompts, conc, rep, env))\\n            stamp = (\\n                res[\\"started_utc\\"]\\n                .replace(\\":\\", \\"\\")\\n                .replace(\\"-\\", \\"\\")\\n                .replace(\\"+0000\\", \\"Z\\")\\n            )\\n            path = out_dir / f\\"{args.label}-c{conc}-r{rep}-{stamp}.json\\"\\n            path.write_text(\\n                json.dumps(res, separators=(\\",\\", \\":\\")) + \\"\\\\n\\",\\n                encoding=\\"utf-8\\",\\n                newline=\\"\\\\n\\",  # LF on Windows too\\n            )  # compact raw data\\n            s = res[\\"summary\\"]\\n            print(\\n                f\\"{path.name}: ok {s[\'ok\']}/{s[\'requests\']} out_tok/s {s[\'output_tok_s\']} \\"\\n                f\\"ttft_p50 {s[\'ttft_s\'][\'p50\']} e2e_p95 {s[\'e2e_s\'][\'p95\']}\\",\\n                flush=True,\\n            )\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"], "benchmark/loadgen/metrics.py": ["c58866fb6430059795c5cbeedc5a36ea0798c0c16b05d8161c160d43192fcc35", "\\"\\"\\"Pure metric functions. Times are seconds. Shared by run.py, report.py and the tests.\\"\\"\\"\\n\\nimport itertools\\nimport math\\nimport statistics\\n\\n\\ndef percentile(values, q):\\n    \\"\\"\\"Linear interpolation between closest ranks (same as numpy\'s default). None for empty input.\\"\\"\\"\\n    xs = sorted(values)\\n    if not xs:\\n        return None\\n    k = (len(xs) - 1) * q / 100\\n    lo, hi = math.floor(k), math.ceil(k)\\n    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)\\n\\n\\ndef dist(values):\\n    \\"\\"\\"mean, p50, p95, p99 of a list (all None if empty).\\"\\"\\"\\n    return {\\n        \\"mean\\": statistics.fmean(values) if values else None,\\n        \\"p50\\": percentile(values, 50),\\n        \\"p95\\": percentile(values, 95),\\n        \\"p99\\": percentile(values, 99),\\n    }\\n\\n\\ndef request_metrics(r):\\n    \\"\\"\\"TTFT, inter-token gaps, TPOT and e2e for one request record.\\n\\n    r has start, end, token_times (arrival time of each streamed content chunk) and output_tokens.\\n    ITL is the gap between consecutive chunks. TPOT is (last token - first token) / (output_tokens - 1),\\n    which stays correct when a chunk carries more than one token.\\n    \\"\\"\\"\\n    tt = r[\\"token_times\\"]\\n    ttft = tt[0] - r[\\"start\\"] if tt else None\\n    itl = [b - a for a, b in itertools.pairwise(tt)]\\n    n = r[\\"output_tokens\\"] or 0\\n    tpot = (tt[-1] - tt[0]) / (n - 1) if tt and n > 1 else None\\n    e2e = r[\\"end\\"] - r[\\"start\\"]\\n    return {\\n        \\"ttft\\": ttft,\\n        \\"itl\\": itl,\\n        \\"tpot\\": tpot,\\n        \\"e2e\\": e2e,\\n        \\"tok_s\\": n / e2e if e2e > 0 else None,\\n    }\\n\\n\\ndef summarize(records):\\n    \\"\\"\\"Summary over measured (non-warmup) requests.\\"\\"\\"\\n    measured = [r for r in records if not r[\\"warmup\\"]]\\n    ok = [r for r in measured if not r[\\"error\\"]]\\n    per = [request_metrics(r) for r in ok]\\n\\n    def pick(k):\\n        return [m[k] for m in per if m[k] is not None]\\n\\n    wall = (\\n        (max(r[\\"end\\"] for r in measured) - min(r[\\"start\\"] for r in measured))\\n        if measured\\n        else 0\\n    )\\n    out_tokens = sum(r[\\"output_tokens\\"] or 0 for r in ok)\\n    warm = [\\n        request_metrics(r)[\\"ttft\\"]\\n        for r in records\\n        if r[\\"warmup\\"] and not r[\\"error\\"] and r[\\"token_times\\"]\\n    ]\\n    return {\\n        \\"requests\\": len(measured),\\n        \\"ok\\": len(ok),\\n        \\"errors\\": len(measured) - len(ok),\\n        \\"error_rate\\": (len(measured) - len(ok)) / len(measured) if measured else None,\\n        \\"wall_s\\": wall,\\n        \\"output_tokens\\": out_tokens,\\n        \\"output_tok_s\\": out_tokens / wall if wall > 0 else None,\\n        \\"request_s\\": len(ok) / wall if wall > 0 else None,\\n        \\"ttft_s\\": dist(pick(\\"ttft\\")),\\n        \\"itl_s\\": dist([g for m in per for g in m[\\"itl\\"]]),\\n        \\"tpot_s\\": dist(pick(\\"tpot\\")),\\n        \\"e2e_s\\": dist(pick(\\"e2e\\")),\\n        \\"per_request_tok_s\\": dist(pick(\\"tok_s\\")),\\n        \\"warmup_ttft_mean_s\\": statistics.fmean(warm) if warm else None,\\n    }\\n\\n\\nTAIL_GAP_S = 0.25  # last streamed chunk this long before the response ended ...\\nTAIL_GAP_ITL = 10  # ... and this many times the request\'s median chunk gap = the stream went quiet early\\n\\n\\ndef stream_truncated(r):\\n    \\"\\"\\"True if the server stopped streaming text before the request ended: fewer chunks than output tokens and a\\n    long silent tail. Such a request\'s TPOT and ITL undercount (its tokens arrived in no streamed chunk).\\"\\"\\"\\n    tt, n = r[\\"token_times\\"], r.get(\\"output_tokens\\") or 0\\n    if not tt or len(tt) >= n:\\n        return False\\n    gaps = sorted(b - a for a, b in itertools.pairwise(tt))\\n    median = gaps[len(gaps) // 2] if gaps else 0\\n    tail = r[\\"end\\"] - tt[-1]\\n    return tail > TAIL_GAP_S and tail > TAIL_GAP_ITL * median\\n\\n\\ndef decode_checks(records, max_tokens, ignore_eos):\\n    \\"\\"\\"Over measured successful requests: how many decoded a token count other than max_tokens (only checked with\\n    ignore_eos, where every request must decode exactly max_tokens), and how many streams were truncated.\\"\\"\\"\\n    ok = [r for r in records if not r[\\"warmup\\"] and not r[\\"error\\"]]\\n    return {\\n        \\"checked\\": len(ok),\\n        \\"output_tokens_not_max\\": sum(\\n            1 for r in ok if r.get(\\"output_tokens\\") != max_tokens\\n        )\\n        if ignore_eos\\n        else None,\\n        \\"streams_truncated\\": sum(1 for r in ok if stream_truncated(r)),\\n    }\\n\\n\\ndef cv(values):\\n    \\"\\"\\"Coefficient of variation (stdev / mean) across repeats; None if under 2 values or mean 0.\\"\\"\\"\\n    xs = [v for v in values if v is not None]\\n    if len(xs) < 2 or statistics.fmean(xs) == 0:\\n        return None\\n    return statistics.stdev(xs) / abs(statistics.fmean(xs))\\n"], "benchmark/loadgen/prompts.jsonl": ["72fb01ab947799b7fcc757f90652501b203bff4bd445d3654d36aff01913ea7e", "{\\"id\\": \\"l06\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 6:\\\\nDay 1, shift 2: conveyor CV-3, vibration rose to 9.1 mm/s.\\\\nDay 2, shift 1: fan F-12, alignment checked, offset 0.55 mm.\\\\nDay 3, shift 3: pump P-101, alignment checked, offset 0.38 mm.\\\\nDay 4, shift 1: boiler B-2, vibration rose to 12.0 mm/s.\\\\nDay 5, shift 3: boiler B-2, bearing temperature reached 76 C.\\\\nDay 6, shift 3: mixer M-5, alignment checked, offset 0.4 mm.\\\\nDay 7, shift 2: compressor C-7, bearing temperature reached 90 C.\\\\nDay 8, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 9, shift 1: pump P-101, motor current 9 A above baseline.\\\\nDay 10, shift 2: pump P-101, bearing temperature reached 78 C.\\\\nDay 11, shift 2: conveyor CV-3, vibration rose to 2.6 mm/s.\\\\nDay 12, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 13, shift 2: compressor C-7, alignment checked, offset 0.57 mm.\\\\nDay 14, shift 3: mixer M-5, motor current 5 A above baseline.\\\\nDay 15, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 16, shift 3: mixer M-5, bearing temperature reached 91 C.\\\\nDay 17, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 3: fan F-12, motor current 4 A above baseline.\\\\nDay 19, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 20, shift 3: press PR-9, vibration rose to 2.7 mm/s.\\\\nDay 21, shift 3: fan F-12, motor current 15 A above baseline.\\\\nDay 22, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 23, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 24, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 25, shift 3: compressor C-7, bearing temperature reached 64 C.\\\\nDay 26, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 27, shift 3: fan F-12, motor current 15 A above baseline.\\\\nDay 28, shift 1: mixer M-5, seal leak observed, 17 drops per minute.\\\\nDay 29, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 30, shift 3: chiller CH-4, vibration rose to 11.8 mm/s.\\\\nDay 31, shift 2: conveyor CV-3, operator reported unusual noise.\\\\nDay 32, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 33, shift 2: boiler B-2, vibration rose to 6.7 mm/s.\\\\nDay 34, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 35, shift 2: chiller CH-4, seal leak observed, 3 drops per minute.\\\\nDay 36, shift 2: conveyor CV-3, alignment checked, offset 0.21 mm.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"s09\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is vibration analysis used for?\\"}\\n{\\"id\\": \\"s33\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a spare parts criticality ranking?\\"}\\n{\\"id\\": \\"l13\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 13:\\\\nDay 1, shift 1: compressor C-7, oil level checked and topped up.\\\\nDay 2, shift 3: boiler B-2, alignment checked, offset 0.12 mm.\\\\nDay 3, shift 3: chiller CH-4, vibration rose to 11.6 mm/s.\\\\nDay 4, shift 1: compressor C-7, oil level checked and topped up.\\\\nDay 5, shift 1: compressor C-7, alignment checked, offset 0.08 mm.\\\\nDay 6, shift 3: pump P-101, alignment checked, offset 0.3 mm.\\\\nDay 7, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 8, shift 3: fan F-12, vibration rose to 8.9 mm/s.\\\\nDay 9, shift 3: chiller CH-4, bearing temperature reached 82 C.\\\\nDay 10, shift 3: pump P-101, seal leak observed, 2 drops per minute.\\\\nDay 11, shift 3: press PR-9, motor current 8 A above baseline.\\\\nDay 12, shift 1: boiler B-2, bearing temperature reached 60 C.\\\\nDay 13, shift 2: press PR-9, filter replaced during scheduled stop.\\\\nDay 14, shift 3: chiller CH-4, bearing temperature reached 79 C.\\\\nDay 15, shift 3: chiller CH-4, motor current 7 A above baseline.\\\\nDay 16, shift 1: boiler B-2, operator reported unusual noise.\\\\nDay 17, shift 1: mixer M-5, alignment checked, offset 0.28 mm.\\\\nDay 18, shift 3: press PR-9, bearing temperature reached 80 C.\\\\nDay 19, shift 1: press PR-9, vibration rose to 7.6 mm/s.\\\\nDay 20, shift 3: press PR-9, seal leak observed, 1 drops per minute.\\\\nDay 21, shift 3: compressor C-7, alignment checked, offset 0.46 mm.\\\\nDay 22, shift 2: pump P-101, bearing temperature reached 91 C.\\\\nDay 23, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 24, shift 3: boiler B-2, motor current 2 A above baseline.\\\\nDay 25, shift 1: chiller CH-4, alignment checked, offset 0.1 mm.\\\\nDay 26, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 27, shift 1: conveyor CV-3, motor current 6 A above baseline.\\\\nDay 28, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 29, shift 3: boiler B-2, alignment checked, offset 0.1 mm.\\\\nDay 30, shift 3: boiler B-2, vibration rose to 11.3 mm/s.\\\\nDay 31, shift 3: boiler B-2, operator reported unusual noise.\\\\nDay 32, shift 1: conveyor CV-3, vibration rose to 6.7 mm/s.\\\\nDay 33, shift 1: compressor C-7, alignment checked, offset 0.5 mm.\\\\nDay 34, shift 2: compressor C-7, motor current 10 A above baseline.\\\\nDay 35, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 36, shift 1: compressor C-7, vibration rose to 6.2 mm/s.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"s06\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"List three causes of motor overheating.\\"}\\n{\\"id\\": \\"s41\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is oil analysis used for?\\"}\\n{\\"id\\": \\"s03\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe the difference between preventive and predictive maintenance.\\"}\\n{\\"id\\": \\"s08\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain lockout tagout in one paragraph.\\"}\\n{\\"id\\": \\"s35\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a thermocouple?\\"}\\n{\\"id\\": \\"l12\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 12:\\\\nDay 1, shift 2: boiler B-2, operator reported unusual noise.\\\\nDay 2, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 3, shift 2: boiler B-2, oil level checked and topped up.\\\\nDay 4, shift 2: conveyor CV-3, bearing temperature reached 62 C.\\\\nDay 5, shift 1: mixer M-5, operator reported unusual noise.\\\\nDay 6, shift 1: conveyor CV-3, bearing temperature reached 92 C.\\\\nDay 7, shift 1: compressor C-7, vibration rose to 6.7 mm/s.\\\\nDay 8, shift 3: pump P-101, motor current 10 A above baseline.\\\\nDay 9, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 10, shift 1: fan F-12, bearing temperature reached 87 C.\\\\nDay 11, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 12, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 13, shift 1: conveyor CV-3, vibration rose to 9.7 mm/s.\\\\nDay 14, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 15, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 16, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 17, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 18, shift 2: compressor C-7, seal leak observed, 7 drops per minute.\\\\nDay 19, shift 1: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 20, shift 2: press PR-9, seal leak observed, 3 drops per minute.\\\\nDay 21, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 22, shift 1: conveyor CV-3, vibration rose to 11.0 mm/s.\\\\nDay 23, shift 3: chiller CH-4, operator reported unusual noise.\\\\nDay 24, shift 2: pump P-101, filter replaced during scheduled stop.\\\\nDay 25, shift 2: press PR-9, motor current 12 A above baseline.\\\\nDay 26, shift 3: chiller CH-4, alignment checked, offset 0.37 mm.\\\\nDay 27, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 28, shift 2: boiler B-2, bearing temperature reached 63 C.\\\\nDay 29, shift 3: boiler B-2, bearing temperature reached 92 C.\\\\nDay 30, shift 3: fan F-12, filter replaced during scheduled stop.\\\\nDay 31, shift 3: boiler B-2, oil level checked and topped up.\\\\nDay 32, shift 1: compressor C-7, operator reported unusual noise.\\\\nDay 33, shift 3: fan F-12, motor current 2 A above baseline.\\\\nDay 34, shift 3: press PR-9, oil level checked and topped up.\\\\nDay 35, shift 3: conveyor CV-3, alignment checked, offset 0.57 mm.\\\\nDay 36, shift 1: conveyor CV-3, motor current 6 A above baseline.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"l08\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 8:\\\\nDay 1, shift 2: pump P-101, motor current 4 A above baseline.\\\\nDay 2, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 3, shift 1: chiller CH-4, motor current 5 A above baseline.\\\\nDay 4, shift 3: fan F-12, operator reported unusual noise.\\\\nDay 5, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 6, shift 3: press PR-9, vibration rose to 3.1 mm/s.\\\\nDay 7, shift 2: conveyor CV-3, operator reported unusual noise.\\\\nDay 8, shift 2: boiler B-2, bearing temperature reached 74 C.\\\\nDay 9, shift 2: conveyor CV-3, vibration rose to 5.8 mm/s.\\\\nDay 10, shift 1: mixer M-5, motor current 12 A above baseline.\\\\nDay 11, shift 3: boiler B-2, motor current 13 A above baseline.\\\\nDay 12, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 13, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 14, shift 1: mixer M-5, oil level checked and topped up.\\\\nDay 15, shift 1: fan F-12, bearing temperature reached 81 C.\\\\nDay 16, shift 3: chiller CH-4, vibration rose to 7.4 mm/s.\\\\nDay 17, shift 3: boiler B-2, vibration rose to 11.1 mm/s.\\\\nDay 18, shift 1: conveyor CV-3, vibration rose to 9.7 mm/s.\\\\nDay 19, shift 2: mixer M-5, motor current 13 A above baseline.\\\\nDay 20, shift 3: fan F-12, alignment checked, offset 0.37 mm.\\\\nDay 21, shift 3: compressor C-7, bearing temperature reached 89 C.\\\\nDay 22, shift 3: mixer M-5, motor current 10 A above baseline.\\\\nDay 23, shift 3: mixer M-5, alignment checked, offset 0.56 mm.\\\\nDay 24, shift 2: compressor C-7, bearing temperature reached 73 C.\\\\nDay 25, shift 1: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 26, shift 2: press PR-9, bearing temperature reached 93 C.\\\\nDay 27, shift 1: chiller CH-4, bearing temperature reached 89 C.\\\\nDay 28, shift 1: press PR-9, seal leak observed, 4 drops per minute.\\\\nDay 29, shift 2: conveyor CV-3, bearing temperature reached 79 C.\\\\nDay 30, shift 2: chiller CH-4, motor current 6 A above baseline.\\\\nDay 31, shift 3: mixer M-5, operator reported unusual noise.\\\\nDay 32, shift 3: press PR-9, seal leak observed, 7 drops per minute.\\\\nDay 33, shift 3: fan F-12, vibration rose to 2.2 mm/s.\\\\nDay 34, shift 2: chiller CH-4, motor current 3 A above baseline.\\\\nDay 35, shift 2: conveyor CV-3, operator reported unusual noise.\\\\nDay 36, shift 1: chiller CH-4, bearing temperature reached 69 C.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"s00\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the purpose of a pressure relief valve?\\"}\\n{\\"id\\": \\"s40\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe what a borescope inspection is.\\"}\\n{\\"id\\": \\"l01\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 1:\\\\nDay 1, shift 3: pump P-101, motor current 4 A above baseline.\\\\nDay 2, shift 3: conveyor CV-3, bearing temperature reached 70 C.\\\\nDay 3, shift 2: mixer M-5, motor current 11 A above baseline.\\\\nDay 4, shift 1: pump P-101, motor current 14 A above baseline.\\\\nDay 5, shift 2: compressor C-7, motor current 7 A above baseline.\\\\nDay 6, shift 2: press PR-9, vibration rose to 2.3 mm/s.\\\\nDay 7, shift 3: conveyor CV-3, alignment checked, offset 0.07 mm.\\\\nDay 8, shift 1: fan F-12, motor current 12 A above baseline.\\\\nDay 9, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 10, shift 2: press PR-9, motor current 6 A above baseline.\\\\nDay 11, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 12, shift 2: pump P-101, oil level checked and topped up.\\\\nDay 13, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 14, shift 2: mixer M-5, oil level checked and topped up.\\\\nDay 15, shift 2: mixer M-5, alignment checked, offset 0.31 mm.\\\\nDay 16, shift 1: fan F-12, bearing temperature reached 88 C.\\\\nDay 17, shift 2: compressor C-7, alignment checked, offset 0.49 mm.\\\\nDay 18, shift 3: boiler B-2, bearing temperature reached 73 C.\\\\nDay 19, shift 1: boiler B-2, alignment checked, offset 0.56 mm.\\\\nDay 20, shift 2: press PR-9, bearing temperature reached 66 C.\\\\nDay 21, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 22, shift 3: pump P-101, filter replaced during scheduled stop.\\\\nDay 23, shift 2: conveyor CV-3, seal leak observed, 20 drops per minute.\\\\nDay 24, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 25, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 26, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 27, shift 3: boiler B-2, motor current 8 A above baseline.\\\\nDay 28, shift 3: mixer M-5, bearing temperature reached 59 C.\\\\nDay 29, shift 2: mixer M-5, alignment checked, offset 0.51 mm.\\\\nDay 30, shift 1: press PR-9, bearing temperature reached 56 C.\\\\nDay 31, shift 2: conveyor CV-3, seal leak observed, 30 drops per minute.\\\\nDay 32, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 33, shift 2: conveyor CV-3, bearing temperature reached 69 C.\\\\nDay 34, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 35, shift 2: boiler B-2, motor current 8 A above baseline.\\\\nDay 36, shift 2: boiler B-2, oil level checked and topped up.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"l02\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 2:\\\\nDay 1, shift 3: boiler B-2, vibration rose to 6.1 mm/s.\\\\nDay 2, shift 3: boiler B-2, alignment checked, offset 0.21 mm.\\\\nDay 3, shift 2: compressor C-7, alignment checked, offset 0.17 mm.\\\\nDay 4, shift 3: mixer M-5, bearing temperature reached 66 C.\\\\nDay 5, shift 1: fan F-12, operator reported unusual noise.\\\\nDay 6, shift 2: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 7, shift 1: pump P-101, vibration rose to 6.1 mm/s.\\\\nDay 8, shift 2: boiler B-2, alignment checked, offset 0.3 mm.\\\\nDay 9, shift 2: conveyor CV-3, seal leak observed, 29 drops per minute.\\\\nDay 10, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 11, shift 1: press PR-9, vibration rose to 6.2 mm/s.\\\\nDay 12, shift 3: compressor C-7, seal leak observed, 23 drops per minute.\\\\nDay 13, shift 2: mixer M-5, alignment checked, offset 0.55 mm.\\\\nDay 14, shift 1: boiler B-2, bearing temperature reached 75 C.\\\\nDay 15, shift 3: fan F-12, alignment checked, offset 0.38 mm.\\\\nDay 16, shift 2: pump P-101, motor current 5 A above baseline.\\\\nDay 17, shift 3: conveyor CV-3, vibration rose to 4.7 mm/s.\\\\nDay 18, shift 1: pump P-101, motor current 13 A above baseline.\\\\nDay 19, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 20, shift 1: fan F-12, seal leak observed, 15 drops per minute.\\\\nDay 21, shift 1: fan F-12, motor current 13 A above baseline.\\\\nDay 22, shift 1: press PR-9, motor current 3 A above baseline.\\\\nDay 23, shift 2: pump P-101, motor current 11 A above baseline.\\\\nDay 24, shift 3: fan F-12, vibration rose to 6.2 mm/s.\\\\nDay 25, shift 2: boiler B-2, seal leak observed, 16 drops per minute.\\\\nDay 26, shift 3: press PR-9, seal leak observed, 5 drops per minute.\\\\nDay 27, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 28, shift 3: press PR-9, vibration rose to 4.6 mm/s.\\\\nDay 29, shift 2: pump P-101, vibration rose to 4.0 mm/s.\\\\nDay 30, shift 2: chiller CH-4, motor current 8 A above baseline.\\\\nDay 31, shift 2: pump P-101, motor current 3 A above baseline.\\\\nDay 32, shift 3: mixer M-5, filter replaced during scheduled stop.\\\\nDay 33, shift 1: conveyor CV-3, alignment checked, offset 0.57 mm.\\\\nDay 34, shift 1: pump P-101, seal leak observed, 27 drops per minute.\\\\nDay 35, shift 2: boiler B-2, seal leak observed, 30 drops per minute.\\\\nDay 36, shift 1: boiler B-2, motor current 12 A above baseline.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"l11\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 11:\\\\nDay 1, shift 1: fan F-12, motor current 8 A above baseline.\\\\nDay 2, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 3, shift 3: pump P-101, motor current 3 A above baseline.\\\\nDay 4, shift 2: compressor C-7, vibration rose to 6.1 mm/s.\\\\nDay 5, shift 1: compressor C-7, bearing temperature reached 84 C.\\\\nDay 6, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 7, shift 3: mixer M-5, seal leak observed, 12 drops per minute.\\\\nDay 8, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 9, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 10, shift 2: mixer M-5, motor current 7 A above baseline.\\\\nDay 11, shift 2: conveyor CV-3, motor current 11 A above baseline.\\\\nDay 12, shift 1: compressor C-7, motor current 6 A above baseline.\\\\nDay 13, shift 2: mixer M-5, bearing temperature reached 86 C.\\\\nDay 14, shift 1: press PR-9, alignment checked, offset 0.31 mm.\\\\nDay 15, shift 2: pump P-101, motor current 4 A above baseline.\\\\nDay 16, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 17, shift 2: boiler B-2, vibration rose to 5.9 mm/s.\\\\nDay 18, shift 3: compressor C-7, oil level checked and topped up.\\\\nDay 19, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 20, shift 3: pump P-101, bearing temperature reached 70 C.\\\\nDay 21, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 22, shift 2: chiller CH-4, alignment checked, offset 0.09 mm.\\\\nDay 23, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 24, shift 3: chiller CH-4, motor current 7 A above baseline.\\\\nDay 25, shift 2: press PR-9, vibration rose to 9.1 mm/s.\\\\nDay 26, shift 3: compressor C-7, motor current 15 A above baseline.\\\\nDay 27, shift 1: fan F-12, alignment checked, offset 0.5 mm.\\\\nDay 28, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 29, shift 1: mixer M-5, bearing temperature reached 92 C.\\\\nDay 30, shift 1: fan F-12, seal leak observed, 28 drops per minute.\\\\nDay 31, shift 2: mixer M-5, bearing temperature reached 87 C.\\\\nDay 32, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 33, shift 2: fan F-12, bearing temperature reached 56 C.\\\\nDay 34, shift 2: chiller CH-4, motor current 11 A above baseline.\\\\nDay 35, shift 1: press PR-9, bearing temperature reached 66 C.\\\\nDay 36, shift 3: press PR-9, alignment checked, offset 0.11 mm.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"s44\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what a Gantt chart shows in a shutdown plan.\\"}\\n{\\"id\\": \\"s29\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the difference between accuracy and precision of a sensor?\\"}\\n{\\"id\\": \\"s30\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why do pipes experience water hammer?\\"}\\n{\\"id\\": \\"s07\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the function of a heat exchanger?\\"}\\n{\\"id\\": \\"s38\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what shaft misalignment causes.\\"}\\n{\\"id\\": \\"s19\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a failure mode and effects analysis?\\"}\\n{\\"id\\": \\"l03\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 3:\\\\nDay 1, shift 1: chiller CH-4, operator reported unusual noise.\\\\nDay 2, shift 1: press PR-9, alignment checked, offset 0.22 mm.\\\\nDay 3, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 4, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 5, shift 1: press PR-9, seal leak observed, 28 drops per minute.\\\\nDay 6, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 7, shift 2: compressor C-7, oil level checked and topped up.\\\\nDay 8, shift 2: chiller CH-4, motor current 7 A above baseline.\\\\nDay 9, shift 3: mixer M-5, motor current 9 A above baseline.\\\\nDay 10, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 11, shift 2: fan F-12, motor current 8 A above baseline.\\\\nDay 12, shift 1: mixer M-5, bearing temperature reached 63 C.\\\\nDay 13, shift 3: mixer M-5, vibration rose to 6.1 mm/s.\\\\nDay 14, shift 3: compressor C-7, bearing temperature reached 78 C.\\\\nDay 15, shift 3: pump P-101, vibration rose to 8.8 mm/s.\\\\nDay 16, shift 2: compressor C-7, seal leak observed, 8 drops per minute.\\\\nDay 17, shift 2: compressor C-7, bearing temperature reached 73 C.\\\\nDay 18, shift 1: press PR-9, vibration rose to 6.6 mm/s.\\\\nDay 19, shift 1: boiler B-2, filter replaced during scheduled stop.\\\\nDay 20, shift 2: chiller CH-4, seal leak observed, 2 drops per minute.\\\\nDay 21, shift 1: mixer M-5, operator reported unusual noise.\\\\nDay 22, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 23, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 24, shift 1: compressor C-7, bearing temperature reached 57 C.\\\\nDay 25, shift 1: fan F-12, motor current 10 A above baseline.\\\\nDay 26, shift 3: press PR-9, alignment checked, offset 0.4 mm.\\\\nDay 27, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 28, shift 2: fan F-12, bearing temperature reached 61 C.\\\\nDay 29, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 30, shift 1: fan F-12, alignment checked, offset 0.19 mm.\\\\nDay 31, shift 1: compressor C-7, bearing temperature reached 80 C.\\\\nDay 32, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 33, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 34, shift 2: mixer M-5, vibration rose to 4.0 mm/s.\\\\nDay 35, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 36, shift 2: fan F-12, alignment checked, offset 0.42 mm.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"s22\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How is corrosion prevented in steel pipes?\\"}\\n{\\"id\\": \\"s14\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain the difference between AC and DC motors.\\"}\\n{\\"id\\": \\"s21\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what a safety integrity level is.\\"}\\n{\\"id\\": \\"s12\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Define overall equipment effectiveness.\\"}\\n{\\"id\\": \\"l14\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 14:\\\\nDay 1, shift 1: fan F-12, alignment checked, offset 0.22 mm.\\\\nDay 2, shift 1: boiler B-2, vibration rose to 8.7 mm/s.\\\\nDay 3, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 4, shift 3: conveyor CV-3, oil level checked and topped up.\\\\nDay 5, shift 2: boiler B-2, vibration rose to 9.5 mm/s.\\\\nDay 6, shift 3: press PR-9, bearing temperature reached 81 C.\\\\nDay 7, shift 2: chiller CH-4, vibration rose to 9.3 mm/s.\\\\nDay 8, shift 3: pump P-101, filter replaced during scheduled stop.\\\\nDay 9, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 10, shift 2: fan F-12, seal leak observed, 8 drops per minute.\\\\nDay 11, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 12, shift 3: chiller CH-4, bearing temperature reached 83 C.\\\\nDay 13, shift 2: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 14, shift 3: mixer M-5, motor current 15 A above baseline.\\\\nDay 15, shift 2: fan F-12, vibration rose to 11.6 mm/s.\\\\nDay 16, shift 2: press PR-9, bearing temperature reached 81 C.\\\\nDay 17, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 19, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 20, shift 3: boiler B-2, bearing temperature reached 70 C.\\\\nDay 21, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 22, shift 1: press PR-9, seal leak observed, 6 drops per minute.\\\\nDay 23, shift 1: conveyor CV-3, seal leak observed, 21 drops per minute.\\\\nDay 24, shift 2: pump P-101, motor current 5 A above baseline.\\\\nDay 25, shift 2: fan F-12, operator reported unusual noise.\\\\nDay 26, shift 3: chiller CH-4, operator reported unusual noise.\\\\nDay 27, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 28, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 29, shift 2: compressor C-7, alignment checked, offset 0.47 mm.\\\\nDay 30, shift 1: press PR-9, filter replaced during scheduled stop.\\\\nDay 31, shift 3: chiller CH-4, motor current 4 A above baseline.\\\\nDay 32, shift 1: conveyor CV-3, bearing temperature reached 92 C.\\\\nDay 33, shift 3: compressor C-7, oil level checked and topped up.\\\\nDay 34, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 35, shift 2: press PR-9, bearing temperature reached 61 C.\\\\nDay 36, shift 3: fan F-12, operator reported unusual noise.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"s15\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a root cause analysis?\\"}\\n{\\"id\\": \\"s32\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe what a calibration certificate contains.\\"}\\n{\\"id\\": \\"s36\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How does an ultrasonic flow meter work?\\"}\\n{\\"id\\": \\"s45\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why is grounding important in electrical panels?\\"}\\n{\\"id\\": \\"s16\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why is lubrication important for gearboxes?\\"}\\n{\\"id\\": \\"l00\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 0:\\\\nDay 1, shift 1: press PR-9, bearing temperature reached 92 C.\\\\nDay 2, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 3, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 4, shift 3: compressor C-7, alignment checked, offset 0.08 mm.\\\\nDay 5, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 6, shift 2: press PR-9, vibration rose to 11.4 mm/s.\\\\nDay 7, shift 1: mixer M-5, operator reported unusual noise.\\\\nDay 8, shift 2: conveyor CV-3, vibration rose to 11.4 mm/s.\\\\nDay 9, shift 1: pump P-101, seal leak observed, 9 drops per minute.\\\\nDay 10, shift 2: press PR-9, vibration rose to 7.6 mm/s.\\\\nDay 11, shift 1: press PR-9, vibration rose to 9.8 mm/s.\\\\nDay 12, shift 1: press PR-9, alignment checked, offset 0.28 mm.\\\\nDay 13, shift 3: compressor C-7, alignment checked, offset 0.24 mm.\\\\nDay 14, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 15, shift 1: compressor C-7, seal leak observed, 2 drops per minute.\\\\nDay 16, shift 3: fan F-12, filter replaced during scheduled stop.\\\\nDay 17, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 1: boiler B-2, motor current 10 A above baseline.\\\\nDay 19, shift 3: boiler B-2, filter replaced during scheduled stop.\\\\nDay 20, shift 3: press PR-9, motor current 11 A above baseline.\\\\nDay 21, shift 1: compressor C-7, motor current 12 A above baseline.\\\\nDay 22, shift 2: boiler B-2, vibration rose to 6.3 mm/s.\\\\nDay 23, shift 1: chiller CH-4, seal leak observed, 5 drops per minute.\\\\nDay 24, shift 2: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 25, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 26, shift 2: boiler B-2, alignment checked, offset 0.31 mm.\\\\nDay 27, shift 2: mixer M-5, alignment checked, offset 0.2 mm.\\\\nDay 28, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 29, shift 2: pump P-101, alignment checked, offset 0.58 mm.\\\\nDay 30, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 31, shift 3: chiller CH-4, motor current 11 A above baseline.\\\\nDay 32, shift 2: chiller CH-4, bearing temperature reached 93 C.\\\\nDay 33, shift 1: press PR-9, filter replaced during scheduled stop.\\\\nDay 34, shift 1: chiller CH-4, motor current 8 A above baseline.\\\\nDay 35, shift 1: boiler B-2, operator reported unusual noise.\\\\nDay 36, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"s25\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a condition monitoring sensor?\\"}\\n{\\"id\\": \\"s47\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain the purpose of a mean time to repair metric.\\"}\\n{\\"id\\": \\"s34\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain the bathtub curve of failure rates.\\"}\\n{\\"id\\": \\"s04\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a PID controller?\\"}\\n{\\"id\\": \\"s37\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a confined space permit?\\"}\\n{\\"id\\": \\"s27\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a Pareto chart used for in quality control?\\"}\\n{\\"id\\": \\"s39\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the purpose of an emergency stop circuit?\\"}\\n{\\"id\\": \\"s20\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What does a torque wrench do?\\"}\\n{\\"id\\": \\"l10\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 10:\\\\nDay 1, shift 2: fan F-12, oil level checked and topped up.\\\\nDay 2, shift 3: press PR-9, motor current 11 A above baseline.\\\\nDay 3, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 4, shift 2: conveyor CV-3, motor current 6 A above baseline.\\\\nDay 5, shift 3: press PR-9, motor current 12 A above baseline.\\\\nDay 6, shift 2: mixer M-5, seal leak observed, 25 drops per minute.\\\\nDay 7, shift 3: fan F-12, bearing temperature reached 86 C.\\\\nDay 8, shift 1: chiller CH-4, seal leak observed, 19 drops per minute.\\\\nDay 9, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 10, shift 1: mixer M-5, alignment checked, offset 0.35 mm.\\\\nDay 11, shift 2: compressor C-7, motor current 4 A above baseline.\\\\nDay 12, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 13, shift 3: compressor C-7, vibration rose to 7.8 mm/s.\\\\nDay 14, shift 3: chiller CH-4, bearing temperature reached 57 C.\\\\nDay 15, shift 2: mixer M-5, filter replaced during scheduled stop.\\\\nDay 16, shift 3: mixer M-5, vibration rose to 4.8 mm/s.\\\\nDay 17, shift 3: fan F-12, oil level checked and topped up.\\\\nDay 18, shift 1: boiler B-2, operator reported unusual noise.\\\\nDay 19, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 20, shift 3: fan F-12, seal leak observed, 8 drops per minute.\\\\nDay 21, shift 2: fan F-12, vibration rose to 6.6 mm/s.\\\\nDay 22, shift 2: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 23, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 24, shift 2: mixer M-5, oil level checked and topped up.\\\\nDay 25, shift 3: conveyor CV-3, vibration rose to 2.7 mm/s.\\\\nDay 26, shift 1: mixer M-5, filter replaced during scheduled stop.\\\\nDay 27, shift 1: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 28, shift 3: pump P-101, oil level checked and topped up.\\\\nDay 29, shift 1: fan F-12, motor current 9 A above baseline.\\\\nDay 30, shift 3: fan F-12, motor current 6 A above baseline.\\\\nDay 31, shift 3: chiller CH-4, alignment checked, offset 0.44 mm.\\\\nDay 32, shift 2: chiller CH-4, vibration rose to 4.6 mm/s.\\\\nDay 33, shift 1: pump P-101, seal leak observed, 7 drops per minute.\\\\nDay 34, shift 1: fan F-12, filter replaced during scheduled stop.\\\\nDay 35, shift 2: boiler B-2, operator reported unusual noise.\\\\nDay 36, shift 1: pump P-101, oil level checked and topped up.\\\\n\\\\nWhat maintenance actions would you schedule next week?\\"}\\n{\\"id\\": \\"s13\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is thermal imaging used for in maintenance?\\"}\\n{\\"id\\": \\"s17\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is SCADA?\\"}\\n{\\"id\\": \\"l15\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 15:\\\\nDay 1, shift 2: compressor C-7, oil level checked and topped up.\\\\nDay 2, shift 2: conveyor CV-3, alignment checked, offset 0.5 mm.\\\\nDay 3, shift 1: mixer M-5, seal leak observed, 4 drops per minute.\\\\nDay 4, shift 3: chiller CH-4, seal leak observed, 30 drops per minute.\\\\nDay 5, shift 2: press PR-9, seal leak observed, 9 drops per minute.\\\\nDay 6, shift 1: pump P-101, alignment checked, offset 0.55 mm.\\\\nDay 7, shift 3: pump P-101, oil level checked and topped up.\\\\nDay 8, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 9, shift 3: compressor C-7, operator reported unusual noise.\\\\nDay 10, shift 3: compressor C-7, bearing temperature reached 75 C.\\\\nDay 11, shift 1: press PR-9, vibration rose to 7.1 mm/s.\\\\nDay 12, shift 2: mixer M-5, bearing temperature reached 92 C.\\\\nDay 13, shift 3: mixer M-5, seal leak observed, 14 drops per minute.\\\\nDay 14, shift 3: boiler B-2, operator reported unusual noise.\\\\nDay 15, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 16, shift 2: chiller CH-4, seal leak observed, 23 drops per minute.\\\\nDay 17, shift 1: mixer M-5, oil level checked and topped up.\\\\nDay 18, shift 1: press PR-9, alignment checked, offset 0.5 mm.\\\\nDay 19, shift 1: boiler B-2, seal leak observed, 15 drops per minute.\\\\nDay 20, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 21, shift 3: boiler B-2, bearing temperature reached 69 C.\\\\nDay 22, shift 2: fan F-12, vibration rose to 9.6 mm/s.\\\\nDay 23, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 24, shift 3: pump P-101, seal leak observed, 23 drops per minute.\\\\nDay 25, shift 1: press PR-9, seal leak observed, 7 drops per minute.\\\\nDay 26, shift 1: press PR-9, vibration rose to 11.7 mm/s.\\\\nDay 27, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 28, shift 1: chiller CH-4, seal leak observed, 15 drops per minute.\\\\nDay 29, shift 2: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 30, shift 3: pump P-101, operator reported unusual noise.\\\\nDay 31, shift 2: fan F-12, alignment checked, offset 0.5 mm.\\\\nDay 32, shift 2: chiller CH-4, oil level checked and topped up.\\\\nDay 33, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 34, shift 1: fan F-12, bearing temperature reached 56 C.\\\\nDay 35, shift 1: mixer M-5, motor current 2 A above baseline.\\\\nDay 36, shift 3: press PR-9, alignment checked, offset 0.16 mm.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"s02\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What does MTBF stand for and how is it used?\\"}\\n{\\"id\\": \\"s05\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Why do bearings fail early?\\"}\\n{\\"id\\": \\"s28\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How do you size a fuse for a motor circuit?\\"}\\n{\\"id\\": \\"s01\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain cavitation in centrifugal pumps.\\"}\\n{\\"id\\": \\"s31\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a work order in a CMMS?\\"}\\n{\\"id\\": \\"s26\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Explain what an interlock is.\\"}\\n{\\"id\\": \\"s43\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a hazard and operability study?\\"}\\n{\\"id\\": \\"s18\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe how a compressor works.\\"}\\n{\\"id\\": \\"s42\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How do you read a P&ID diagram?\\"}\\n{\\"id\\": \\"l07\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 7:\\\\nDay 1, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 2, shift 3: press PR-9, seal leak observed, 13 drops per minute.\\\\nDay 3, shift 1: press PR-9, motor current 5 A above baseline.\\\\nDay 4, shift 3: conveyor CV-3, motor current 13 A above baseline.\\\\nDay 5, shift 1: conveyor CV-3, seal leak observed, 14 drops per minute.\\\\nDay 6, shift 1: fan F-12, vibration rose to 6.4 mm/s.\\\\nDay 7, shift 3: compressor C-7, operator reported unusual noise.\\\\nDay 8, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 9, shift 1: fan F-12, operator reported unusual noise.\\\\nDay 10, shift 2: conveyor CV-3, motor current 4 A above baseline.\\\\nDay 11, shift 2: compressor C-7, operator reported unusual noise.\\\\nDay 12, shift 1: pump P-101, alignment checked, offset 0.5 mm.\\\\nDay 13, shift 2: chiller CH-4, seal leak observed, 19 drops per minute.\\\\nDay 14, shift 3: conveyor CV-3, alignment checked, offset 0.11 mm.\\\\nDay 15, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 16, shift 3: boiler B-2, vibration rose to 3.3 mm/s.\\\\nDay 17, shift 2: boiler B-2, seal leak observed, 5 drops per minute.\\\\nDay 18, shift 3: mixer M-5, seal leak observed, 24 drops per minute.\\\\nDay 19, shift 1: conveyor CV-3, operator reported unusual noise.\\\\nDay 20, shift 1: pump P-101, vibration rose to 10.0 mm/s.\\\\nDay 21, shift 3: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 22, shift 1: compressor C-7, oil level checked and topped up.\\\\nDay 23, shift 3: pump P-101, motor current 11 A above baseline.\\\\nDay 24, shift 1: conveyor CV-3, oil level checked and topped up.\\\\nDay 25, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 26, shift 2: conveyor CV-3, alignment checked, offset 0.33 mm.\\\\nDay 27, shift 2: chiller CH-4, bearing temperature reached 87 C.\\\\nDay 28, shift 1: conveyor CV-3, filter replaced during scheduled stop.\\\\nDay 29, shift 2: compressor C-7, seal leak observed, 10 drops per minute.\\\\nDay 30, shift 3: chiller CH-4, vibration rose to 8.4 mm/s.\\\\nDay 31, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 32, shift 1: pump P-101, alignment checked, offset 0.08 mm.\\\\nDay 33, shift 1: press PR-9, seal leak observed, 2 drops per minute.\\\\nDay 34, shift 1: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 35, shift 3: mixer M-5, motor current 12 A above baseline.\\\\nDay 36, shift 2: press PR-9, seal leak observed, 3 drops per minute.\\\\n\\\\nList every asset with a temperature or vibration problem.\\"}\\n{\\"id\\": \\"l05\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 5:\\\\nDay 1, shift 2: pump P-101, bearing temperature reached 94 C.\\\\nDay 2, shift 3: mixer M-5, oil level checked and topped up.\\\\nDay 3, shift 1: mixer M-5, filter replaced during scheduled stop.\\\\nDay 4, shift 3: mixer M-5, operator reported unusual noise.\\\\nDay 5, shift 3: press PR-9, bearing temperature reached 65 C.\\\\nDay 6, shift 1: conveyor CV-3, vibration rose to 9.9 mm/s.\\\\nDay 7, shift 3: fan F-12, motor current 5 A above baseline.\\\\nDay 8, shift 2: conveyor CV-3, seal leak observed, 16 drops per minute.\\\\nDay 9, shift 1: press PR-9, operator reported unusual noise.\\\\nDay 10, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 11, shift 3: pump P-101, oil level checked and topped up.\\\\nDay 12, shift 3: fan F-12, motor current 14 A above baseline.\\\\nDay 13, shift 1: chiller CH-4, vibration rose to 7.7 mm/s.\\\\nDay 14, shift 3: compressor C-7, vibration rose to 2.5 mm/s.\\\\nDay 15, shift 2: press PR-9, motor current 15 A above baseline.\\\\nDay 16, shift 1: chiller CH-4, motor current 13 A above baseline.\\\\nDay 17, shift 2: press PR-9, filter replaced during scheduled stop.\\\\nDay 18, shift 2: pump P-101, filter replaced during scheduled stop.\\\\nDay 19, shift 3: chiller CH-4, vibration rose to 7.8 mm/s.\\\\nDay 20, shift 3: compressor C-7, bearing temperature reached 89 C.\\\\nDay 21, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 22, shift 3: pump P-101, alignment checked, offset 0.14 mm.\\\\nDay 23, shift 1: mixer M-5, motor current 3 A above baseline.\\\\nDay 24, shift 2: compressor C-7, alignment checked, offset 0.06 mm.\\\\nDay 25, shift 2: boiler B-2, operator reported unusual noise.\\\\nDay 26, shift 1: mixer M-5, oil level checked and topped up.\\\\nDay 27, shift 3: fan F-12, motor current 4 A above baseline.\\\\nDay 28, shift 1: chiller CH-4, bearing temperature reached 90 C.\\\\nDay 29, shift 1: chiller CH-4, seal leak observed, 4 drops per minute.\\\\nDay 30, shift 1: press PR-9, bearing temperature reached 72 C.\\\\nDay 31, shift 3: compressor C-7, operator reported unusual noise.\\\\nDay 32, shift 2: boiler B-2, seal leak observed, 30 drops per minute.\\\\nDay 33, shift 1: mixer M-5, filter replaced during scheduled stop.\\\\nDay 34, shift 1: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 35, shift 2: fan F-12, bearing temperature reached 71 C.\\\\nDay 36, shift 3: mixer M-5, vibration rose to 9.9 mm/s.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"s24\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"Describe the main parts of a hydraulic system.\\"}\\n{\\"id\\": \\"l04\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 4:\\\\nDay 1, shift 2: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 2, shift 3: conveyor CV-3, alignment checked, offset 0.19 mm.\\\\nDay 3, shift 1: compressor C-7, seal leak observed, 26 drops per minute.\\\\nDay 4, shift 3: pump P-101, seal leak observed, 25 drops per minute.\\\\nDay 5, shift 2: boiler B-2, oil level checked and topped up.\\\\nDay 6, shift 3: chiller CH-4, oil level checked and topped up.\\\\nDay 7, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 8, shift 1: fan F-12, operator reported unusual noise.\\\\nDay 9, shift 3: compressor C-7, filter replaced during scheduled stop.\\\\nDay 10, shift 2: mixer M-5, bearing temperature reached 80 C.\\\\nDay 11, shift 2: chiller CH-4, operator reported unusual noise.\\\\nDay 12, shift 3: chiller CH-4, seal leak observed, 3 drops per minute.\\\\nDay 13, shift 1: pump P-101, vibration rose to 2.9 mm/s.\\\\nDay 14, shift 2: chiller CH-4, operator reported unusual noise.\\\\nDay 15, shift 2: pump P-101, motor current 9 A above baseline.\\\\nDay 16, shift 2: press PR-9, bearing temperature reached 83 C.\\\\nDay 17, shift 1: pump P-101, oil level checked and topped up.\\\\nDay 18, shift 1: conveyor CV-3, motor current 15 A above baseline.\\\\nDay 19, shift 2: boiler B-2, bearing temperature reached 76 C.\\\\nDay 20, shift 1: compressor C-7, filter replaced during scheduled stop.\\\\nDay 21, shift 1: compressor C-7, vibration rose to 4.5 mm/s.\\\\nDay 22, shift 3: compressor C-7, seal leak observed, 23 drops per minute.\\\\nDay 23, shift 3: fan F-12, seal leak observed, 11 drops per minute.\\\\nDay 24, shift 2: conveyor CV-3, bearing temperature reached 77 C.\\\\nDay 25, shift 1: mixer M-5, alignment checked, offset 0.19 mm.\\\\nDay 26, shift 1: boiler B-2, seal leak observed, 22 drops per minute.\\\\nDay 27, shift 2: chiller CH-4, seal leak observed, 29 drops per minute.\\\\nDay 28, shift 3: compressor C-7, alignment checked, offset 0.4 mm.\\\\nDay 29, shift 2: fan F-12, alignment checked, offset 0.36 mm.\\\\nDay 30, shift 2: fan F-12, motor current 7 A above baseline.\\\\nDay 31, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 32, shift 1: fan F-12, bearing temperature reached 71 C.\\\\nDay 33, shift 1: chiller CH-4, filter replaced during scheduled stop.\\\\nDay 34, shift 3: conveyor CV-3, seal leak observed, 10 drops per minute.\\\\nDay 35, shift 2: fan F-12, filter replaced during scheduled stop.\\\\nDay 36, shift 2: conveyor CV-3, bearing temperature reached 56 C.\\\\n\\\\nWhich asset is most at risk of failure and why?\\"}\\n{\\"id\\": \\"s46\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is a reliability centered maintenance program?\\"}\\n{\\"id\\": \\"l09\\", \\"kind\\": \\"long\\", \\"prompt\\": \\"Maintenance log for line 9:\\\\nDay 1, shift 3: boiler B-2, bearing temperature reached 81 C.\\\\nDay 2, shift 2: fan F-12, seal leak observed, 2 drops per minute.\\\\nDay 3, shift 1: pump P-101, operator reported unusual noise.\\\\nDay 4, shift 1: pump P-101, motor current 12 A above baseline.\\\\nDay 5, shift 1: press PR-9, filter replaced during scheduled stop.\\\\nDay 6, shift 3: conveyor CV-3, operator reported unusual noise.\\\\nDay 7, shift 3: chiller CH-4, bearing temperature reached 86 C.\\\\nDay 8, shift 2: compressor C-7, vibration rose to 5.0 mm/s.\\\\nDay 9, shift 1: boiler B-2, bearing temperature reached 59 C.\\\\nDay 10, shift 2: boiler B-2, oil level checked and topped up.\\\\nDay 11, shift 3: chiller CH-4, oil level checked and topped up.\\\\nDay 12, shift 3: fan F-12, seal leak observed, 25 drops per minute.\\\\nDay 13, shift 2: compressor C-7, filter replaced during scheduled stop.\\\\nDay 14, shift 3: press PR-9, motor current 9 A above baseline.\\\\nDay 15, shift 2: pump P-101, filter replaced during scheduled stop.\\\\nDay 16, shift 3: conveyor CV-3, bearing temperature reached 71 C.\\\\nDay 17, shift 1: fan F-12, oil level checked and topped up.\\\\nDay 18, shift 2: mixer M-5, alignment checked, offset 0.32 mm.\\\\nDay 19, shift 1: press PR-9, vibration rose to 6.0 mm/s.\\\\nDay 20, shift 1: conveyor CV-3, bearing temperature reached 76 C.\\\\nDay 21, shift 3: press PR-9, oil level checked and topped up.\\\\nDay 22, shift 2: mixer M-5, operator reported unusual noise.\\\\nDay 23, shift 3: pump P-101, seal leak observed, 6 drops per minute.\\\\nDay 24, shift 3: boiler B-2, vibration rose to 8.6 mm/s.\\\\nDay 25, shift 1: chiller CH-4, oil level checked and topped up.\\\\nDay 26, shift 3: fan F-12, motor current 14 A above baseline.\\\\nDay 27, shift 1: fan F-12, bearing temperature reached 82 C.\\\\nDay 28, shift 1: press PR-9, oil level checked and topped up.\\\\nDay 29, shift 1: mixer M-5, vibration rose to 2.5 mm/s.\\\\nDay 30, shift 1: fan F-12, alignment checked, offset 0.08 mm.\\\\nDay 31, shift 1: fan F-12, seal leak observed, 18 drops per minute.\\\\nDay 32, shift 2: compressor C-7, motor current 11 A above baseline.\\\\nDay 33, shift 3: press PR-9, alignment checked, offset 0.21 mm.\\\\nDay 34, shift 3: compressor C-7, oil level checked and topped up.\\\\nDay 35, shift 2: pump P-101, alignment checked, offset 0.46 mm.\\\\nDay 36, shift 3: chiller CH-4, bearing temperature reached 61 C.\\\\n\\\\nSummarise the main trends in this log.\\"}\\n{\\"id\\": \\"s23\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the purpose of a check valve?\\"}\\n{\\"id\\": \\"s10\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"How does a variable frequency drive save energy?\\"}\\n{\\"id\\": \\"s11\\", \\"kind\\": \\"short\\", \\"prompt\\": \\"What is the role of a PLC in a factory?\\"}\\n"], "benchmark/accuracy/run_gsm8k.sh": ["67b262aa1074bacae97c9cc9bba97671f3d4b411d10c3c349b1c37a8f72d0dbb", "#!/usr/bin/env bash\\n# GSM8K accuracy against any OpenAI-compatible chat endpoint (llama.cpp or vLLM).\\n# Usage: run_gsm8k.sh BASE_URL MODEL OUT_DIR [LIMIT] [NUM_CONCURRENT]\\n#   BASE_URL  e.g. http://127.0.0.1:8080/v1/chat/completions\\n#   MODEL     model name the server expects (vLLM: --served-model-name; llama.cpp: anything)\\n#   LIMIT     first N test items (default 250, the fixed Phase 2 subset)\\n# Needs lm_eval on PATH (pip install -r benchmark/requirements.txt).\\nset -euo pipefail\\nBASE_URL=$1 MODEL=$2 OUT_DIR=$3 LIMIT=${4:-250} NUM_CONCURRENT=${5:-2}\\nHERE=$(cd \\"$(dirname \\"$0\\")\\" && pwd)\\nmkdir -p \\"$OUT_DIR\\"\\nlm_eval run \\\\\\n  --model local-chat-completions \\\\\\n  --model_args \\"model=$MODEL,base_url=$BASE_URL,num_concurrent=$NUM_CONCURRENT,max_retries=3,timeout=600,tokenizer_backend=None\\" \\\\\\n  --include_path \\"$HERE/tasks\\" \\\\\\n  --tasks gsm8k_pinned \\\\\\n  --apply_chat_template \\\\\\n  --fewshot_as_multiturn \\\\\\n  --limit \\"$LIMIT\\" \\\\\\n  --seed 0,1234,1234,1234 \\\\\\n  --log_samples \\\\\\n  --output_path \\"$OUT_DIR\\" 2>&1 | tee \\"$OUT_DIR/lm_eval.log\\"\\n"], "benchmark/accuracy/tasks/gsm8k_pinned.yaml": ["8fcecb0b938ca1f34db1e2ab4efdc5b75383253d9a2739dee9180c26a65ea2f5", "# Copy of lm-eval 0.4.13 tasks/gsm8k/gsm8k.yaml with the dataset revision pinned.\\ntask: gsm8k_pinned\\ndataset_path: openai/gsm8k\\ndataset_name: main\\ndataset_kwargs:\\n  revision: 740312add88f781978c0658806c59bc2815b9866\\noutput_type: generate_until\\ntraining_split: train\\nfewshot_split: train\\ntest_split: test\\ndoc_to_text: \\"Question: {{question}}\\\\nAnswer:\\"\\ndoc_to_target: \\"{{answer}}\\"\\nmetric_list:\\n  - metric: exact_match\\n    aggregation: mean\\n    higher_is_better: true\\n    ignore_case: true\\n    ignore_punctuation: false\\n    regexes_to_ignore:\\n      - \\",\\"\\n      - \\"\\\\\\\\$\\"\\n      - \\"(?s).*#### \\"\\n      - \\"\\\\\\\\.$\\"\\ngeneration_kwargs:\\n  until:\\n    - \\"Question:\\"\\n    - \\"</s>\\"\\n    - \\"<|im_end|>\\"\\n  do_sample: false\\n  temperature: 0.0\\nrepeats: 1\\nnum_fewshot: 5\\nfilter_list:\\n  - name: \\"strict-match\\"\\n    filter:\\n      - function: \\"regex\\"\\n        regex_pattern: \\"#### (\\\\\\\\-?[0-9\\\\\\\\.\\\\\\\\,]+)\\"\\n      - function: \\"take_first\\"\\n  - name: \\"flexible-extract\\"\\n    filter:\\n      - function: \\"regex\\"\\n        group_select: -1\\n        regex_pattern: \\"(-?[$0-9.,]{2,})|(-?[0-9]+)\\"\\n      - function: \\"take_first\\"\\nmetadata:\\n  version: 3.0\\n"], "benchmark/requirements.txt": ["c91bc10ea14e9da3c2e3124076c3d64827a21990af64e6de52e781bbe1744a60", "# This file was autogenerated by uv via the following command:\\n#    uv pip compile --universal --python-version 3.12 benchmark/requirements.in -o benchmark/requirements.txt\\nabsl-py==2.5.0\\n    # via rouge-score\\naiohappyeyeballs==2.7.1\\n    # via aiohttp\\naiohttp==3.14.4\\n    # via\\n    #   fsspec\\n    #   lm-eval\\naiosignal==1.4.0\\n    # via aiohttp\\nanyio==4.15.1\\n    # via\\n    #   httpx\\n    #   httpx2\\nattrs==26.1.0\\n    # via aiohttp\\ncertifi==2026.7.22\\n    # via\\n    #   httpcore\\n    #   httpx\\n    #   requests\\nchardet==6.0.0.post1\\n    # via mbstrdecoder\\ncharset-normalizer==3.5.2\\n    # via requests\\nclick==8.5.0\\n    # via\\n    #   huggingface-hub\\n    #   nltk\\ncloudpickle==3.1.2\\n    # via joblib\\ncolorama==0.4.6\\n    # via\\n    #   pytest\\n    #   sacrebleu\\n    #   tqdm\\ncontourpy==1.4.0\\n    # via matplotlib\\ncycler==0.12.1\\n    # via matplotlib\\ndataproperty==1.1.1\\n    # via\\n    #   pytablewriter\\n    #   tabledata\\ndatasets==5.1.0\\n    # via\\n    #   evaluate\\n    #   lm-eval\\ndefusedxml==0.7.1\\n    # via nltk\\ndill==0.4.1\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   lm-eval\\n    #   multiprocess\\nevaluate==0.4.6\\n    # via lm-eval\\nfilelock==4.0.12\\n    # via\\n    #   datasets\\n    #   huggingface-hub\\nfonttools==4.66.1\\n    # via matplotlib\\nfrozenlist==1.8.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\nfsspec==2026.7.0\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\nh11==0.16.0\\n    # via\\n    #   httpcore\\n    #   httpcore2\\nhf-xet==1.7.0 ; platform_machine == \'AMD64\' or platform_machine == \'aarch64\' or platform_machine == \'amd64\' or platform_machine == \'arm64\' or platform_machine == \'x86_64\'\\n    # via huggingface-hub\\nhttpcore==1.0.9\\n    # via httpx\\nhttpcore2==2.13.1 ; sys_platform != \'emscripten\'\\n    # via httpx2\\nhttpx==0.28.1\\n    # via -r benchmark/requirements.in\\nhttpx2==2.13.1\\n    # via huggingface-hub\\nhttpx2-jsfetch==1.0 ; sys_platform == \'emscripten\'\\n    # via httpx2\\nhuggingface-hub==2.1.1\\n    # via\\n    #   datasets\\n    #   evaluate\\nidna==3.20\\n    # via\\n    #   anyio\\n    #   httpx\\n    #   httpx2\\n    #   requests\\n    #   yarl\\niniconfig==2.3.1\\n    # via pytest\\njinja2==3.1.6\\n    # via lm-eval\\njoblib==1.6.0\\n    # via\\n    #   nltk\\n    #   scikit-learn\\nkiwisolver==1.5.1\\n    # via matplotlib\\nlm-eval==0.4.13\\n    # via -r benchmark/requirements.in\\nlxml==6.1.3\\n    # via sacrebleu\\nmarkupsafe==3.0.4\\n    # via jinja2\\nmatplotlib==3.11.2\\n    # via -r benchmark/requirements.in\\nmbstrdecoder==1.1.5\\n    # via\\n    #   dataproperty\\n    #   pytablewriter\\n    #   typepy\\nmore-itertools==11.1.0\\n    # via lm-eval\\nmultidict==6.9.1\\n    # via\\n    #   aiohttp\\n    #   yarl\\nmultiprocess==0.70.19\\n    # via\\n    #   datasets\\n    #   evaluate\\nnarwhals==2.26.0\\n    # via scikit-learn\\nnltk==3.10.3\\n    # via rouge-score\\nnumpy==2.5.3\\n    # via\\n    #   contourpy\\n    #   datasets\\n    #   evaluate\\n    #   lm-eval\\n    #   matplotlib\\n    #   pandas\\n    #   rouge-score\\n    #   sacrebleu\\n    #   scikit-learn\\n    #   scipy\\npackaging==26.3\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\n    #   matplotlib\\n    #   pytest\\n    #   typepy\\npandas==3.0.6\\n    # via\\n    #   datasets\\n    #   evaluate\\npathvalidate==3.3.1\\n    # via pytablewriter\\npillow==12.3.0\\n    # via matplotlib\\npluggy==1.6.0\\n    # via pytest\\nportalocker==4.4.0\\n    # via sacrebleu\\npropcache==0.5.4\\n    # via\\n    #   aiohttp\\n    #   yarl\\npyarrow==25.0.1\\n    # via datasets\\npygments==2.21.0\\n    # via pytest\\npyparsing==3.3.3\\n    # via matplotlib\\npytablewriter==1.2.1\\n    # via lm-eval\\npytest==9.1.1\\n    # via -r benchmark/requirements.in\\npython-dateutil==2.9.0.post0\\n    # via\\n    #   matplotlib\\n    #   pandas\\n    #   typepy\\npytz==2026.5\\n    # via typepy\\npyyaml==6.0.3\\n    # via\\n    #   datasets\\n    #   huggingface-hub\\nregex==2026.9.29\\n    # via\\n    #   nltk\\n    #   sacrebleu\\n    #   tiktoken\\nrequests==2.34.2\\n    # via\\n    #   evaluate\\n    #   lm-eval\\n    #   tiktoken\\nrouge-score==0.1.2\\n    # via lm-eval\\nsacrebleu==2.6.0\\n    # via lm-eval\\nscikit-learn==1.9.1\\n    # via lm-eval\\nscipy==1.18.1\\n    # via scikit-learn\\nsetuptools==84.0.0\\n    # via pytablewriter\\nsix==1.17.0\\n    # via\\n    #   python-dateutil\\n    #   rouge-score\\nsqlitedict==2.1.0\\n    # via lm-eval\\ntabledata==1.3.5\\n    # via pytablewriter\\ntabulate==0.10.0\\n    # via sacrebleu\\ntcolorpy==0.1.7\\n    # via pytablewriter\\ntenacity==9.2.1\\n    # via lm-eval\\nthreadpoolctl==3.7.0\\n    # via scikit-learn\\ntiktoken==0.14.0\\n    # via lm-eval\\ntqdm==4.70.1\\n    # via\\n    #   datasets\\n    #   evaluate\\n    #   huggingface-hub\\n    #   lm-eval\\n    #   nltk\\ntruststore==0.10.4 ; sys_platform != \'emscripten\'\\n    # via\\n    #   httpcore2\\n    #   httpx2\\ntypepy==1.3.5\\n    # via\\n    #   dataproperty\\n    #   pytablewriter\\n    #   tabledata\\ntyping-extensions==4.16.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\n    #   anyio\\n    #   httpx2\\n    #   huggingface-hub\\n    #   lm-eval\\ntzdata==2026.5 ; sys_platform == \'emscripten\' or sys_platform == \'win32\'\\n    # via pandas\\nurllib3==2.8.0\\n    # via requests\\nword2number==1.1\\n    # via lm-eval\\nxxhash==4.0.1\\n    # via\\n    #   datasets\\n    #   evaluate\\nyarl==1.25.1\\n    # via aiohttp\\n"], "notebooks/vllm-requirements.txt": ["9218775e0b0c2b1a0675402c53a77297525f0d2675d18548f2db5a83841bfe7e", "# Generated from notebooks/vllm-requirements.in (see the commands there). Install with --torch-backend cu129.\\nagent-detector==2.0.0\\n    # via fastapi-cloud-cli\\naiohappyeyeballs==2.7.1\\n    # via aiohttp\\naiohttp==3.14.4\\n    # via vllm\\naiosignal==1.4.0\\n    # via aiohttp\\nannotated-doc==0.0.5\\n    # via\\n    #   fastapi\\n    #   typer\\nannotated-types==0.8.0\\n    # via pydantic\\nanthropic==1.11.0\\n    # via vllm\\nanyio==4.15.1\\n    # via\\n    #   anthropic\\n    #   httpx\\n    #   httpx2\\n    #   mcp\\n    #   openai\\n    #   sse-starlette\\n    #   starlette\\n    #   watchfiles\\napache-tvm-ffi==0.1.11\\n    # via\\n    #   flashinfer-python\\n    #   nvidia-cudnn-frontend\\n    #   quack-kernels\\n    #   tilelang\\n    #   tokenspeed-mla\\n    #   vllm\\n    #   xgrammar\\nastor==0.8.1\\n    # via depyf\\nattrs==26.1.0\\n    # via\\n    #   aiohttp\\n    #   jsonschema\\n    #   referencing\\nblake3==1.0.10\\n    # via vllm\\ncachetools==7.2.1\\n    # via vllm\\ncbor2==6.1.5\\n    # via vllm\\ncertifi==2026.7.22\\n    # via\\n    #   httpcore\\n    #   httpx\\n    #   requests\\n    #   sentry-sdk\\ncffi==2.1.1\\n    # via cryptography\\ncharset-normalizer==3.5.2\\n    # via requests\\nclick==8.5.0\\n    # via\\n    #   flashinfer-python\\n    #   huggingface-hub\\n    #   rich-toolkit\\n    #   uvicorn\\ncloudpickle==3.1.2\\n    # via\\n    #   tilelang\\n    #   vllm\\ncompressed-tensors==0.17.0\\n    # via vllm\\ncryptography==50.0.2\\n    # via pyjwt\\ncuda-bindings==12.9.9\\n    # via\\n    #   cuda-python\\n    #   humming-kernels\\n    #   torch\\ncuda-core==1.2.1\\n    # via\\n    #   nccl-extensions\\n    #   nccl4py\\ncuda-pathfinder==1.8.3\\n    # via\\n    #   cuda-bindings\\n    #   cuda-core\\n    #   nccl-extensions\\n    #   nccl4py\\ncuda-python==12.9.7\\n    # via\\n    #   flashinfer-python\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\ncuda-tile==1.6.0\\n    # via flashinfer-python\\ncuda-toolkit==12.9.1\\n    # via torch\\ndepyf==0.20.0\\n    # via vllm\\ndetect-installer==0.2.1\\n    # via fastapi-cloud-cli\\ndill==0.4.1\\n    # via depyf\\ndnspython==2.8.0\\n    # via email-validator\\ndocstring-parser==0.18.0\\n    # via anthropic\\neinops==0.8.2\\n    # via\\n    #   flashinfer-python\\n    #   quack-kernels\\n    #   vllm\\nemail-validator==2.3.0\\n    # via\\n    #   fastapi\\n    #   pydantic\\nfastapi==0.136.3\\n    # via\\n    #   model-hosting-container-standards\\n    #   vllm\\nfastapi-cli==0.0.32\\n    # via fastapi\\nfastapi-cloud-cli==0.26.0\\n    # via fastapi-cli\\nfastar==0.12.0\\n    # via\\n    #   fastapi\\n    #   fastapi-cloud-cli\\nfastsafetensors==0.4.0\\n    # via vllm\\nfilelock==4.0.12\\n    # via\\n    #   huggingface-hub\\n    #   torch\\n    #   vllm\\nflashinfer-python==0.7.0.post1\\n    # via vllm\\nfrozenlist==1.8.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\nfsspec==2026.9.0\\n    # via\\n    #   huggingface-hub\\n    #   torch\\ngoogleapis-common-protos==1.75.5\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\ngrpcio==1.84.0\\n    # via opentelemetry-exporter-otlp-proto-grpc\\nh11==0.16.0\\n    # via\\n    #   httpcore\\n    #   httpcore2\\n    #   uvicorn\\nhf-xet==1.7.0\\n    # via huggingface-hub\\nhttpcore==1.0.9\\n    # via httpx\\nhttpcore2==2.13.1\\n    # via httpx2\\nhttptools==0.8.0\\n    # via uvicorn\\nhttpx==0.28.1\\n    # via\\n    #   fastapi\\n    #   fastapi-cloud-cli\\n    #   huggingface-hub\\n    #   model-hosting-container-standards\\nhttpx2==2.13.1\\n    # via\\n    #   anthropic\\n    #   mcp\\n    #   openai\\nhuggingface-hub==1.33.0\\n    # via\\n    #   tokenizers\\n    #   transformers\\n    #   vllm\\nhumming-kernels==0.1.16\\n    # via\\n    #   --override notebooks/vllm-overrides.txt\\n    #   vllm\\nidna==3.20\\n    # via\\n    #   anyio\\n    #   email-validator\\n    #   httpx\\n    #   httpx2\\n    #   requests\\n    #   yarl\\nijson==3.5.1\\n    # via vllm\\ninstanttensor==0.2.1\\n    # via vllm\\ninteregular==0.3.3\\n    # via lm-format-enforcer\\njinja2==3.1.6\\n    # via\\n    #   fastapi\\n    #   humming-kernels\\n    #   torch\\njiter==0.17.0\\n    # via\\n    #   anthropic\\n    #   openai\\njmespath==1.1.0\\n    # via model-hosting-container-standards\\njsonschema==4.26.0\\n    # via\\n    #   mcp\\n    #   mistral-common\\n    #   vllm\\njsonschema-specifications==2025.9.1\\n    # via jsonschema\\nlark==1.2.2\\n    # via vllm\\nllguidance==1.7.6\\n    # via vllm\\nllvmlite==0.47.0\\n    # via numba\\nlm-format-enforcer==0.11.3\\n    # via vllm\\nloguru==0.7.3\\n    # via compressed-tensors\\nmarkdown-it-py==4.2.0\\n    # via rich\\nmarkupsafe==3.0.4\\n    # via jinja2\\nmcp==2.3.0\\n    # via vllm\\nmcp-types==2.3.0\\n    # via mcp\\nmdurl==0.1.2\\n    # via markdown-it-py\\nmistral-common==1.12.0\\n    # via vllm\\nml-dtypes==0.6.0\\n    # via tilelang\\nmodel-hosting-container-standards==0.1.16\\n    # via vllm\\nmpmath==1.3.0\\n    # via sympy\\nmsgspec==0.22.0\\n    # via vllm\\nmultidict==6.9.1\\n    # via\\n    #   aiohttp\\n    #   yarl\\nnccl-extensions==0.1.0\\n    # via flashinfer-python\\nnccl4py==0.6.0\\n    # via\\n    #   flashinfer-python\\n    #   nccl-extensions\\nnetworkx==3.7\\n    # via torch\\nninja==1.13.2\\n    # via\\n    #   flashinfer-python\\n    #   vllm\\nnumba==0.65.0\\n    # via vllm\\nnumpy==2.3.5\\n    # via\\n    #   cuda-core\\n    #   flashinfer-python\\n    #   humming-kernels\\n    #   mistral-common\\n    #   ml-dtypes\\n    #   nccl-extensions\\n    #   nccl4py\\n    #   numba\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\n    #   opencv-python-headless\\n    #   tilelang\\n    #   torchvision\\n    #   transformers\\n    #   vllm\\n    #   xgrammar\\nnvidia-cublas-cu12==12.9.1.4\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cudnn-cu12\\n    #   nvidia-cusolver-cu12\\nnvidia-cuda-cccl-cu12==12.9.27\\n    # via humming-kernels\\nnvidia-cuda-cupti-cu12==12.9.79\\n    # via cuda-toolkit\\nnvidia-cuda-nvcc-cu12==12.9.86\\n    # via humming-kernels\\nnvidia-cuda-nvdisasm==13.4.92\\n    # via\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\nnvidia-cuda-nvrtc-cu12==12.9.86\\n    # via\\n    #   cuda-toolkit\\n    #   humming-kernels\\nnvidia-cuda-runtime-cu12==12.9.79\\n    # via\\n    #   cuda-toolkit\\n    #   humming-kernels\\nnvidia-cudnn-cu12==9.20.0.48\\n    # via torch\\nnvidia-cudnn-frontend==1.30.0\\n    # via\\n    #   flashinfer-python\\n    #   vllm\\nnvidia-cufft-cu12==11.4.1.4\\n    # via cuda-toolkit\\nnvidia-cufile-cu12==1.14.1.1\\n    # via cuda-toolkit\\nnvidia-curand-cu12==10.3.10.19\\n    # via cuda-toolkit\\nnvidia-cusolver-cu12==11.7.5.82\\n    # via cuda-toolkit\\nnvidia-cusparse-cu12==12.5.10.65\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cusolver-cu12\\nnvidia-cusparselt-cu12==0.8.1\\n    # via torch\\nnvidia-cutlass-dsl==4.7.1\\n    # via\\n    #   flashinfer-python\\n    #   nvidia-cudnn-frontend\\n    #   quack-kernels\\n    #   tokenspeed-mla\\n    #   vllm\\nnvidia-cutlass-dsl-libs-base==4.7.1\\n    # via\\n    #   nvidia-cutlass-dsl\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\nnvidia-cutlass-dsl-libs-core==4.7.1\\n    # via nvidia-cutlass-dsl-libs-base\\nnvidia-cutlass-dsl-libs-cu12==4.7.1\\n    # via nvidia-cutlass-dsl\\nnvidia-cutlass-dsl-libs-cu13==4.7.1\\n    # via nvidia-cutlass-dsl\\nnvidia-ml-py==13.615.71\\n    # via flashinfer-python\\nnvidia-nccl-cu12==2.29.7\\n    # via torch\\nnvidia-nvjitlink-cu12==12.9.86\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cufft-cu12\\n    #   nvidia-cusolver-cu12\\n    #   nvidia-cusparse-cu12\\nnvidia-nvshmem-cu12==3.4.5\\n    # via torch\\nnvidia-nvtx-cu12==12.9.79\\n    # via cuda-toolkit\\nnvtx==0.2.15\\n    # via vllm\\nopenai==3.26.0\\n    # via vllm\\nopencv-python-headless==5.0.0.93\\n    # via\\n    #   mistral-common\\n    #   vllm\\nopentelemetry-api==1.45.1\\n    # via\\n    #   mcp\\n    #   opentelemetry-exporter-http-transport\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   opentelemetry-sdk\\n    #   opentelemetry-semantic-conventions\\n    #   vllm\\nopentelemetry-exporter-http-transport==0.66b1\\n    # via opentelemetry-exporter-otlp-proto-http\\nopentelemetry-exporter-otlp==1.45.1\\n    # via vllm\\nopentelemetry-exporter-otlp-common==0.66b1\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\nopentelemetry-exporter-otlp-proto-common==1.45.1\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\nopentelemetry-exporter-otlp-proto-grpc==1.45.1\\n    # via opentelemetry-exporter-otlp\\nopentelemetry-exporter-otlp-proto-http==1.45.1\\n    # via opentelemetry-exporter-otlp\\nopentelemetry-proto==1.45.1\\n    # via\\n    #   opentelemetry-exporter-otlp-proto-common\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\nopentelemetry-sdk==1.45.1\\n    # via\\n    #   opentelemetry-exporter-otlp-common\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   opentelemetry-semantic-conventions-ai\\n    #   vllm\\nopentelemetry-semantic-conventions==0.66b1\\n    # via\\n    #   opentelemetry-sdk\\n    #   opentelemetry-semantic-conventions-ai\\nopentelemetry-semantic-conventions-ai==0.5.1\\n    # via vllm\\noss-harmony==0.0.11\\n    # via vllm\\noutlines-core==0.2.14\\n    # via vllm\\npackaging==26.3\\n    # via\\n    #   flashinfer-python\\n    #   huggingface-hub\\n    #   lm-format-enforcer\\n    #   nccl-extensions\\n    #   nccl4py\\n    #   transformers\\npartial-json-parser==0.2.1.1.post7\\n    # via vllm\\npillow==12.3.0\\n    # via\\n    #   mistral-common\\n    #   torchvision\\n    #   vllm\\nprometheus-client==0.26.0\\n    # via\\n    #   prometheus-fastapi-instrumentator\\n    #   vllm\\nprometheus-fastapi-instrumentator==8.1.0\\n    # via vllm\\npropcache==0.5.4\\n    # via\\n    #   aiohttp\\n    #   yarl\\nprotobuf==7.36.2\\n    # via\\n    #   googleapis-common-protos\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\n    #   opentelemetry-proto\\n    #   vllm\\npsutil==7.2.2\\n    # via\\n    #   tilelang\\n    #   vllm\\npy-cpuinfo==9.0.0\\n    # via vllm\\npybase64==1.5.1\\n    # via vllm\\npycountry==26.2.16\\n    # via pydantic-extra-types\\npycparser==3.0\\n    # via cffi\\npydantic==2.13.5\\n    # via\\n    #   anthropic\\n    #   compressed-tensors\\n    #   fastapi\\n    #   fastapi-cloud-cli\\n    #   lm-format-enforcer\\n    #   mcp\\n    #   mcp-types\\n    #   mistral-common\\n    #   model-hosting-container-standards\\n    #   openai\\n    #   oss-harmony\\n    #   pydantic-extra-types\\n    #   pydantic-settings\\n    #   vllm\\n    #   xgrammar\\npydantic-core==2.46.5\\n    # via pydantic\\npydantic-extra-types==2.11.1\\n    # via\\n    #   fastapi\\n    #   mistral-common\\npydantic-settings==2.15.0\\n    # via fastapi\\npygments==2.21.0\\n    # via rich\\npyjwt==2.15.1\\n    # via mcp\\npynvvideocodec==2.0.4\\n    # via vllm\\npython-dotenv==1.2.4\\n    # via\\n    #   pydantic-settings\\n    #   uvicorn\\npython-json-logger==4.2.0\\n    # via vllm\\npython-multipart==0.0.32\\n    # via\\n    #   fastapi\\n    #   mcp\\npyyaml==6.0.3\\n    # via\\n    #   huggingface-hub\\n    #   lm-format-enforcer\\n    #   transformers\\n    #   uvicorn\\n    #   vllm\\npyzmq==27.2.0\\n    # via vllm\\nquack-kernels==0.6.5\\n    # via vllm\\nreferencing==0.37.0\\n    # via\\n    #   jsonschema\\n    #   jsonschema-specifications\\nregex==2026.9.29\\n    # via\\n    #   tiktoken\\n    #   transformers\\n    #   vllm\\nrequests==2.34.2\\n    # via\\n    #   flashinfer-python\\n    #   mistral-common\\n    #   opentelemetry-exporter-http-transport\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   tiktoken\\n    #   vllm\\nrich==15.0.0\\n    # via\\n    #   rich-toolkit\\n    #   typer\\nrich-toolkit==0.20.5\\n    # via\\n    #   fastapi-cli\\n    #   fastapi-cloud-cli\\nrignore==0.8.1\\n    # via fastapi-cloud-cli\\nrpds-py==2026.9.1\\n    # via\\n    #   jsonschema\\n    #   referencing\\nsafetensors==0.8.0\\n    # via\\n    #   humming-kernels\\n    #   transformers\\n    #   vllm\\nsentencepiece==0.2.2\\n    # via vllm\\nsentry-sdk==2.71.0\\n    # via fastapi-cloud-cli\\nsetproctitle==1.3.8\\n    # via vllm\\nsetuptools==80.10.2\\n    # via\\n    #   model-hosting-container-standards\\n    #   torch\\n    #   vllm\\nshellingham==1.5.4\\n    # via typer\\nsix==1.17.0\\n    # via vllm\\nsniffio==1.3.1\\n    # via\\n    #   anthropic\\n    #   openai\\nsse-starlette==3.5.0\\n    # via mcp\\nstarlette==1.7.0\\n    # via\\n    #   fastapi\\n    #   mcp\\n    #   model-hosting-container-standards\\n    #   prometheus-fastapi-instrumentator\\n    #   sse-starlette\\n    #   vllm\\nsupervisor==4.3.0\\n    # via model-hosting-container-standards\\nsympy==1.14.0\\n    # via torch\\ntabulate==0.10.0\\n    # via\\n    #   flashinfer-python\\n    #   humming-kernels\\ntiktoken==0.14.0\\n    # via\\n    #   mistral-common\\n    #   vllm\\ntilelang==0.1.12\\n    # via vllm\\ntokenizers==0.23.2\\n    # via\\n    #   transformers\\n    #   vllm\\ntokenspeed-mla==0.1.8\\n    # via vllm\\ntokenspeed-triton==3.8.10.post20260920\\n    # via tokenspeed-mla\\ntorch==2.13.0+cu129\\n    # via\\n    #   compressed-tensors\\n    #   flashinfer-python\\n    #   humming-kernels\\n    #   instanttensor\\n    #   quack-kernels\\n    #   tilelang\\n    #   tokenspeed-mla\\n    #   torch-c-dlpack-ext\\n    #   torchvision\\n    #   vllm\\n    #   xgrammar\\ntorch-c-dlpack-ext==0.1.5\\n    # via\\n    #   quack-kernels\\n    #   tilelang\\ntorchaudio==2.11.0+cu129\\n    # via vllm\\ntorchcodec==0.16.0+cu129\\n    # via vllm\\ntorchvision==0.28.0+cu129\\n    # via vllm\\ntqdm==4.70.1\\n    # via\\n    #   flashinfer-python\\n    #   huggingface-hub\\n    #   humming-kernels\\n    #   tilelang\\n    #   transformers\\n    #   vllm\\ntransformers==5.17.0\\n    # via\\n    #   compressed-tensors\\n    #   vllm\\n    #   xgrammar\\ntriton==3.7.1\\n    # via\\n    #   humming-kernels\\n    #   torch\\n    #   xgrammar\\ntruststore==0.10.4\\n    # via\\n    #   httpcore2\\n    #   httpx2\\ntyper==0.27.3\\n    # via\\n    #   fastapi-cli\\n    #   fastapi-cloud-cli\\n    #   fastsafetensors\\n    #   transformers\\ntyping-extensions==4.16.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\n    #   anthropic\\n    #   anyio\\n    #   apache-tvm-ffi\\n    #   cuda-tile\\n    #   fastapi\\n    #   grpcio\\n    #   httpx2\\n    #   huggingface-hub\\n    #   mcp\\n    #   mcp-types\\n    #   mistral-common\\n    #   nvidia-cutlass-dsl-libs-base\\n    #   nvidia-cutlass-dsl-libs-core\\n    #   nvidia-cutlass-dsl-libs-cu12\\n    #   nvidia-cutlass-dsl-libs-cu13\\n    #   openai\\n    #   opentelemetry-api\\n    #   opentelemetry-exporter-otlp-proto-grpc\\n    #   opentelemetry-exporter-otlp-proto-http\\n    #   opentelemetry-sdk\\n    #   opentelemetry-semantic-conventions\\n    #   pydantic\\n    #   pydantic-core\\n    #   pydantic-extra-types\\n    #   referencing\\n    #   rich-toolkit\\n    #   starlette\\n    #   tilelang\\n    #   torch\\n    #   typing-inspection\\n    #   vllm\\n    #   xgrammar\\ntyping-inspection==0.4.4\\n    # via\\n    #   fastapi\\n    #   mcp\\n    #   pydantic\\n    #   pydantic-settings\\nurllib3==2.8.0\\n    # via\\n    #   requests\\n    #   sentry-sdk\\nuvicorn==0.54.0\\n    # via\\n    #   fastapi\\n    #   fastapi-cli\\n    #   fastapi-cloud-cli\\n    #   mcp\\nuvloop==0.23.0\\n    # via uvicorn\\nvllm @ https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl\\n    # via -r notebooks/vllm-requirements.in\\nwatchfiles==1.3.0\\n    # via\\n    #   uvicorn\\n    #   vllm\\nwebsockets==17.2\\n    # via uvicorn\\nxgrammar==0.2.7\\n    # via vllm\\nyarl==1.25.1\\n    # via aiohttp\\nz3-solver==4.15.4.0\\n    # via tilelang\\n"], "notebooks/vision-requirements.txt": ["e480b45e5a9fb62b9b327e148c5edd33b048fd01e20e09fc36e8b16e4817f7ad", "# This file was autogenerated by uv via the following command:\\n#    uv pip compile notebooks/vision-requirements.in -o notebooks/vision-requirements.txt --python-version 3.12 --python-platform x86_64-manylinux_2_35 --torch-backend cu129 --index-strategy unsafe-best-match\\naiohappyeyeballs==2.7.1\\n    # via aiohttp\\naiohttp==3.14.4\\n    # via tritonclient\\naiosignal==1.4.0\\n    # via aiohttp\\nannotated-doc==0.0.5\\n    # via typer\\nanyio==4.15.1\\n    # via httpx\\nattrs==26.1.0\\n    # via aiohttp\\nbrotli==1.2.0\\n    # via geventhttpclient\\ncertifi==2026.7.22\\n    # via\\n    #   geventhttpclient\\n    #   httpcore\\n    #   httpx\\n    #   requests\\ncharset-normalizer==3.5.2\\n    # via requests\\ncolorama==0.4.6\\n    # via onnxslim\\ncontourpy==1.3.3\\n    # via matplotlib\\ncuda-bindings==12.9.9\\n    # via torch\\ncuda-pathfinder==1.8.3\\n    # via cuda-bindings\\ncuda-toolkit==12.9.1\\n    # via torch\\ncycler==0.12.1\\n    # via matplotlib\\nfilelock==4.0.12\\n    # via torch\\nflatbuffers==25.12.19\\n    # via onnxruntime-gpu\\nfonttools==4.66.1\\n    # via matplotlib\\nfrozenlist==1.8.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\nfsspec==2026.9.0\\n    # via torch\\ngevent==26.9.0\\n    # via geventhttpclient\\ngeventhttpclient==2.6.0\\n    # via tritonclient\\ngreenlet==3.3.2\\n    # via\\n    #   gevent\\n    #   tritonclient\\ngrpcio==1.84.0\\n    # via\\n    #   nvidia-pytriton\\n    #   tritonclient\\nh11==0.16.0\\n    # via httpcore\\nhttpcore==1.0.9\\n    # via httpx\\nhttpx==0.28.1\\n    # via\\n    #   -r notebooks/vision-requirements.in\\n    #   ultralytics-platform\\nidna==3.20\\n    # via\\n    #   anyio\\n    #   httpx\\n    #   requests\\n    #   yarl\\nimportlib-metadata==9.0.1\\n    # via nvidia-pytriton\\njinja2==3.1.6\\n    # via torch\\nkiwisolver==1.5.1\\n    # via matplotlib\\nmarkdown-it-py==4.2.0\\n    # via rich\\nmarkupsafe==3.0.4\\n    # via jinja2\\nmatplotlib==3.11.2\\n    # via ultralytics\\nmdurl==0.1.2\\n    # via markdown-it-py\\nml-dtypes==0.5.4\\n    # via\\n    #   onnx\\n    #   onnxslim\\n    #   tritonclient\\nmpmath==1.3.0\\n    # via sympy\\nmultidict==6.9.1\\n    # via\\n    #   aiohttp\\n    #   yarl\\nmypy-extensions==1.1.0\\n    # via typing-inspect\\nnetworkx==3.7\\n    # via torch\\nnumpy==1.26.4\\n    # via\\n    #   contourpy\\n    #   matplotlib\\n    #   ml-dtypes\\n    #   nvidia-pytriton\\n    #   onnx\\n    #   onnxruntime-gpu\\n    #   opencv-python\\n    #   torchvision\\n    #   tritonclient\\n    #   ultralytics\\n    #   ultralytics-thop\\nnvidia-cublas-cu12==12.9.1.4\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cudnn-cu12\\n    #   nvidia-cusolver-cu12\\nnvidia-cuda-cupti-cu12==12.9.79\\n    # via cuda-toolkit\\nnvidia-cuda-nvrtc-cu12==12.9.86\\n    # via cuda-toolkit\\nnvidia-cuda-runtime-cu12==12.9.79\\n    # via cuda-toolkit\\nnvidia-cudnn-cu12==9.20.0.48\\n    # via torch\\nnvidia-cufft-cu12==11.4.1.4\\n    # via cuda-toolkit\\nnvidia-cufile-cu12==1.14.1.1\\n    # via cuda-toolkit\\nnvidia-curand-cu12==10.3.10.19\\n    # via cuda-toolkit\\nnvidia-cusolver-cu12==11.7.5.82\\n    # via cuda-toolkit\\nnvidia-cusparse-cu12==12.5.10.65\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cusolver-cu12\\nnvidia-cusparselt-cu12==0.8.1\\n    # via torch\\nnvidia-ml-py==13.615.71\\n    # via ultralytics\\nnvidia-nccl-cu12==2.29.7\\n    # via torch\\nnvidia-nvjitlink-cu12==12.9.86\\n    # via\\n    #   cuda-toolkit\\n    #   nvidia-cufft-cu12\\n    #   nvidia-cusolver-cu12\\n    #   nvidia-cusparse-cu12\\nnvidia-nvshmem-cu12==3.4.5\\n    # via torch\\nnvidia-nvtx-cu12==12.9.79\\n    # via cuda-toolkit\\nnvidia-pytriton==0.7.0\\n    # via -r notebooks/vision-requirements.in\\nonnx==1.23.2\\n    # via\\n    #   -r notebooks/vision-requirements.in\\n    #   onnxslim\\nonnxruntime-gpu==1.26.0\\n    # via -r notebooks/vision-requirements.in\\nonnxslim==0.1.98\\n    # via -r notebooks/vision-requirements.in\\nopencv-python==4.11.0.86\\n    # via ultralytics\\npackaging==26.3\\n    # via\\n    #   matplotlib\\n    #   onnxruntime-gpu\\n    #   onnxslim\\n    #   tritonclient\\npillow==12.3.0\\n    # via\\n    #   matplotlib\\n    #   torchvision\\n    #   ultralytics\\npolars==2.0.0\\n    # via ultralytics\\npolars-runtime-32==2.0.0\\n    # via polars\\npropcache==0.5.4\\n    # via\\n    #   aiohttp\\n    #   yarl\\nprotobuf==6.33.6\\n    # via\\n    #   nvidia-pytriton\\n    #   onnx\\n    #   onnxruntime-gpu\\n    #   tritonclient\\npsutil==7.2.2\\n    # via ultralytics\\npygments==2.21.0\\n    # via rich\\npyparsing==3.3.3\\n    # via matplotlib\\npython-dateutil==2.9.0.post0\\n    # via matplotlib\\npython-rapidjson==1.25\\n    # via tritonclient\\npyyaml==6.0.3\\n    # via ultralytics\\npyzmq==27.2.0\\n    # via nvidia-pytriton\\nrequests==2.34.2\\n    # via ultralytics\\nrich==15.0.0\\n    # via typer\\nsetuptools==84.0.0\\n    # via torch\\nsh==2.4.0\\n    # via nvidia-pytriton\\nshellingham==1.5.4\\n    # via typer\\nsix==1.17.0\\n    # via python-dateutil\\nsympy==1.14.0\\n    # via\\n    #   onnxslim\\n    #   torch\\ntensorrt-cu12==10.16.1.11\\n    # via -r notebooks/vision-requirements.in\\ntensorrt-cu12-bindings==10.16.1.11\\n    # via tensorrt-cu12\\ntensorrt-cu12-libs==10.16.1.11\\n    # via tensorrt-cu12\\ntorch==2.13.0+cu129\\n    # via\\n    #   -r notebooks/vision-requirements.in\\n    #   torchvision\\n    #   ultralytics\\n    #   ultralytics-thop\\ntorchvision==0.28.0+cu129\\n    # via\\n    #   -r notebooks/vision-requirements.in\\n    #   ultralytics\\ntriton==3.7.1\\n    # via torch\\ntritonclient==2.73.0\\n    # via nvidia-pytriton\\ntyper==0.27.3\\n    # via nvidia-pytriton\\ntyping-extensions==4.16.0\\n    # via\\n    #   aiohttp\\n    #   aiosignal\\n    #   anyio\\n    #   grpcio\\n    #   onnx\\n    #   torch\\n    #   typing-inspect\\ntyping-inspect==0.9.0\\n    # via nvidia-pytriton\\nultralytics==8.4.174\\n    # via -r notebooks/vision-requirements.in\\nultralytics-platform==0.1.83\\n    # via ultralytics\\nultralytics-thop==2.2.2\\n    # via ultralytics\\nurllib3==2.8.0\\n    # via\\n    #   requests\\n    #   tritonclient\\nwrapt==2.5.0\\n    # via nvidia-pytriton\\nyarl==1.25.1\\n    # via aiohttp\\nzipp==4.1.1\\n    # via importlib-metadata\\nzope-event==6.2\\n    # via gevent\\nzope-interface==8.6\\n    # via gevent\\n"], "benchmark/vision/run_vision.py": ["0e7b195d50255fc12601ffc1028af01fac8d347392986db6fab4d33d8acc27ae", "\\"\\"\\"Closed-loop batch benchmark for an image model: ONNX Runtime (CPU or CUDA), TensorRT, or a Triton HTTP endpoint.\\n\\nOne request = one batch through the model, sent as soon as the previous one returns (concurrency 1). Images are\\nletterboxed to --imgsz once up front, so the timings cover the model call including host to device copies and\\nthe output copy back, not JPEG decode, resize or NMS. Writes one raw JSON per (batch, repeat) in the loadgen\\nenvelope (config, env, git commit, per-request records, summary), marked \\"workload\\": \\"vision\\".\\n\\npython benchmark/vision/run_vision.py --backend ort-cpu --model yolov8n.onnx --images /tmp/coco500/images/val2017 \\\\\\n  --label vision-cpu-yolov8n-onnx-fp32 --precision fp32 --batch 1 8 --repeats 3 --out-dir benchmark/results/raw\\n\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nimport platform\\nimport shutil\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom PIL import Image\\n\\nHERE = Path(__file__).parent\\nsys.path.insert(0, str(HERE.parent / \\"loadgen\\"))\\nfrom metrics import dist  # noqa: E402\\nfrom run import GpuSampler, cpu_model, git_commit, sh, utcnow  # noqa: E402\\n\\n\\ndef letterbox(path, size):\\n    \\"\\"\\"RGB float32 CHW in [0, 1], resized to fit size x size with aspect kept, padded with 114 (Ultralytics\' value).\\"\\"\\"\\n    im = Image.open(path).convert(\\"RGB\\")\\n    s = size / max(im.size)\\n    im = im.resize(\\n        (max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BILINEAR\\n    )\\n    canvas = Image.new(\\"RGB\\", (size, size), (114, 114, 114))\\n    canvas.paste(im, ((size - im.width) // 2, (size - im.height) // 2))\\n    return np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1) / 255.0\\n\\n\\nclass OrtModel:\\n    def __init__(self, path, device, threads):\\n        import onnxruntime as ort\\n\\n        so = ort.SessionOptions()\\n        if threads:\\n            so.intra_op_num_threads = threads\\n        if device == \\"cuda\\":\\n            ort.preload_dlls()  # CUDA and cuDNN from the nvidia-* wheels in the venv\\n            providers = [\\"CUDAExecutionProvider\\", \\"CPUExecutionProvider\\"]\\n        else:\\n            providers = [\\"CPUExecutionProvider\\"]\\n        self.sess = ort.InferenceSession(path, so, providers=providers)\\n        active = self.sess.get_providers()\\n        if device == \\"cuda\\" and active[0] != \\"CUDAExecutionProvider\\":\\n            raise RuntimeError(\\n                f\\"CUDA requested but ONNX Runtime runs on {active}\\"\\n            )  # no silent CPU fallback\\n        self.inp = self.sess.get_inputs()[0].name\\n        self.info = {\\n            \\"onnxruntime\\": ort.__version__,\\n            \\"providers\\": active,\\n            \\"provider_options\\": self.sess.get_provider_options(),\\n            \\"intra_op_num_threads\\": so.intra_op_num_threads,\\n        }\\n\\n    def __call__(self, x):\\n        return self.sess.run(None, {self.inp: x})\\n\\n\\nclass TrtModel:\\n    \\"\\"\\"TensorRT engine runner with torch CUDA tensors as buffers. Accepts Ultralytics engine files (4-byte length +\\n    JSON metadata before the plan) and plain plans.\\"\\"\\"\\n\\n    def __init__(self, path):\\n        import tensorrt as trt\\n        import torch\\n\\n        data = Path(path).read_bytes()\\n        n = int.from_bytes(data[:4], \\"little\\", signed=True)\\n        try:\\n            self.meta = json.loads(data[4 : 4 + n])\\n            data = data[4 + n :]\\n        except (ValueError, UnicodeDecodeError):\\n            self.meta = None\\n        self.engine = trt.Runtime(\\n            trt.Logger(trt.Logger.WARNING)\\n        ).deserialize_cuda_engine(data)\\n        if self.engine is None:\\n            raise RuntimeError(f\\"TensorRT could not deserialize {path}\\")\\n        self.ctx = self.engine.create_execution_context()\\n        names = [\\n            self.engine.get_tensor_name(i) for i in range(self.engine.num_io_tensors)\\n        ]\\n        self.inp = next(\\n            n for n in names if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT\\n        )\\n        self.outs = [\\n            n\\n            for n in names\\n            if self.engine.get_tensor_mode(n) == trt.TensorIOMode.OUTPUT\\n        ]\\n        dtypes = {trt.float32: torch.float32, trt.float16: torch.float16}\\n        self.dtype = {n: dtypes[self.engine.get_tensor_dtype(n)] for n in names}\\n        self.torch = torch\\n        self.info = {\\n            \\"tensorrt\\": trt.__version__,\\n            \\"io\\": {\\n                n: [\\n                    str(self.engine.get_tensor_dtype(n)),\\n                    list(self.engine.get_tensor_shape(n)),\\n                ]\\n                for n in names\\n            },\\n            \\"metadata\\": self.meta,\\n        }\\n\\n    def __call__(self, x):\\n        torch = self.torch\\n        t = torch.from_numpy(x).to(\\"cuda\\", self.dtype[self.inp])\\n        self.ctx.set_input_shape(self.inp, tuple(t.shape))\\n        self.ctx.set_tensor_address(self.inp, t.data_ptr())\\n        outs = []\\n        for n in self.outs:\\n            o = torch.empty(\\n                tuple(self.ctx.get_tensor_shape(n)), dtype=self.dtype[n], device=\\"cuda\\"\\n            )\\n            self.ctx.set_tensor_address(n, o.data_ptr())\\n            outs.append(o)\\n        if not self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream):\\n            raise RuntimeError(\\"execute_async_v3 returned False\\")\\n        return [o.cpu().numpy() for o in outs]  # .cpu() synchronizes\\n\\n\\nTRITON_SERVING = \\"Python backend via PyTriton running the TensorRT engine; not Triton\'s native TensorRT backend\\"\\n\\n\\nclass TritonModel:\\n    def __init__(self, url, name):\\n        import tritonclient.http as th\\n\\n        self.th, self.name = th, name\\n        self.client = th.InferenceServerClient(url)\\n        md = self.client.get_model_metadata(name)\\n        self.inp = md[\\"inputs\\"][0][\\"name\\"]\\n        self.outs = [o[\\"name\\"] for o in md[\\"outputs\\"]]\\n        import tensorrt\\n\\n        self.info = {\\n            \\"triton_server\\": self.client.get_server_metadata(),\\n            \\"model_metadata\\": md,\\n            # inference/vision/triton_serve.py runs in this same venv, so this is the server\'s TensorRT\\n            \\"tensorrt\\": tensorrt.__version__,\\n            \\"serving\\": TRITON_SERVING,\\n        }\\n\\n    def __call__(self, x):\\n        th = self.th\\n        i = th.InferInput(self.inp, list(x.shape), \\"FP32\\")\\n        i.set_data_from_numpy(x, binary_data=True)\\n        r = self.client.infer(\\n            self.name,\\n            [i],\\n            outputs=[th.InferRequestedOutput(o, binary_data=True) for o in self.outs],\\n        )\\n        return [r.as_numpy(o) for o in self.outs]\\n\\n\\ndef load_model(args):\\n    if args.backend in (\\"ort-cpu\\", \\"ort-cuda\\"):\\n        return OrtModel(args.model, args.backend.removeprefix(\\"ort-\\"), args.threads)\\n    if args.backend == \\"trt\\":\\n        return TrtModel(args.model)\\n    return TritonModel(args.triton_url, args.model)\\n\\n\\ndef summarize(records):\\n    \\"\\"\\"Summary over measured (non-warmup) batch requests. Latency is per batch, in ms.\\"\\"\\"\\n    measured = [r for r in records if not r[\\"warmup\\"]]\\n    ok = [r for r in measured if not r[\\"error\\"]]\\n    wall = (\\n        (max(r[\\"end\\"] for r in measured) - min(r[\\"start\\"] for r in measured))\\n        if measured\\n        else 0\\n    )\\n    images = sum(r[\\"batch\\"] for r in ok)\\n    return {\\n        \\"requests\\": len(measured),\\n        \\"ok\\": len(ok),\\n        \\"errors\\": len(measured) - len(ok),\\n        \\"error_rate\\": (len(measured) - len(ok)) / len(measured) if measured else None,\\n        \\"wall_s\\": wall,\\n        \\"images\\": images,\\n        \\"images_s\\": images / wall if wall > 0 else None,\\n        \\"latency_ms\\": dist([(r[\\"end\\"] - r[\\"start\\"]) * 1000 for r in ok]),\\n    }\\n\\n\\ndef run_once(model, batches, n_warm, n, batch, t0):\\n    \\"\\"\\"n_warm + n requests; request i uses batch slice i (cycling through the preloaded images).\\"\\"\\"\\n    records = []\\n    for i in range(n_warm + n):\\n        x = batches[i % len(batches)]\\n        rec = {\\"i\\": i, \\"batch\\": batch, \\"warmup\\": i < n_warm, \\"error\\": None}\\n        rec[\\"start\\"] = round(time.perf_counter() - t0, 6)\\n        try:\\n            model(x)\\n        except Exception as e:  # noqa: BLE001 recorded, never raised: a failed request is data\\n            rec[\\"error\\"] = f\\"{type(e).__name__}: {e}\\"\\n        rec[\\"end\\"] = round(time.perf_counter() - t0, 6)\\n        records.append(rec)\\n    return records\\n\\n\\ndef scrape(url):\\n    if not url:\\n        return None\\n    import urllib.request\\n\\n    try:\\n        with urllib.request.urlopen(url, timeout=10) as r:\\n            return [\\n                ln\\n                for ln in r.read().decode().splitlines()\\n                if ln and not ln.startswith(\\"#\\")\\n            ]\\n    except OSError as e:\\n        return f\\"scrape failed: {e}\\"\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\n        \\"--backend\\", required=True, choices=[\\"ort-cpu\\", \\"ort-cuda\\", \\"trt\\", \\"triton\\"]\\n    )\\n    p.add_argument(\\n        \\"--model\\", required=True, help=\\"ONNX or engine file; for triton the model name\\"\\n    )\\n    p.add_argument(\\n        \\"--model-file\\",\\n        default=None,\\n        help=\\"triton: the engine file behind the model, for its sha256\\",\\n    )\\n    p.add_argument(\\"--triton-url\\", default=\\"127.0.0.1:8000\\")\\n    p.add_argument(\\n        \\"--metrics-url\\",\\n        default=None,\\n        help=\\"Prometheus /metrics to snapshot before and after\\",\\n    )\\n    p.add_argument(\\n        \\"--images\\",\\n        required=True,\\n        help=\\"directory of .jpg files (sorted, first --num-images used)\\",\\n    )\\n    p.add_argument(\\"--num-images\\", type=int, default=64)\\n    p.add_argument(\\"--imgsz\\", type=int, default=640)\\n    p.add_argument(\\"--batch\\", type=int, nargs=\\"+\\", default=[1])\\n    p.add_argument(\\"--repeats\\", type=int, default=3)\\n    p.add_argument(\\n        \\"--images-per-run\\",\\n        type=int,\\n        default=640,\\n        help=\\"measured requests = max(20, ceil(this / batch))\\",\\n    )\\n    p.add_argument(\\"--warmup\\", type=int, default=10, help=\\"warmup requests per run\\")\\n    p.add_argument(\\n        \\"--threads\\",\\n        type=int,\\n        default=0,\\n        help=\\"ONNX Runtime intra-op threads (0 = its default)\\",\\n    )\\n    p.add_argument(\\"--label\\", required=True)\\n    p.add_argument(\\"--precision\\", required=True)\\n    p.add_argument(\\"--hardware\\", default=None)\\n    p.add_argument(\\n        \\"--server-args\\", default=None, help=\\"server command line, stored verbatim\\"\\n    )\\n    p.add_argument(\\"--git-commit\\", default=None)\\n    p.add_argument(\\"--notes\\", default=None)\\n    p.add_argument(\\n        \\"--smoke\\",\\n        action=\\"store_true\\",\\n        help=\\"mark as smoke run (excluded from RESULTS.md)\\",\\n    )\\n    p.add_argument(\\"--out-dir\\", required=True)\\n    args = p.parse_args()\\n\\n    files = sorted(Path(args.images).glob(\\"*.jpg\\"))[: args.num_images]\\n    if not files:\\n        raise SystemExit(f\\"no .jpg files in {args.images}\\")\\n    imgs = np.stack([letterbox(f, args.imgsz) for f in files])\\n    model_file = args.model_file or (args.model if args.backend != \\"triton\\" else None)\\n    model = load_model(args)\\n    env = {\\n        \\"hardware\\": args.hardware,\\n        \\"cpu_model\\": cpu_model(),\\n        \\"cpu_count\\": os.cpu_count(),\\n        \\"platform\\": platform.platform(),\\n        \\"python\\": platform.python_version(),\\n        \\"numpy\\": np.__version__,\\n        \\"git_commit\\": args.git_commit or git_commit(),\\n        \\"nvidia_smi\\": sh(\\n            \\"nvidia-smi\\",\\n            \\"--query-gpu=name,driver_version,memory.total\\",\\n            \\"--format=csv,noheader\\",\\n        )\\n        if shutil.which(\\"nvidia-smi\\")\\n        else None,\\n        \\"backend\\": model.info,\\n    }\\n    out_dir = Path(args.out_dir)\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    for batch in args.batch:\\n        # Contiguous batches cycling through the preloaded images; built once so the loop only times the model.\\n        batches = [\\n            np.ascontiguousarray(imgs[np.arange(i, i + batch) % len(imgs)])\\n            for i in range(0, len(imgs), batch)\\n        ]\\n        n = max(20, math.ceil(args.images_per_run / batch))\\n        for rep in range(1, args.repeats + 1):\\n            t0 = time.perf_counter()\\n            started = utcnow()\\n            warm = run_once(model, batches, args.warmup, 0, batch, t0)\\n            before = scrape(args.metrics_url)\\n            gpu = GpuSampler()\\n            measured = run_once(model, batches, 0, n, batch, t0)\\n            gpu_samples = gpu.stop()\\n            res = {\\n                \\"schema\\": 1,\\n                \\"workload\\": \\"vision\\",\\n                \\"smoke\\": args.smoke,\\n                \\"label\\": args.label,\\n                \\"engine\\": args.backend,\\n                \\"engine_version\\": model.info[\\"triton_server\\"].get(\\"version\\")\\n                if args.backend == \\"triton\\"\\n                else model.info.get(\\"onnxruntime\\") or model.info.get(\\"tensorrt\\"),\\n                \\"model\\": Path(model_file).name if model_file else args.model,\\n                \\"model_sha256\\": hashlib.sha256(\\n                    Path(model_file).read_bytes()\\n                ).hexdigest()\\n                if model_file\\n                else None,\\n                \\"precision\\": args.precision,\\n                \\"batch\\": batch,\\n                \\"concurrency\\": 1,\\n                \\"repeat\\": rep,\\n                \\"started_utc\\": started,\\n                \\"finished_utc\\": utcnow(),\\n                \\"config\\": {\\n                    \\"images_dir\\": Path(args.images).name,\\n                    \\"images\\": [f.name for f in files],\\n                    \\"imgsz\\": args.imgsz,\\n                    \\"requests\\": n,\\n                    \\"warmup_requests\\": args.warmup,\\n                    \\"timed\\": \\"model call incl. host-device copies; excludes decode, letterbox, NMS\\",\\n                    \\"notes\\": args.notes,\\n                },\\n                \\"env\\": env,\\n                \\"server_args\\": args.server_args,\\n                \\"gpu_samples\\": gpu_samples,\\n                \\"server_metrics_before\\": before,\\n                \\"server_metrics_after\\": scrape(args.metrics_url),\\n                \\"summary\\": summarize(warm + measured),\\n                \\"requests\\": warm + measured,\\n            }\\n            stamp = started.replace(\\":\\", \\"\\").replace(\\"-\\", \\"\\").replace(\\"+0000\\", \\"Z\\")\\n            path = out_dir / f\\"{args.label}-b{batch}-r{rep}-{stamp}.json\\"\\n            path.write_text(\\n                json.dumps(res, separators=(\\",\\", \\":\\")) + \\"\\\\n\\",\\n                encoding=\\"utf-8\\",\\n                newline=\\"\\\\n\\",\\n            )\\n            s = res[\\"summary\\"]\\n            print(\\n                f\\"{path.name}: ok {s[\'ok\']}/{s[\'requests\']} images/s {s[\'images_s\'] or 0:.1f} \\"\\n                f\\"p50 {s[\'latency_ms\'][\'p50\'] or 0:.2f} ms p99 {s[\'latency_ms\'][\'p99\'] or 0:.2f} ms\\",\\n                flush=True,\\n            )\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"], "benchmark/vision/fetch_coco_subset.py": ["e87093ceed80ce3cfe941a129df60e5cc6ba11edd7d728770d57ff353e16e930", "\\"\\"\\"Fetch the fixed COCO val2017 subset for the vision benchmark: the first 500 val2017 image names in sorted order,\\nwith YOLO-format box labels from Ultralytics\' coco2017labels.zip. Stdlib only.\\n\\nEvery image is checked against coco_val2017_first500.txt (name, sha256) and the labels zip against LABELS_SHA256.\\nThe script also checks that the manifest really is the first 500 sorted names of the zip\'s val2017.txt.\\n\\npython benchmark/vision/fetch_coco_subset.py --out /tmp/coco500 [--limit 5]\\n\\nWrites <out>/images/val2017/*.jpg, <out>/labels/val2017/*.txt (Ultralytics layout; images without objects have no\\nlabel file) and <out>/val.txt (image list for an Ultralytics dataset YAML). Downloads are cached in <out>.\\nCOCO annotations are CC BY 4.0; images keep their Flickr licenses. Nothing from COCO is committed to the repo.\\n\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport time\\nimport urllib.request\\nimport zipfile\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\n\\nHERE = Path(__file__).parent\\nMANIFEST = HERE / \\"coco_val2017_first500.txt\\"\\nLABELS_URL = (\\n    \\"https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip\\"\\n)\\nLABELS_SHA256 = \\"51a5175c894a7a1010f90eb4cba613473445f02633b684ed46c0292a997d0234\\"\\nIMAGE_URL = \\"http://images.cocodataset.org/val2017/{}\\"\\n\\n\\ndef sha256(data):\\n    return hashlib.sha256(data).hexdigest()\\n\\n\\ndef download(url, tries=4):\\n    for i in range(tries):\\n        try:\\n            with urllib.request.urlopen(url, timeout=60) as r:\\n                return r.read()\\n        except OSError:\\n            if i == tries - 1:\\n                raise\\n            time.sleep(2**i)\\n    raise AssertionError(\\"unreachable\\")\\n\\n\\ndef cached(path, url, sha):\\n    \\"\\"\\"Bytes of path if it matches sha, else download url, check sha, write path.\\"\\"\\"\\n    if path.exists() and sha256(path.read_bytes()) == sha:\\n        return path.read_bytes()\\n    data = download(url)\\n    if sha256(data) != sha:\\n        raise ValueError(f\\"{url}: sha256 {sha256(data)} != pinned {sha}\\")\\n    path.write_bytes(data)\\n    return data\\n\\n\\ndef manifest():\\n    return [\\n        ln.split()\\n        for ln in MANIFEST.read_text(encoding=\\"utf-8\\").splitlines()\\n        if ln.strip() and not ln.startswith(\\"#\\")\\n    ]\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\"--out\\", required=True)\\n    p.add_argument(\\n        \\"--limit\\",\\n        type=int,\\n        default=0,\\n        help=\\"first N manifest images only (0 = all 500)\\",\\n    )\\n    args = p.parse_args()\\n    out = Path(args.out)\\n    img_dir, lbl_dir = out / \\"images\\" / \\"val2017\\", out / \\"labels\\" / \\"val2017\\"\\n    img_dir.mkdir(parents=True, exist_ok=True)\\n    lbl_dir.mkdir(parents=True, exist_ok=True)\\n    full = manifest()\\n    entries = full[: args.limit or None]\\n\\n    cached(out / \\"coco2017labels.zip\\", LABELS_URL, LABELS_SHA256)\\n    with zipfile.ZipFile(out / \\"coco2017labels.zip\\") as z:\\n        val = sorted(Path(x).name for x in z.read(\\"coco/val2017.txt\\").decode().split())\\n        if val[: len(full)] != [n for n, _ in full]:\\n            raise ValueError(\\n                \\"manifest is not the first N sorted names of coco/val2017.txt\\"\\n            )\\n        names = set(z.namelist())\\n        instances = 0\\n        for name, _ in entries:\\n            member = f\\"coco/labels/val2017/{Path(name).stem}.txt\\"\\n            if member in names:\\n                text = z.read(member)\\n                instances += len(text.splitlines())\\n                (lbl_dir / f\\"{Path(name).stem}.txt\\").write_bytes(text)\\n\\n    with ThreadPoolExecutor(8) as ex:\\n        list(\\n            ex.map(\\n                lambda e: cached(img_dir / e[0], IMAGE_URL.format(e[0]), e[1]), entries\\n            )\\n        )\\n    (out / \\"val.txt\\").write_text(\\n        \\"\\".join(f\\"./images/val2017/{n}\\\\n\\" for n, _ in entries), encoding=\\"utf-8\\"\\n    )\\n    labelled = sum(1 for n, _ in entries if (lbl_dir / f\\"{Path(n).stem}.txt\\").exists())\\n    print(\\n        f\\"{len(entries)} images (sha256 ok), {labelled} with labels, {instances} boxes -> {out}\\"\\n    )\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"], "benchmark/vision/coco_val2017_first500.txt": ["2e49efde9e4b16ea53a3978943c78ad7fad01c2ab9503b820017413871ed0f4f", "# COCO val2017 subset for the vision benchmark: the first 500 image names of val2017 in sorted order\\n# (coco/val2017.txt in Ultralytics coco2017labels.zip, sha256 51a5175c...), sha256 of each file as served by\\n# http://images.cocodataset.org/val2017/ on 2026-10-09. Checked by fetch_coco_subset.py.\\n000000000139.jpg ffe0f0cec3b2e27aab1967229cdf0a0d7751dcdd5800322f0b8ac0dffb3b8a8d\\n000000000285.jpg f3a2974ce3686332609124c70e3e6a2e3aca43fccf1cd1bd7c5c03820977f57d\\n000000000632.jpg a4cd7f45ac1ce27eaafb254b23af7c0b18a064be08870ceaaf03b2147f2ce550\\n000000000724.jpg 5c0e559c75d3969c8e3e297b61f61063f78045c9d4802b526ba616361f3823fd\\n000000000776.jpg 1dd31e9059c491992be2f562624eb4093e17aee08b4f7baf5ff9ea24543b0a33\\n000000000785.jpg 83981537a7baeafbeb9c8cb67b3484dc26433f574b3685d021fa537e277e4726\\n000000000802.jpg d5b79e7fa716f85ca86f46e5f518da9b6c5e26414925624d76e6476861aec495\\n000000000872.jpg c2aa138ee3a59b057a7ba6fc5a6a18e62af531aa7dab78a7bfd33c1cd7e55eb6\\n000000000885.jpg 1c67b783f78b18ddb5dd20aaba4d4aacaf98e5277051688a96da096afa9cdf1f\\n000000001000.jpg 24bb77a31928404e45a0454b06f6a0bd54a8db103590c9d7917288a1e0269f05\\n000000001268.jpg 8e85a2e71ee512fe748a3a3be9bb40b8591e8c501c0f3896efcb5be5999ec236\\n000000001296.jpg 34a98da7c11bc8811e6eec445145bb16f3d5f4338945ba116edcc1fe554d181b\\n000000001353.jpg 1cccef217a1e5eec2acc74d7c08a105da3ef6fdb8514e221cdea9b0b373b60ea\\n000000001425.jpg 71a37c25cdf944cf0a88b66d00a66cf899a88bd55c7ac80fcf1c76329d617e6c\\n000000001490.jpg a044804a0b0186a1267acae34975932f14ede96c4a2b78fcee74e6c2fbd8414d\\n000000001503.jpg 06c721f0a62bcb9fcb2d2d0cc0b9edd63092084a8773a18c4b5a7bd6fb4f095d\\n000000001532.jpg 974f2af6d81f65627a680ea9f42bdee1067a0ddcdb9f8e7a945bacd21d4dc2ce\\n000000001584.jpg 77131e2f74aa7e374f951a69157cf7f51e00201f81bacfa8a9ab7309b245553f\\n000000001675.jpg 6ba641627c08ef424b7dc3e6cee069aa0dd49615a31c54ae0de1ecd1fabd9dea\\n000000001761.jpg c06b1845c54bb886a8fb9f8a6e908b29ceb661ffda1ca8df9a27f123806de648\\n000000001818.jpg e638b349929d13bb27e148f4303a576e0a964ef81c0907d36748e0adb822a8e6\\n000000001993.jpg 039f3bfb862023bde7f835a36c32d8f59a3e66488863c2a47b7b746e78c44186\\n000000002006.jpg eda4af13818b566f8fa4f479e76286ad90cd5da2e8dfd530be0ab3637e1482ed\\n000000002149.jpg 3ef0da375244c3fdbb45bcfe5689e8e46fb616c00250f9dce6aa34d301801ac5\\n000000002153.jpg 8795ff243cf20405bd442631d5d1cb7e60665dd459c8d57f5764333808dd49fd\\n000000002157.jpg 1a038609e5ceb0719105b1438d257c3d588e45384cb7ffe77b137e4d7e3ed477\\n000000002261.jpg 1a0ad595a5675bdc7a5e95c04def56109c00d61168fd826158322fd538d70279\\n000000002299.jpg d53d17dd8ecded4ae25555a0b8e193445846c7db7a6199c3ea35b09b3021f99e\\n000000002431.jpg fa0ce7b066c2fe7bd0892aa2c6361943148db9766de60241e69d8026a18e05ad\\n000000002473.jpg 2f1bfa7634a64f7999ba5488ec904ad1f85bbc63d0fe4b9fe35f52abbf8c5f4f\\n000000002532.jpg fc017f5308d348bdb6011f2312b10853ccd780a4a7369f48774ef67f499f2e77\\n000000002587.jpg bc37072a4f32bc99d6b02a684dd9bc69bd63567382315e5f497c9bb4a44c7208\\n000000002592.jpg ffa3d472d44184b1f48878f7b097189723e57a24f36c44c29e4bc561110fbf41\\n000000002685.jpg 2430b9bffb5ff52dfbaf7b922ac8018547e233a5996a6b017bd1a206a46c5bdb\\n000000002923.jpg 46d2a5feb3f953b59316daee47b45332f07ebde1f88eb361f29b35c1fbbb62e0\\n000000003156.jpg 3c929365d99581f03b70a810819df4781ac024d88dec7e592229cbea7489a779\\n000000003255.jpg c075f51a798e65d23e1c65a3fcc7736aeae4e03fbdd126f6d6b82cb07957d28b\\n000000003501.jpg 9615e4dc9099e860748ce3f839ed077fa6a1fa39a10b909c00600bf4e2b3f392\\n000000003553.jpg 4286197d2b6fc18809a37a4ef3035109509fcf52782fb83894835faaa5743bea\\n000000003661.jpg 8746b068c166782f4b4fb7e6be9d16e5311e9c15bc867e21409b7f5d2eddefb4\\n000000003845.jpg d5c4394c712e5c24586f2531e2adf25d14e398993cc12af561ca760f20881df0\\n000000003934.jpg c7b1180c75a332e8768a4ca76b1fe8b3e3e073e2b925d94f43fa5439535d5dcf\\n000000004134.jpg 2c922f6ec30f9c4eca2b373c8808f63979c26f4d89242754b3576bf78db1108b\\n000000004395.jpg 14ffbd7ab7a01e0dd7fc9676a688bd2c916766e2edb6e1d12ac9ae0ed07b122a\\n000000004495.jpg cf670ca35fb16e59ecce82eecf9bf1f60d2ee333be81b61c4ea390be2c7e7b2b\\n000000004765.jpg 90fab997d1351b1c2ae1a437689245f962dbfa58e577911c9c83991490aadc5e\\n000000004795.jpg 6f8994bac6aef0602d4aa4397ecd4b7d579f5b18f66058a2df021352dd22bdc6\\n000000005001.jpg 99600c56dc25c72dc1cc1d1f2b09aa33b97142560b54d034fa83a38fcf3c5b0f\\n000000005037.jpg b47318f2a6f652027f6435c462fae0567d925f67ab3279d08d5da2f322322dda\\n000000005060.jpg 5d3d8315695f7c718b649c06afe32c792bcce9d4a97f544fc1ba19bc592d8cd3\\n000000005193.jpg 2f0360e89d71b146cbfaee05e934303864fa61d9bf1a1f5cc0c2e508b8191b8d\\n000000005477.jpg 3b3ed6a27e07126c2175d168405a9bffe60b0f03555d0ac4b33ec0788bd745f2\\n000000005503.jpg ff95bf5ddb4bdbd7f4f0f96d963cd941a0a2b38cfcf7832f25113e8441a443c6\\n000000005529.jpg e06b55bf935e76ac31cd8b8a56a3978e2f552d7c967ff8a0b8d0ae3b3445db0e\\n000000005586.jpg d7c4ce5acc9c578c41ff04aa60e610423e304e66b77fff92df501c207460b349\\n000000005600.jpg 37e5625d5de54eb524ad2425d855e1d9d54fcb29c3475dacffc86ab48bcaec39\\n000000005992.jpg c33842fbbd9626843b93ede50dd4908221cb0c9bf9837f58861354eec2a4ec56\\n000000006012.jpg 7cd0d627b15c09f373aa613d013fb2d8ae6bd20bcce0d98aa31b963e6bcca495\\n000000006040.jpg ac74264bbc3356e0cb87cdc2edac2e2f7e519ae2c16b0a4e2b943a695163ebf6\\n000000006213.jpg 0f8ce814489c07920e1f3c3a08842ff832f66340a4487d49de2feb31a45b8132\\n000000006460.jpg 2526fc01ddf5bef5c5e6e1f665d63b6bc95d8ec14eab286dc080e95f9a3642cd\\n000000006471.jpg 67fe4f0a7b71ecee846273d654bb8387c180229dc6c4a3060152aed0a776762f\\n000000006614.jpg 5c4214d0503ea4ad699298cdca84ab631496d58d0ccdc4f6bafba9553fe067c6\\n000000006723.jpg 1d3ba71e9002721a2bc18791f3c86578d6de38fb61640803657276d53e73663f\\n000000006763.jpg e3d8430df8ae6fd8021ca48ce988bcb8a1ae648be8bd3b73b1f0cc9c37f82484\\n000000006771.jpg b2cc81a8ab0c23da44dda4b6526515c68dbcb1abf939a371f750ffe22db15f95\\n000000006818.jpg 99479f61ef2931ea70e161db0859e8706c6e48f13e8cb0798959f52433b01b98\\n000000006894.jpg 8e2059e2f11b0c3051d105255221dd1474ab9a80907de64f0908ae8234e97c8a\\n000000006954.jpg 59471a9bed5c28ac2f3ad56af1799340d6acb90f7c87d1f60360d1a9466e89ea\\n000000007088.jpg 323e51d944faa233acfa36f23598bad7f8bae8107d089949ae792267bac03770\\n000000007108.jpg fe22617b02d85f3d180db14af3c300e95d259ae81035452a03f57b79e056fb78\\n000000007278.jpg b088f1fc8e86da21084e8c98b7dc9406c0da61d74e740925af6a7adeda3c3f34\\n000000007281.jpg 57f3508e600d3370320487dde8665f208f11612a11caac0801c7f5fc19188ac3\\n000000007386.jpg 1bb19f0a2a950a944d178c921fddaddda6f133df6367e9f75ffcaafe2cd43f83\\n000000007511.jpg 105e70c573b87d7e3f034e1dc08cedcced39be4e52197139f52616c920dda3d7\\n000000007574.jpg 4f26c847c2f49574978e96733efaec720dd1d1ba95a8220d0de012b7317eed47\\n000000007784.jpg 2b472f58ae604977c3c288320aed7fa8823043758e1b40b9cede91921d7154eb\\n000000007795.jpg adba86ef9a0dd1557c6cd23a4833d2cec71881a3979ceff831136c367e5f2507\\n000000007816.jpg 00132e042709fc7a66797374a42ff72d493a945d01afe2a48e5a526116a2aabe\\n000000007818.jpg 1bb3ef4577a7c11c55545532c7ec81fe1336105e5f1b9f38e47129eae9365187\\n000000007888.jpg 7c735e89b3699afc510053412ddea8e0370a1a5a2c9096713a5b255567747442\\n000000007977.jpg 0acb919d23df8f5126d63556668272ed26d53fa97fbad958958c2bbe8164d690\\n000000007991.jpg 63c6fae2e3defd4bc1c6f346806aef03eb11ccbd7c6bda3e479d1fbafd486db1\\n000000008021.jpg 235537407867a7b894e78b3aecb55b33b14b64cbb23ca376a00d34c62a1a63a6\\n000000008211.jpg 8a52b7177994a207b4f463d591c3c0f1ffd00b42dcdf17cc8f8984bfae17d110\\n000000008277.jpg 692fc5469af5fffdc0dac05e709a0ac1380dca32d4c57436903b122f3c3558dc\\n000000008532.jpg 237e72d5ffb557b5052b918c01a4fb15e6bb7dd2151bba9c0538de0024c941f9\\n000000008629.jpg cb4972816eb01ebee8f3bde47a4993972951671e990dca797ad0fa9e0f385c11\\n000000008690.jpg feca143d5e249b9fcaca7090c2400125ee19c7ad883c11fffe55e8bb451e7429\\n000000008762.jpg 8b8aa2412ac3502dbb4b8a7d7987359d4049cf70e7177748e907e3ddfa3211ce\\n000000008844.jpg bd8133fe5862ea2b0b308db3c1d96b23012f2f667d43cb524d35ac1b710670a4\\n000000008899.jpg 5643dd54684df9750a16660fcb033e3acda2eff9c809cb69acb428b1184cadd1\\n000000009378.jpg 39ac6327aa1b887ff2ae06588576f497d642bbbf3682eb1eada4574087fcc1e5\\n000000009400.jpg 1a613f9b09db3b7bedb626af3e3951ac1709f6036d4e1405f6e8463816108222\\n000000009448.jpg bd6ac1c94ef4599bd7f9d0f43ba0dab52b0afc34a872ee3acc83a20c8e8a6f3f\\n000000009483.jpg c7e6e15f4db067c3e09ab8605759be9d850e2d5c45ea4dfb9cda8f5d85308ea4\\n000000009590.jpg 93cdd55e59cf7d32c481ca5b00b1b9c64f8b8f44020f64a97752a767fd7571ee\\n000000009769.jpg f5e6d63081ba8fbb2a59770144d7c2350de6a1a8288e15565df7dd1bd87d85bf\\n000000009772.jpg ab798f0ee5d1bed52376a5ee8bfbd973cb4c4fface481a9bc7a3ff5d9c57b4c7\\n000000009891.jpg dc8292c102663e64a5fcf7477e8ab5f5289660099817e7d52f40e3918784dc0e\\n000000009914.jpg 8033e21b153383291f443ff8c89459e405441eac88e4ee617825f19182106d5c\\n000000010092.jpg 70eae5520682d2ad23ff01df57fc4f29cb3f5d4c1d63753d3b7f120f6a0d939b\\n000000010363.jpg 2fd23c4c491520182a703d0df9c604f6b7ee8a592ad8743897870cf0260f47b2\\n000000010583.jpg f163a37f278b6edd09952332066e01a00cf80e3137b427dd49732b91edc3fa9b\\n000000010707.jpg a361abb7579193709ef3debbec6298bf0c540bece429c4aadb19be82e04ab6a0\\n000000010764.jpg 83bf9d868bb6de24ac1ef63df6cd7116e11da51d84daabd91b9b24fcc74d63c1\\n000000010977.jpg fba520c270003f70a09bdbb9ab8ca856a4a6cde110c64f05879e96b755ca7af6\\n000000010995.jpg 19ee23abae8a3da6b5914987e991ea9ee8a9743309ae17db732426c8fa860fe2\\n000000011051.jpg fb0f943cc1b027c64e5dbbc486b1d62d431244565e68b730e2ff8264d51116db\\n000000011122.jpg 74e9d13ddccbc52d2fd9da5984279a6218b250668e1ae1c6450be9d1ca2bc41a\\n000000011149.jpg 5973b8f1d5a30de98b1dcb715f1c94928a676fff8e45283515d0474e51ac0abc\\n000000011197.jpg b6dbcc5c9e52f51480c4974ae6b15003ce25cd98fb936e2f26694fb7058fdbcb\\n000000011511.jpg 4e24ed1528891621658ff01d23e8a74735cd18720d41f7fd2264044497ad2a0e\\n000000011615.jpg dc3d5cce23371a2b7d0be6bf423b5989e29195323b428ef0e639a1d97a3932ac\\n000000011699.jpg cb8a70787b3b7abc56a8ec8a6efcea9a4d2ccabd645b17d73054b9514759559f\\n000000011760.jpg 929474fc4adc38fbd6a392607e27ae8785488ed13676d103c5361e9091333c49\\n000000011813.jpg a7387846c67b6900e02eeca0b3fdddfde24d44e336950f38ee2b54a91d5874be\\n000000012062.jpg 319d58f0fa008d076b766671a5b607f94614f6005f773adced54a57567aa28fb\\n000000012120.jpg 5078da2ea4554eb91703e5fa0339c4d0c875ded1783a8f4ee32ece2b558178c4\\n000000012280.jpg 158ef0b15ff392039468263fa18735f81b39de9e8a7753e1e1aba7037262da56\\n000000012576.jpg e819f39d02674a83e2c3ef722e93e9b3fd6f9e6bf016dc8d873f576e31169445\\n000000012639.jpg fb055f9572e8e059559a950b626a7a731683844f875b66523cbd8d29605d903f\\n000000012667.jpg 0bf182b59b89f534d6f89a2b57c976c47b7f9a95e5e47a7f236b285b9211af14\\n000000012670.jpg 00c989b165bdde8eb54cbe7d1fcfb0ae7b2ac7f5f23d83baea723e7d2acf7099\\n000000012748.jpg 9ccd35fa48729821628123c5d2a57742f49ece28f9813817cfd0ea3867f40831\\n000000013004.jpg 09d489992329d2a17186a82781c7d23d8c687ee580567babc7494f9bc8b1220a\\n000000013177.jpg f93626195530f46ed849091160f8cccce1a255ca886fd5945f5719adc12dc789\\n000000013201.jpg b99aa5fe13f26b9350d136cbb103c95896d81bfa504087888f13f76f5bd4adce\\n000000013291.jpg 1e46187ec49947bfd8dd22f4d774bddfe3e29ca863fe09530516598998b8e640\\n000000013348.jpg 8371f759628048bd294db70d9fbacb6d4f2558a12c25337fa04fea6418c9922a\\n000000013546.jpg 63129a14da61b57cbcd9b3403bc76a73d091e1940e1b416ae9f5c70b7a1dbb54\\n000000013597.jpg e2b4ec9273cf30313e97ba917e85fec88cd9bbcf599781373deae5101ae8f327\\n000000013659.jpg b357d19dee467587a8788ac4e5b45677f4fde1b62726c690795e188ae75d822b\\n000000013729.jpg a3b3689660e2db3d49e131e9856d3d06bb376400e8fa2afc1ce64c5c4765a4ef\\n000000013774.jpg d4a83955582202cd1c3586807245cec71bb38c856eee74d452c1ca752eed8655\\n000000013923.jpg 91494500cd053f1c4e547edc53f62edce2b088588bf9b4dbc8ceaea86ad4d49e\\n000000014007.jpg 4fa388665548f563e50c107089669a589812c4a14cb1dd3b87860fb25bdd4b45\\n000000014038.jpg 7dc972133bcc2e8e49ca1c2b7c20678eecb0c853e78f085f7270f295e1349350\\n000000014226.jpg 7342b8ee76ae63bfa942cb6849300cb39618b2453b2a51ef48650970bdeb837a\\n000000014380.jpg f9e17866f4db47a88724f051f0cd2a2650168c09ab6908bc4f75e5dba398ba1b\\n000000014439.jpg fc23e512cfe33e0b7f5e3445dc4189eb053fed6a16964b2454775a6df3fa5eba\\n000000014473.jpg 1b0fad3e8260366a27f9cb20fdd0f9d7c669bb40a535f6364b8c2e1c03d7b5c2\\n000000014831.jpg ac9629cb598e5d30bbb0a174a01ac4e398de3d54ef8c5bac4f81a738597c39cd\\n000000014888.jpg aba25018b65a2a52ce7d77e68c6501e532084cd327e6cb230225e302fca95f4c\\n000000015079.jpg a44d7b20692b108c648ba73018390297bad07d23b8be13dd24bc637bd31534ba\\n000000015254.jpg a03e74678aedee90fe6777e9ba4eef8b3025ba0b6f426723b2cf2b35c5ffef4f\\n000000015272.jpg d4a09d9517fea54811b7fabb1274117a107c4cb8fdccfbcae4ced6dd82034ce5\\n000000015278.jpg 7f16701e7994ba6bfffe83abaa9d87b128c0d612e11f014001c51a6fc10ade58\\n000000015335.jpg 9fbfdb443c249bdac83b8d0c9d4120a817cdf3987a7826a90e9fcbc95b535986\\n000000015338.jpg b9e7f3f58e1568b0af83d0494d21614a38bb212da2d4b14bc36bd002c0b6f0ad\\n000000015440.jpg 477f0dd9751336fc5e5ef0f7c406506998049de4edae936311a4400c934c8419\\n000000015497.jpg fc0fff04645dfadedbc34e23860dc62c45a213ac22d18b8194d269b66774d745\\n000000015517.jpg db13a24631806ad53b865d8edb4926d455cf54ae60e72b45778e5e58fb164369\\n000000015597.jpg e34e098c9a41160f1f6cd0181d946ddca4e31bb94af38c060e36d6ec4f406e79\\n000000015660.jpg d908b9557857567ac19501f12f6c090ba3568d11421a2fe6296133a27e48a991\\n000000015746.jpg ba79e69d4272bc31ea4fb9f3731e020a9c8acf1fbf3b6ffb78528d70e2e183f7\\n000000015751.jpg e0929307bb91557c288fe98307c198a838abe835820322596232842784f6ce72\\n000000015956.jpg c561d4e58f9141d15a5c4231075a51dbbf257e2901acc482765e15dcee1375b0\\n000000016010.jpg 40ba5b198298d09e33f6428e28779a9cd9bc55756b2f9ae15c258672d6363ba3\\n000000016228.jpg b34d80f1f13989eb0c7031f95373a9e26c698397cb9511b422dcf5daba60110e\\n000000016249.jpg 6b8ddae2940c3f03b1cec347e49fb9cf1c944a6142d71dbe7c7a3c2187852b34\\n000000016439.jpg 8f09a3ac88ba6c74d0d8d0097e8a5aa0c58b7fdc3f51f36beafe8fa958f5e724\\n000000016451.jpg 83fade2162445d292c33005ad31d9ad05de5a4691489b0762bc81ec401cb2cbc\\n000000016502.jpg 171ce16f5646725403aed126091a0e3c88333a6fe4ea5c36f7c9b21ccfe2b30c\\n000000016598.jpg a794c8d8258774f04fcf04921d0daf10d60a35d038117279ff0880a2accd49b3\\n000000016958.jpg 8595370fdfc1ca1382a3eb464e53b2ae6c483d76d426d85b4fe0511ff25c0333\\n000000017029.jpg d619327903c6b05520f0d6768807c067cce2d6d908d2d83da018a1da3b1792cf\\n000000017031.jpg cbe18fd21bd5691cb27085cda1f10c3c5556d5918fcd04363a5e5b145ee3177b\\n000000017115.jpg ad0fc963b409ee072d6cc837cb58e9f0bf64de41fb4e5f13f20cc70dfe3c617b\\n000000017178.jpg 89dd925868177ba4f2e4f82eb5f530bb8b839079bb58f872d78d8ab322c5a305\\n000000017182.jpg 0ec7cbf35b5ff0352f9da0653eda2aaf5b578e7f2b5c8480c5b6c19fd2435141\\n000000017207.jpg fd477a26cf3682083ae6633c3778fcce9e53fdfbb0670232d608865ddb606884\\n000000017379.jpg 52b592adcb73170d7d07994a8781e92feddf3c7f1c45173a050c7ecfe12e7743\\n000000017436.jpg 8ea0f1195969c20030b2a7d90d008643582abeb0b7bff59a2fdd555d79c14ebf\\n000000017627.jpg bad16bff668a52e7df757c6382244c86b416315b86a52749dd7bc6ec68e7092f\\n000000017714.jpg ff331d262fea90025ec3b628757b047259f244b5a65550b1d0d079578179387a\\n000000017899.jpg 2e1e630f37d1f6f1b0c06941ea56431c9be8701c928df634b8dbe9121e735fbb\\n000000017905.jpg 5fcff2c9895b398fa086180c2fbe39b17a4e07d47282bd0b184599764afba33d\\n000000017959.jpg 8d525d9c133ee9ce8110fd8e2cca767d051bd22e8822c8f9ede6e72b7ff398d8\\n000000018150.jpg c49c2bb7e3ea8951c997f9984d13cb9f2bf8c1a0b4f7c398c0317f36435059f0\\n000000018193.jpg ee9037876bab7f2b504cd4be49144bbde27f32ddc05361cd6f3c843c947dd911\\n000000018380.jpg 5f019938d891305fab6ae0d0e9edc79e0d74407cdbd0ecce324aada0ab93a86e\\n000000018491.jpg 195369c44ad1810c524d5523a556356b2ee384310d7493310ccb8165c42767ed\\n000000018519.jpg 87b0a064b596b4e821ebb371077c053e836c0ea667ca491075804bb3c9ce95ea\\n000000018575.jpg fe6cdcb9053cb1621bda79d13e20cad49b898678fb2279f772964d65698c3c6a\\n000000018737.jpg 61b0ee16f913053290e0ee24971a6833b21290ee6a74039a0e0f3faa72dcf61c\\n000000018770.jpg 92edb27a1c69e8e77d6ee17ddd2f12489d1852ff4c32c8357b329ba1fd26b046\\n000000018833.jpg 3aaed3d8164deb6f7e5e18f3b1b3fc1b481d32078b342a07b629a7bdefc14e4d\\n000000018837.jpg 70f986167a5f1ea723ee2a25273f39ee1d1d7a3bf8b2fa3a37eca946111f1d93\\n000000019042.jpg c97e2e9a24160678ad908797a306d146354f66b196627a0bbb564c733b041007\\n000000019109.jpg 5469f123ab838c9fdcc567c13148399dcb4cfa4eeee80e6e56a13627808a6d54\\n000000019221.jpg 0085f80b9136c625b63fed2c233c432c3189841d5bf630a10e09ee1554c9bc79\\n000000019402.jpg b08173e02dc865e7f13fecbcbed574b70271c07fef6023236d0dea8d6bf77619\\n000000019432.jpg fa54de5a592c892ed3aceb11ebacff27abad01dc3bcef308943b097c1d65c0f0\\n000000019742.jpg 6ccc37eeba5118114222f8de61228e95cf6c28fb631c9e759fcdcd450061337a\\n000000019786.jpg 109307de9a894df50386a58dec9347e6c280ee024f426933e2e34bfd57478960\\n000000019924.jpg b0bb42c6674456ecd9155b790a6a595646aacfe45ce80bd7667ac75fb7fe8bc8\\n000000020059.jpg 7bf0e3dc0cc83a62dfc9c9bfd57163cb541574a0b866fd1a555fdd24cd2859f2\\n000000020107.jpg 7c7e83890969c3bb7647cda61e17fa8ad3dc33210dbc9db616afb71f20958d3d\\n000000020247.jpg 92a0ab00f3a0b9f6de5162449a8b79167363897fb489c172eebc244bccef23cb\\n000000020333.jpg c32e6edfc1e53898d98edde0f92cd5755ca6fe980b8f522fb29d9676aaf88294\\n000000020553.jpg 739fee1c09f6d7c1670d2f558f70d2aea2ea6863a964c5a9582ca3bb70a55cb5\\n000000020571.jpg ee40916130fd5be75f3357c910c392b8bdda7f494078992472f67587d62b50a4\\n000000020992.jpg c705bfad8bb9333fd641c31a14514ed56f04f32082707a132ada26b365142378\\n000000021167.jpg b77df96f91ed9a0842559ea8a238d602d4a09069b9129c43a9d62a27361df156\\n000000021465.jpg ceb0a4848d7dea0d9c0576a91a8f8c9e93a9a090efbb3e6f641a9c327e7c679c\\n000000021503.jpg 83f04a37053c9bfaaaae137edb80a262fb8604af40064bfdee7899129ee5ce77\\n000000021604.jpg 129977690e3a8a14f333033c0a2295ae8a8f6d0cbcfa6498273e4912d1ca3e46\\n000000021839.jpg a943ad725c69a532f7a37b6a95e20fc5d7f070f1c5637c44107fbf031a9fe265\\n000000021879.jpg e18240615b9733cad806a5fca799dfbec344f98f5585722305e99a57346c76cc\\n000000021903.jpg 3fa9ccedfc3ac829c07e14b0e7ef4211b5963f7700dc6e30dde3054f704efa09\\n000000022192.jpg 3e9ffc06c76b3b856c1f6911aaeefac685ebe80a3bbd72d4d3c6254825ab159b\\n000000022371.jpg 7023b465285e7fcc25c9272a9b24efc5df076ad982513ff6dec4c3c7842e734e\\n000000022396.jpg 7f40854ecf8edcf2638d2cc5aa8c422d07875478c65514506c81d0200985f6be\\n000000022479.jpg 9f09af60f0a4f214a8b34ff6badbd25111cd8438c39097197a629255408cdb75\\n000000022589.jpg a613ba4b6d895c6844d151efe922430660ef602ae40d4e16e1762f06b2d17335\\n000000022623.jpg 93699b28cbecfe8f38fd82e3201ed1c7310fe664206a97306c806a4000c09a80\\n000000022705.jpg 01627a2e4b79593b0debcffcdcf9b6c795d0d9e71aa2a3e9e22a9db05a70bc85\\n000000022755.jpg e419f292592075e18a264216c6ef3f8a597f027d9407e278d15811a9fc9c8952\\n000000022892.jpg feafcf6539422208c296d7e15151a9bb0a19fb14db57d93a3ebf733a21fb7843\\n000000022935.jpg 90fad7bfc9fab88492d9722ea1aa14ee77514d8c10dbda86086bd37d8a05acf0\\n000000022969.jpg a4cd0db37176242a6911559074b58a67306c6a774df9dd07f1c9a30112ecbc30\\n000000023023.jpg 6c33ee9f7ee0332811ba3612c8ffa3636cbbe503e3b6f8c1f7f6ca0f2703ca3d\\n000000023034.jpg 2a571a9358b1d8096bad1d317fa81c179d1137feae2b6688881e9b286f0b7c1f\\n000000023126.jpg 65917720776abafb2706e037fda7a9d90eb17b659bfa49ac20d19b06abff2e9e\\n000000023230.jpg f13b4e3bd41de8c752f4954f97f80db3323fe44b0cc656456df8a97560318566\\n000000023272.jpg 3a20a6c0f0c7d33b9c4d9a4fe9023446ce3f92e90b37792cb791c8e3d3ac4cb4\\n000000023359.jpg 5f4a78725b9f016407a69420d14222ecdc424994be22085318cdb6071c1ad7b7\\n000000023666.jpg c6e0dcc40c55570c04bb709097e3d0c6c6ada2ef55ca2a2601c67545581897d4\\n000000023751.jpg 8f8d858e640624d5bbca80d54a936f77efb3ab7b50317613c3863ffdd9a4a9bc\\n000000023781.jpg 085b1c7f25bf1d149b6dff82dfdb0a83fbf6e875c01d5d374070eb582aada80e\\n000000023899.jpg b7ef13fe4d8ab47a77e55dadd30295110ca97b7b3c3b8c85603009e7dc497983\\n000000023937.jpg 7789236aff12bffa0cf4ab954fb68531bea594d42e78cb80bbeac45175c5e144\\n000000024021.jpg 5b65fe7a6d334fa5f7d0328c37470d7d68d6e711b960d37f61ff133be78c4450\\n000000024027.jpg 63901c04b597683789f649524a489ad0d6bcf03929ec3fa334278b8a69fe369d\\n000000024144.jpg 5d0374a85c317f0b64b32c0ecf4e9a5cfc6bca105e3f703b4776cfc9aa0a20c4\\n000000024243.jpg d5149ccbf5243f035138d8c982bcef0db2cf20e93cf249c73183957b9cff10ff\\n000000024567.jpg 44891e1a27e3a262a3d879fa27d2749b31ca71d648f800952f4dccc248bc3743\\n000000024610.jpg aa7b8f989b3584fa3b41c743f5d1b3c45f9f6ead8c31d7a64f9b253b5c2868cb\\n000000024919.jpg 9c721d584521c2ae58bc0eb03b57c4ddb0a013d6af0ec08fd223ee7c11fc7b45\\n000000025057.jpg 8208c5aa7a2aa8b805e7581289a91c8ed0cad5cd8a90e826d1ea0ca59d1cb773\\n000000025096.jpg 2115b6f6929bdbc88618a070f61bffaf944dfce7d666b4c8227151524d3bfc28\\n000000025139.jpg 9c1c256049d80c05aec424d06263b1e0f204c3e5c20352a76b9897fa3ae581f3\\n000000025181.jpg 91928323bb24e153995d938c8048039885285fa7d7a947a4ec5edd07f3b15e19\\n000000025228.jpg 2b6b45ffe7197a9006a98ed31b8f55495bb876e0c224f1695810f04e61f626cd\\n000000025386.jpg 64daf1359646414e79338f609083292162bf91e560e170c72a4666d95d341a44\\n000000025393.jpg 677e993476aed8ceca9bd9b8edbeb3c8874ed19488c9050cb8b55057a5f1547a\\n000000025394.jpg f13f54a3cbe28140004425b1b010c51bd5be0e4c4da1d4b1e382e3e0b426e316\\n000000025424.jpg 89e8025ecb6f48125ac826dd5fe51409deae64e98133472adbe6ae62b6a61705\\n000000025560.jpg 14ce4ee59869b1461f30c64c09e1240edec49c820933c90830b6090d4219c416\\n000000025593.jpg 83bafce2fe5f41a25e6f0e0f43c7e6d972641f0c3fdbb26441877366a5cb77ed\\n000000025603.jpg f75a5898681ea62ef547b8c002e8e0eec00c50aa6b995334ab967798cff6e6dc\\n000000025986.jpg 721b718fe15a4f3739c0673acde4fa224ae7839f9317e7e66d256653551ea23b\\n000000026204.jpg 173a86fef1dbf7dd4c91dbc6ed7f1df26fdfdc398681aa04e558ee165f3ea326\\n000000026465.jpg f941bf1156d9fc2831c487a93caf78ae3367ddaa5ba4f6cda4384af13ea91ff0\\n000000026564.jpg ecefff4893149e6ac22348a59c8e4d3c1b3a751266a15bd38e3e6d041bf39c52\\n000000026690.jpg a59b160b83fba25fb745b42efbab6d99202b4984dbbc1cdbe8d00201b561f8f4\\n000000026926.jpg 7126855b6fe8548a6b9da18d8424f803c0eeb37deb3f5764f95f4d60d1915b37\\n000000026941.jpg 2e4081f3a5a5a747adb86d3f85dbd3e22db944ba8c2aa3f6119cb4146de4d2ad\\n000000027186.jpg 2e60a537145c388aba560db6312aecd5fddf5bce7a8f75adfd06492bdcd34e2e\\n000000027620.jpg 035a452ad79eb267a669fad48b220b1bf7d2e3500a5d736a195dc007df01bbfe\\n000000027696.jpg 9f24277c956430eb4502f44345bcfca81c43cfcca7184a75820e2ea1dde8cc64\\n000000027768.jpg f9d68320358d980c1ef077c986316729fc3bd42b7e18e4433bf29b157f1f20f6\\n000000027932.jpg 9ab6d8c5a124427d77add9827dc6ad58ce94efb003c34e5e20699acd82f43731\\n000000027972.jpg aeb3671152fdb0f319a96bdcf659751a925e52d5ab867e7c412273f7bfe73845\\n000000027982.jpg ef3fd6c41531be0ac195cf5efa67c803ca8630c066ef5c046d802ff8ea4ba440\\n000000028285.jpg a37d71f6459efc47c22ec687945599cf4ec81c1056e6f0256eca3776834879b4\\n000000028449.jpg 520039fa3e2dad9a09632c7f5c5babb2ac30ce78137d3d140372ea6d43cd3ff2\\n000000028452.jpg b9adf909244cbd3c6105af33723d6664bf7e88cd7ce4de697f990828d4c16a76\\n000000028809.jpg ae2c7be44df33d27135b6aa2f5d82d4c8bb9784e99469a04be708ee68943a45d\\n000000028993.jpg 9d582d15dc1a53b419f8c32253d4eb1bbe0fa20ada973e4d24e72af55b880a83\\n000000029187.jpg 636a078ae06c6779d11fe5bc889506a6cbf5cad7a3c82dccf1d7d98130bbef73\\n000000029393.jpg 378763dec71a25c7cb44ff9e345c84ca0fe2d8bf993b88775c360549edf3b733\\n000000029397.jpg 4185e741b14c3e6e2b9da920217a988d08cb89dfc41d377c3aeb4f1e41d8c2f3\\n000000029596.jpg b8e8d8f5fb32fb90cc1c064bbc0fad3846b3a6c1fc6773fe74700a86b184c315\\n000000029640.jpg 11957ae08b2b2a3802cb671efab513fdc32ee83a31b390341f2db228b6015c8d\\n000000029675.jpg 59647741cd07304fd4ef6309bf4707c6abf96bc8e654ce632be41f590e5cf65d\\n000000029984.jpg 1c1ea8336e8c0e5baa9e6ebd7c732aa49b7b30e411fd9cd97fcc3ded654777a3\\n000000030213.jpg 27fdc374629bb5fc3c059eb7f5516b940596f0b50087adf364b6b09922089eaf\\n000000030494.jpg eabc12936cefe60de3cc9cf4bdb49fbb0f9435f188ec54dfb067f597b51c0c3f\\n000000030504.jpg 6a07fb2882010ba47140c43ffd7cd4a51331f197666ea9837b43bc077e88f552\\n000000030675.jpg 66c6259b826b869e0d2dba078a0de6d9c82617843c2d5448e9b34566bab6f979\\n000000030785.jpg 5a389b114f8aa2833b93c491616ee551ad6b3c316b2470265998013585f1387f\\n000000030828.jpg 0c9fb9dc3bd6a11defb1ebecc1c3a50914a0007e53cf4b16f1cffe7b363fbd79\\n000000031050.jpg 6b16283fc0f115b7c14288a8e7949ccfedc10440842a5de7064f9c58c4c7fba7\\n000000031093.jpg 769ceca7a3baf1319cffe4ad462561859d1e3f1029034fb87a12ef69acdf91bd\\n000000031118.jpg 2cdfc92ff0193efb0a93f9021841308f1cb204d1ac4810d00aa32a515ef033f7\\n000000031217.jpg c3bd88960d608cb0406e870e224fdf0a5cfd8365537c2fee54bd10453dd402f0\\n000000031248.jpg fc5a3defaf6427f8f59d0a68d85ef6f73b25fa8d55fed78393f31a9e3e0e6270\\n000000031269.jpg 91de0bc1f19dbacec3234adfd8e9eb160c5a08bf1dcfc4b6ccc08af09611708a\\n000000031296.jpg e44caeefa73a14822ea5d47f5c84023ef7883bc064bcc089ca2f368a8945e2ac\\n000000031322.jpg 10f00a49fd34b83b1656f627061df90603bfd3e3f2ea8119975993ed0ffa5e47\\n000000031620.jpg 61512f0a301dd85e22fabacf58d2f04d003cc89428c127c5c6aafad9f4c30ee1\\n000000031735.jpg 49568f890bcbfddb7afd18ba8afc6c1b20ffe6611a888a3cb93150d24b253ffa\\n000000031749.jpg 7fa4284ccb0f885ee85ba6d12222dabb404d8a0e3b081bbaf89148f95ccb7ca0\\n000000031817.jpg 7b2e9e578996e44b1e9ab8d7658a3de6970de0d8738abb560866fd998bf399b4\\n000000032038.jpg 5afb15a7ca2105d97e1f3a095dcbb1bdbe92232b1aaad9b7ab3bca63ee8adc3d\\n000000032081.jpg 2acde5232e9dd81c012d6e0314f733985f603c95d7d482d5a52e4191daa00bae\\n000000032285.jpg 080779f05a0141b2d762ac043d96928fdc56bab73edb8ddb57e872501c8e940a\\n000000032334.jpg 68388310570b436dfccb56686a02d54621673b2d95074bfb6dab46388f22066d\\n000000032570.jpg f193d982ca276820bb038610d4ac45fb2de46715510186b834c984a7f099a365\\n000000032610.jpg a77e88c3a333e9264c55f6c92c97d54e1e41b9c48f9307509c39f17ab14fb382\\n000000032735.jpg 309105454f28c30038cce8c44d48300b626548477957b3d056bfacf9da3127e4\\n000000032811.jpg bee917a2ebb2c2a35fab43f9673164e64b1512c846bdca19bdb629611b80fb04\\n000000032817.jpg b0028b5505cb3969c3bca9e63a6e0691b5ba85f8db31dcf68eb46881f3c98b78\\n000000032861.jpg 4381cc5866c6c160595d85695f0b7d7ad30b077f788c208f769a38d3a0a681e8\\n000000032887.jpg f85c17ab33d54631410d6335a5eaa1de780904b777f1da15f9c06ce6e61d3520\\n000000032901.jpg 92ab99b7bc4581270d47ad6b1f7f1b06c9b7615e226c68cd8691726cbc53c264\\n000000032941.jpg 2b881cc082e3d79f6685320ddf02e04ec534e7f904a090a4e627eb445cb26ebc\\n000000033005.jpg 840a583d45613d662220754de433f96bac6e094372546cf2f3090f73e645d7ef\\n000000033104.jpg 24d3088e3db4b4a04c83f0c1cf3dddbc33c02389eb2591d9d03c4a3cf73796ed\\n000000033109.jpg d4c1905fc9f846478feaf8cd048d39a2d50829d1dae1ef954d9b0e015c081b4d\\n000000033114.jpg 6ad530e1d20b1b23d52b880192d8ef314cae28a93b9300561d419ea6a2bbdbfc\\n000000033221.jpg b7031fe2396f64142e19ba4f143687112bf555995233a7e48bfc9a57ab4ee4ea\\n000000033368.jpg 8744e00ee252788e713def22b9d93710db7676dd5bda496241d14fe03661187c\\n000000033638.jpg 997707b5024a71c18d33a03a6f8627556f0b309c92f03962bf0da3f83f5f8f0e\\n000000033707.jpg 78d30a901a0cdf841fcb9342dc1262d61feaae89e4ef5ee1f0de9e275bfe3464\\n000000033759.jpg 1752e2ea7125cfc42c0b5e73390e647f98d0e32d7df9c528f6e839b108d672b7\\n000000033854.jpg b86eeebe566cee1e853f153675db94ba23d6642956f29ea473804c7852cb6b37\\n000000034071.jpg a948458ba76bbb64a2a1ce28a88ffb43c7ced9f03f983648fe7ece83a8826b75\\n000000034139.jpg 1173eb43bb487f08c3620b12daf67c98a6bcc14b7893521393055bfb84db33c5\\n000000034205.jpg 48fbf607e09ff475a83d2f67ae74e59da68e7992ecb43245c4ba65a8ae3c2509\\n000000034257.jpg d6d088e414bb3638a5f578e5f3ea11d9e587e6a5efb72a23a796b61ef60dd99c\\n000000034417.jpg 2fbaf1efa2a724925d68f07b64d84a02c0ce7cbbe9f3e26262625659f9f31c68\\n000000034452.jpg a1b87d38128e4507b46e9cb99ab1fe96b062f3838af2be0618e6e3fc5abdd5e5\\n000000034760.jpg 0fd67be645460db96f578a810d5d59b84ec4ca8f7ee905a1b538b8591211a1a9\\n000000034873.jpg 376dbffac6db0b092cc07b43dfc34886725682d8e641caeff54710ac8f61ffb6\\n000000035062.jpg ced2374ae4276360bd7ca8f1aa2cf46bfebd62af2ac1eaa04a8e64fccc150333\\n000000035197.jpg ba3bbe396c1ef5ceb856b4e3d5017207db5320a2883fecb2ceca0d6fc79c4fd8\\n000000035279.jpg ad30a4335a65035ef96c5503151ea47370e6f66e65b70a44c26ed1c4d580df95\\n000000035326.jpg 585b9ab944861e297b49d352117de3ab59a5c3bd55f3a298bdf91b7a3a544678\\n000000035682.jpg 1844cfa76d23a9768578f2c27306ecbad99326d2d374b070af9c56d50058c155\\n000000035770.jpg bfec13432579cd1aac19f4dc14884a1b16b0708101271bd03cca378591f36ec7\\n000000035963.jpg 4ccc0b243ad62dde6d6500eb86f7a670f631302bb97d0e5bebd92f1d784d456b\\n000000036494.jpg 08defd539aa6aaea4f85c03b650b6b3d4ceb5d47e65a29d9aa17d18915daf9b8\\n000000036539.jpg e16c9d4bab618469d1d841a4b905fa01232f42c0f18884b349b243b736c66d86\\n000000036660.jpg 331fa26c3a7931f3752e56783e16f147d9b0e25cb63f4ea01cb4aea2d7e3e1ed\\n000000036678.jpg b1a453d5b19059f02e49f9056c31e1b4c09e58b8bd3093b0bd3266149d8dd1a4\\n000000036844.jpg 45107746c8c33123a387d8293890c4e08e0d8959f4cf5987e022556758ab5f1a\\n000000036861.jpg 9799ce2cc5ff5360c97711b51bcb3a1e515416f56b975db40d884c831c2f7c99\\n000000036936.jpg 335f0411b17499cd23577b270b9a1aa8519cc44f6f9a246fd666415c833f4ca0\\n000000037670.jpg e393e95e4a6bb9d702502932da4e771c8f2e09109e92205f8c643bec8a1e4e4a\\n000000037689.jpg c8abab45d17f644945876d409ba6eada408fd2f4121739abf6fa9379920cbf09\\n000000037740.jpg a43be814045dd63e0bcd676b10c993113d30183380b04044f2a4eeb05bdf0d27\\n000000037751.jpg 5c19f466d3c9d49f855b98ce907e7452dcac3349e1a4f9977a18a0c9fdaf5a0c\\n000000037777.jpg fe513b68ba46bc146114c469fdff52c288226073f03d73d3300d4fa739937369\\n000000037988.jpg 0c5fec72e3f1ed81b648718bbb7436be61218bc24b78f56c2dd365196050fb59\\n000000038048.jpg 38eeb64d6890bd5364992e704acc366af246bdad6ca33c8623e73c7e118f340e\\n000000038070.jpg 9d8c62b8231ffdeb64e8b9b2558b366cf86cb65408d825123fac1b44e8d653da\\n000000038118.jpg ea735adca70a80e2d2e55495563e5c09d90cb44a0bd1458f54daa9a7eddca040\\n000000038210.jpg 0317a0094f357ea07c293ee9c32e58a87e5e0a5438c6d9c09250266df1249a00\\n000000038576.jpg abdeced448bfb1d0aec615f9927f23ec679360f1fd954968b1ed2024d386509b\\n000000038678.jpg 7e3bf523b3a2baa280716e31687874a031a51219767b20965a62cc7bd1f299a6\\n000000038825.jpg c97b8b6f195541932de28e6a79bacd569d65d29e0a32f04274885acc0cab505c\\n000000038829.jpg 73f5b0fe070aaec65bbb7263d16d3d717a75304333fe939dcf086bd85d6ff2f9\\n000000039405.jpg d52567494fa41938b09e0734c1f2726b5031c0473fc727b152c3848bc560138c\\n000000039477.jpg 66afd33bb79d40a43a8486acafb399b524a6c49b17f694aa1daa8f697aec70a1\\n000000039480.jpg 01bcb7230363caf883ccc2c419995c00ccc011254cdf3fdec5f5f33172496083\\n000000039484.jpg dd5ee58026d819fb399dff4339efd842c936746c404abda74ad42cc2a499bc5b\\n000000039551.jpg 4c84a652af01f02005d2116c351bdadcb1b82445c2fae9cd47adaccf6470a136\\n000000039670.jpg 78fb0e5a938f9deb4a359dda8e0e90f8634edfdf703a93131423348331ed347b\\n000000039769.jpg dea9e7ef97386345f7cff32f9055da4982da5471c48d575146c796ab4563b04e\\n000000039785.jpg 887b76f9732bdbb0038f0338cd1a687b49623114352bdde634a3a25a12454b99\\n000000039914.jpg b49ad1d74810ff1d6909256d2b55ccd8593fcb2f6b9f059638d9fc2c9bf059df\\n000000039951.jpg 471847fe98e8c4672ed59c9cc73fe3699c9654152f667a65a586fb2c24500834\\n000000039956.jpg 76e73337d2385cf701750bc38a42f0205153af090244e66fb7a261052cb05217\\n000000040036.jpg 4668e4529c5e8a2e60df6fe4fb3882c089ce93e59ea8c8a5cd86856b6d6b66dc\\n000000040083.jpg 89b9c3771c9428e0e25134cde9bc0a962318a477d267ef47f8e1872526207da7\\n000000040471.jpg 77aaef65849b5bdb4df3ba5bb16028c3a4918b563fcb2c191114a32c9c6c9d77\\n000000040757.jpg 60f4f240bb8fdd983afdb8966fcb7994e74d36b8e65d77f47066ea95fa297dba\\n000000041488.jpg b33827c6a6a0fe3cf547390cd6ceab75afe4b307bd19a9fd767cde16cccba2e3\\n000000041633.jpg 9aaff8c1163dbc4d38c44079f1dbccc64221d4c93fa45404249595951a70ce16\\n000000041635.jpg a894f9b6c0edc1dd58214d1fcc3b469e6c98b2c9c4c2e04930408d790841ab40\\n000000041872.jpg 9ef41d1689519ff9d93fcd59fdd4ab43e9ba205f76b23fc1aeaf4f2d5cc5337e\\n000000041888.jpg a53fa56864d2c170551c0584813aa5a39b586f100bc16dc26010e9af88f16cea\\n000000041990.jpg 0218020651b6cdcc0051563f75750c8200d34fc49bf34cc053cd59c1f13cad03\\n000000042070.jpg 818c02b337b36d985a332535bd5526fa8f958823b1c910e86579fe4294dcbcb1\\n000000042102.jpg 13adc5e53fd4b2e2610663744fbc76be64abf81a2932bf2acf26afb5e107368c\\n000000042178.jpg a93b3e69c542397170396ea0a1805ff740924aedee8deb336b82fa9277b82b48\\n000000042276.jpg 756af170de5d32112b9c0cf5af399dc4dd19a2c455e4d352c3c60577d1ebe3a6\\n000000042296.jpg 179182e03cb432fbb2004dc7366e16dc83c1e7cb834a8110caeef9a29f521378\\n000000042528.jpg 842d6691981b2204da0e59afb546a251846336ae71e1e95f6dd87b346d0a5efc\\n000000042563.jpg fbf3f5a306636048f1c1400d5eda16b2103160d0088adee63eb4c26fdb95fcfa\\n000000042628.jpg 3f0ee21a4508802e2d1c9991355f2c9b501063e24f665d23136e7b05f8e9521a\\n000000042888.jpg f7385d75be709221eff6d74d0c720fb5afb4e895706b5c842fa45d987f22b1ee\\n000000042889.jpg 8a5181cdf9425417913d378a6f64833e606586202e9297ad308c4c95fc38e025\\n000000043314.jpg 360eceae3e66803c8e19bf9a45b9498f0d4f6153ae2d6f832cbbdaeb899e393f\\n000000043435.jpg c3b039effaf2412a450faff11d2dd0c5f5ed0090d12d550ce7a0eab9cec51974\\n000000043581.jpg 9288cbba23a3f1a399f8e6825f6c37e2a24491b8225cd1635a60327c423fbf2f\\n000000043737.jpg 87e63848d72793260c580c8653f3ad0529682e49782dceb54385c0aa5417a10d\\n000000043816.jpg f2f806d70581cc12926f575e567b39dc0e7ab5c411a9f264d65d3c31f8c6d102\\n000000044068.jpg f87295880cbe28c49af27753a63cf939b78192da5dfed7d01d9a8087b8976af3\\n000000044195.jpg a99b38c679287df5161a985e8cfab1f98d2045a7b78e0652715b4052cb7abdd3\\n000000044260.jpg 6dbe2bfaba661155f667db007740c249774a7d0ca0f5e9d384f6bc095492d39f\\n000000044279.jpg ec4f47bbaa7a948c06f4e748cfb9c8388facc711b1b69e98aa0dfdc656e6b95d\\n000000044590.jpg 5ef55da25c21159d1aeaf61eee0c81b2127dc626eac012c68d2e9a81d2bd7789\\n000000044652.jpg 807b8765af872023b004a7b2bf0f158c8a176dfa8c07d405f107eb9857c4b2e3\\n000000044699.jpg 24c2eef8c2c8929f3b2032f30fc89c0894c6f1a375d6ae016e3ade301a625165\\n000000044877.jpg 0035195299802ffbdd572dbe8195497d09a622124184be208f69d610f4fe7a15\\n000000045070.jpg 0048f03ce376c1c3de6143ea8462c609cf0ff15847e20a20d042f0c4d968eeb8\\n000000045090.jpg 131ff9a0a419179b934091c97a57ea257b02b8ed80762bc59d639d980a9707e5\\n000000045229.jpg 998d76a1367d02919e07a672c028b5f41ed2713000357cf3a8b6832ad7ca2368\\n000000045472.jpg 0e7a75145b74abbf6e26aa3ab0bafa2f2625d4e8014c7bd4ae938c12e08a8131\\n000000045550.jpg 830f564b71f327b0bcc78afcc4b8df86b41a13fa9bee13e2b0dc20bf9d827e9f\\n000000045596.jpg 6067fb9c3fc361df097d97cfce2bbfb895929b74cfe5ced4759445f8acdc5159\\n000000045728.jpg ee8c2a9618c4ddf0dea13bf308d61af2d62fdb4f927409aa56fae182e0671df5\\n000000046031.jpg 1f9fdd33f7fef7edf43716034335384a6c33fc032d3ca5984eec495346be828c\\n000000046048.jpg 5d8055ed7fabde0ed90bf84d6d89256c53e7fecefb369624f8bf3b0e3a464eb0\\n000000046252.jpg 2ee9e8fa409776fed080bb30f32f51d7d459b2edb606d88dbe86b98a69a4d5ed\\n000000046378.jpg 927a104a4eb1ba7960946cf89ed6cc199bd7c840674787b8e77b74a7dd60aafe\\n000000046463.jpg ab7f51b790223a301a4c2509cf2ff78554149b76c2ccfb54f303076d6fa412b1\\n000000046497.jpg af32ef895a7c4e954f3077b36b8d0a852606d9e4861a1e0ea9c18d7e3dfcfa82\\n000000046804.jpg 88b4672b1228dc5cb2d770a7c78ede27384a1f5bb8c429d5220f46d91f25f293\\n000000046872.jpg fd991bcccf8342f3ab5c5228dce3873765964a545365ac3f31d69b32c8a9258d\\n000000047010.jpg d3d2e08209430be0d54b9aab3b6bd7c8ce4c2ddd79b9ed7f994dbb3bc5134856\\n000000047112.jpg 33e728e922af408a6d088927b3d85720208274feef41c1efdc2a4a6ec4e10cbb\\n000000047121.jpg ff6c860fbb42455a29074dece1073bf8edb5f5e7fa862784e00129822a8ad89c\\n000000047571.jpg 0398da07fb61f7dd096005253c01a669358c81507ac913c2d86fdd8ca396853a\\n000000047585.jpg 42d85b5e2fc3f5d1e9d3ee8993fc6063aacccb5817c97dc6a443ff6450373e42\\n000000047740.jpg d58fbf97d596915335f1d9869ac59ec09ee16efa472bc49d820ed4cddd3b7af9\\n000000047769.jpg 05ded59f58ff9936caaecf0be4cbe8ac15aae965828d9d11734825deb0fb7ea6\\n000000047801.jpg b621506b8eef99a00ba3ce791f66181077b18513b589edd04bc9caaf0ea06f92\\n000000047819.jpg e2f26106de0bf2dd49cda97b43771ca7788d9405cc78c1340445ead747d7d72b\\n000000047828.jpg 5f6e1f03f620dbe602f3d447952712d9eb922f23b5be0bb3f226f0dee5cc80da\\n000000048153.jpg b83b9e68be063e9049ed73374e3c52ef6d401fa3bcefc38d8c49c9ac82a64d49\\n000000048396.jpg 7bcd2860143837c1dd81c63236d02526e832ba83f181162f101f232dff0b35f4\\n000000048504.jpg 7d194d283ced6a30a2f642cd58817b8007578b531c5346910a2520726dee3972\\n000000048555.jpg e931bb55b630e775a69395b87e118a76b66d7a216978000228b98e1dff015698\\n000000048564.jpg acaa81114ed1f6ba7a1d4ceea3376aee72f489afbe7db5e1e0515516978bfda7\\n000000048924.jpg 7e4db164281de37dafa6d3a4e861f6777d0202661e92d11fb2b739e39fa8c391\\n000000049060.jpg 24fdb66de0ab05eab30ceeceba3b4725084e075b8e40d91bb9d28201cd70b484\\n000000049091.jpg dfbab4608aea75760036a4e366130ef5bfb49d5f8699c77ec14f36154924bb74\\n000000049259.jpg 75deea304839610c4f1d2312046b8c7d01ba6f350512a507d95abf2c7859ba7f\\n000000049269.jpg 2fb0ce55959869151baba2d1197fb786615539d9184ea880983f7f6c4908e822\\n000000049759.jpg b2e2e3a373ca6925746ce645689b6d38a730ce3f6f29b6d2fd67d18b2ba0600a\\n000000049761.jpg cdda00ff492efe03cc180f7a2bf67cf6d5b2879250368b2a413a0f938f028cdf\\n000000049810.jpg d8fd48490a34f0f1e206a6206653881036e889e79c022f1ef97fb05dafdccf9a\\n000000050006.jpg 6da72ab0cf83655e8095cb8cc29b74dd017496d5dd45cef2c7328556efb62196\\n000000050145.jpg c354e4a8ffd65115feaf41ed70dfc39e32eebcbb645469a0ef00e2518bbe2887\\n000000050149.jpg 51e695b644f5f3431e44edd73d3ea54e1501335635c84743d0d7e24240c1497f\\n000000050165.jpg 05045394108b6625e1032f812213ed369f187755fab7112a183fb21fcdfd05c4\\n000000050326.jpg 2242c57b366c8e95b8dbe33cb4a44e12a91b1b4d7a1552b385cc98376904f0af\\n000000050331.jpg 2b2f516489343749b236c3c834831dfb82bad6fe053585717a0dd53964f234c4\\n000000050380.jpg df1d17f3f65d75e67dc90f14c701fa65c097c8ac0a2e0d1b7bd226018fc57722\\n000000050638.jpg 7bddfb9dcb6fced3a5d40fa18ba6bd6ee2f769b37d980989620007b29765ff84\\n000000050679.jpg a4531c707b2f6e12203915a08d60210fe666cda426dfcd7cb5623d33e9c147da\\n000000050811.jpg c69067b49ec7f1f07045130495c56fd29da44178965a33861b51a503a041acfa\\n000000050828.jpg b5231b3dfd3833e5abc1f60a6e632de13b697c313ffefef7b5697665c962e33f\\n000000050844.jpg 82e1bd15d418dffb439de80cf445b733a186286253732895c93d79464bcb6165\\n000000050896.jpg 6749a935b073e9a3958a463675fd201b81c66eaa1b0517bdd44d040146e17e5c\\n000000050943.jpg e3b0b34cf67c4db9ce9c92f4877d5c7c2f326109f02b6d2ea42c51b78a3748fa\\n000000051008.jpg cd615d2609c7ea6fa5c9c132fa8efd924477a05f0333470cf496117b7513fc3e\\n000000051309.jpg 843073a1b40d454f132da06607d3ed525ce46eddd87c7f4de0b59378b8a02164\\n000000051314.jpg cf479cb75f4ac8f7297ec510f3de332aeb13f0b69fb9bc24f72daa7cbb236828\\n000000051326.jpg f1e5fecbdb78c1af6ea0a286840076f9f8102d5e5622af183df2c3de13a5e679\\n000000051598.jpg d727cfdbd87883caabe464636cebdac19c89cae7d948d19c1f86f4230edfcdbf\\n000000051610.jpg c4b6fdea9764b29e9ef8015c0bf6c1bb45c396f17fa0a425cafa9f2c32a025e6\\n000000051712.jpg ee2064827182164ba79766e6ef5492397170c4d0466e6ae2980faca0ef7cc9e5\\n000000051738.jpg d44181d442f78b7c16f4fb7abc4b23e2dc64dcafc7a56f643e944ddc8ec2d4f0\\n000000051938.jpg 83af2d377338ed8dfe2608d8c8da1273b11dfb4c534b6426cdec0d93c6ed2892\\n000000051961.jpg 4b2300cc85b7d63ac7e7744c51f0413cff08530890c6525946e7004bf20e02ad\\n000000051976.jpg 970ea1c19a32bb86693413379669676443fa1ddb0ac153fc9fac48a56c6a25de\\n000000052007.jpg 83345dfd3d5402861ec0a5ae0767b01439733c798cf81637b304c309abfd085e\\n000000052017.jpg 7190d285796bad80a5949d171b6c05b8ad354c1b418981c55ed85376c624d48b\\n000000052412.jpg e4e5f67f6ccf0dae68dd5a0ccaa3dee71169a2ec2904ca46209998e8476f4fab\\n000000052413.jpg d32a2e65c67dd31a5f9783ba6749782e3e617043234cebe3b57cfca264c80f81\\n000000052462.jpg b427a75d07b65b8c4f6c1f9e818a8e45bc25bd4fa604bed16411b3884c3d036e\\n000000052507.jpg 7e6a1ae4cf32d80d068033a9c3ceeec13fb835e90460687bc4cd458fe9abdb02\\n000000052565.jpg a1d362810f97e0dd00ecda4f1d427aec52ba3361c1a15f00cc525d9dc8216ad3\\n000000052591.jpg f137597498947e8bdc1f81e2d72972faff7bfda12e5910392019ffd5ffaea57b\\n000000052891.jpg e55b86a1cda874117ba1bd855e38dbc9b5427b840c362d72330019676ec47a44\\n000000052996.jpg 0be041c42db05b3e7a7074e5fc857a7b5d616697504048f751ea5025e856f406\\n000000053505.jpg ac6b9836a7f1383ed0bc1273664e104f1c2ec48e07506d2e88dfac9dd7150e5e\\n000000053529.jpg 560b4e0fc5844448f15891aa241443abc17b670698b3c56c4ae17db37a8cfdbc\\n000000053624.jpg 7dcb4159280778d8c16a81d08a1742b62bc71abb62aa456eb45e7179367a8a1b\\n000000053626.jpg b609e8bb0a6363d08b129b858886c083b6729d35fba713562334dfe0a1a409ec\\n000000053909.jpg 80df4444d544c27440621d44cc8726f9b49fe26f55550f0690861a0ec73c620e\\n000000053994.jpg 0f1eb7ded7f9443a89430ccdf4375f58ec10185f76afef4f4ff23f15c856780e\\n000000054123.jpg 66c735e467dca95c984fbabdd324686f14b4aa3ea9c6a719c0174fe4b4da2209\\n000000054164.jpg 32ce2bb44c2650a805d6282b852f1d5c312cec7df0f8296835bf71005acbe29f\\n000000054592.jpg 850a7dbd3351b791b86e7be0b8f24f461e12141d648b89bf1aff5f10135516f4\\n000000054593.jpg 2927085bfe4c7b09fc4cea90216f71beedcaed0ec501cdb06630bf40b126ec25\\n000000054605.jpg 7faf996ad9ff347643c1e94acadf2ff62c16d042344ef04567cdd3382bcccc75\\n000000054628.jpg 59a36063d39f73283c1825712303d6ebc5e208dd4708eaf98f974f7acf48b00a\\n000000054654.jpg e3e0a05408aa5e2a0a6554a51a4aa8c6707b149e97865ecd11a46be9ca6df16b\\n000000054931.jpg a49694d5dc982701f33a1d449a3ff7ebf953d1947ca141352097fb2528b10d1b\\n000000054967.jpg d40210c4e2241a0ac4de79f4cdc7a6c859dafb1b2b18469146f992c714fe9b2d\\n000000055002.jpg df75b3c0b09e519fa531f8bb57d27bb6eb86f0e2a951c47c8defe593746fef77\\n000000055022.jpg 183e2018079631fa23f89ce3d4f7d9ff0decaec41355a8348ed5dd5dfef33988\\n000000055072.jpg 9924357680ebc985df4e3335dfa7b3ea5e5621d32757d665ba485dff87d2c787\\n000000055150.jpg a3cc1286a509c3778a503994298b6cb5f7a9cd18133b531cf66564eb24604d8e\\n000000055167.jpg 370ff0c32b47f0200f7a0d2bfa507f2684d24a29fbfde06f75ed354910ac371b\\n000000055299.jpg 1edeecdf29069ce361a48149950ad5d3a819ed3e0678dc1f2a8659cbb164b11d\\n000000055528.jpg 284e74d6f4d0af3eb7441b3576ea4aa730861796ede9ec5ba60206a09ab25f20\\n000000055950.jpg 671beab7c211e59ba35ecac10c629fab690696ab64a5f0a107b8deb638ee96e8\\n000000056127.jpg 14c77b33e28f12794669c699498bed38dbe6f34a43868152d806529e97288196\\n000000056288.jpg 6f98dd8e6b6c0d301919aa847ddfc0b9356914219349ea67942b1afbebbd68f9\\n000000056344.jpg 2aedd846bfe767e86dc858282c286ff9bcd3f1303ef51dababc4bd459f5a5060\\n000000056350.jpg 811d5c94c05837766da90281d5a0ac773725f2dbb9009743906f686cea4e89e4\\n000000056545.jpg 852f95c0f7d5d72ec994fcead05a2a5501e7459966a360d92c26a5827584c744\\n"], "benchmark/vision/eval_map.py": ["fbb10642ed8e263edab8504a6d9f653812f545188e9f8c4af794de830ecb0833", "\\"\\"\\"COCO mAP50-95 and mAP50 of a YOLOv8n model file on the fixed val2017 subset (fetch_coco_subset.py), with\\nUltralytics\' own validator (square 640 letterbox, NMS, COCO-style metrics), batch 1, rect off, conf 0.001, IoU 0.7.\\n\\npython benchmark/vision/eval_map.py MODEL DATA_DIR --label trt-fp16 --precision fp16 --out results/raw/vision-accuracy\\nWrites <out>/<label>.json marked \\"workload\\": \\"vision-accuracy\\".\\n\\"\\"\\"\\n\\nimport argparse\\nimport datetime as dt\\nimport hashlib\\nimport json\\nimport tempfile\\nfrom pathlib import Path\\n\\nHERE = Path(__file__).parent\\n\\n\\ndef sha256(path):\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef data_yaml_dict(data, names):\\n    \\"\\"\\"Ultralytics dataset YAML for the subset. check_det_dataset (8.4.174) requires a train key even for val-only\\n    use, so train points at the same image list; nothing is trained.\\"\\"\\"\\n    return {\\"path\\": str(data), \\"train\\": \\"val.txt\\", \\"val\\": \\"val.txt\\", \\"names\\": names}\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\"model\\")\\n    p.add_argument(\\"data_dir\\")\\n    p.add_argument(\\"--label\\", required=True)\\n    p.add_argument(\\"--precision\\", required=True)\\n    p.add_argument(\\"--device\\", default=\\"0\\")\\n    p.add_argument(\\"--git-commit\\", default=None)\\n    p.add_argument(\\"--hardware\\", default=None)\\n    p.add_argument(\\"--out\\", required=True)\\n    args = p.parse_args()\\n\\n    import torch\\n    import ultralytics\\n    import yaml\\n    from ultralytics import YOLO\\n\\n    started = dt.datetime.now(dt.UTC).isoformat(timespec=\\"seconds\\")\\n    names = yaml.safe_load(\\n        (Path(ultralytics.__file__).parent / \\"cfg/datasets/coco.yaml\\").read_text()\\n    )[\\"names\\"]\\n    data = Path(args.data_dir).resolve()\\n    images = len((data / \\"val.txt\\").read_text().splitlines())\\n    work = Path(tempfile.mkdtemp())\\n    data_yaml = work / \\"coco-subset.yaml\\"\\n    data_yaml.write_text(yaml.safe_dump(data_yaml_dict(data, names)))\\n    val_args = {\\n        \\"imgsz\\": 640,\\n        \\"batch\\": 1,\\n        # Square 640x640 letterbox for every model. Model.val defaults rect=True and the validator only turns it\\n        # off for non-PyTorch static models, so without this PyTorch and dynamic ONNX would use rectangular\\n        # batches while the static TensorRT engine uses square ones, mixing preprocessing into the FP16 delta.\\n        \\"rect\\": False,\\n        \\"conf\\": 0.001,\\n        \\"iou\\": 0.7,\\n        \\"plots\\": False,\\n        \\"verbose\\": False,\\n    }\\n    r = YOLO(args.model, task=\\"detect\\").val(\\n        data=str(data_yaml),\\n        device=args.device,\\n        project=str(work),\\n        name=args.label,\\n        **val_args,\\n    )\\n    res = {\\n        \\"schema\\": 1,\\n        \\"workload\\": \\"vision-accuracy\\",\\n        \\"label\\": args.label,\\n        \\"model\\": Path(args.model).name,\\n        \\"model_sha256\\": sha256(args.model),\\n        \\"precision\\": args.precision,\\n        \\"started_utc\\": started,\\n        \\"finished_utc\\": dt.datetime.now(dt.UTC).isoformat(timespec=\\"seconds\\"),\\n        \\"dataset\\": {\\n            \\"name\\": \\"COCO val2017, first 500 sorted image names\\",\\n            \\"images\\": images,\\n            \\"manifest_sha256\\": sha256(HERE / \\"coco_val2017_first500.txt\\"),\\n        },\\n        \\"val_args\\": val_args,\\n        \\"env\\": {\\n            \\"hardware\\": args.hardware,\\n            \\"git_commit\\": args.git_commit,\\n            \\"ultralytics\\": ultralytics.__version__,\\n            \\"torch\\": torch.__version__,\\n        },\\n        \\"metrics\\": {k: float(v) for k, v in r.results_dict.items()},\\n        \\"speed_ms\\": r.speed,\\n    }\\n    out = Path(args.out)\\n    out.mkdir(parents=True, exist_ok=True)\\n    (out / f\\"{args.label}.json\\").write_text(\\n        json.dumps(res, indent=1) + \\"\\\\n\\", encoding=\\"utf-8\\", newline=\\"\\\\n\\"\\n    )\\n    print(args.label, json.dumps(res[\\"metrics\\"]))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"], "inference/vision/export.py": ["af8c0ac0e6bef7e4eca2f01fcf4a5a3760c0fbee61291c7f46cf0ddd1776c0fa", "\\"\\"\\"Export YOLOv8n (Ultralytics, AGPL-3.0) to an ONNX FP32 model and TensorRT FP16 engines, with sha256 of each.\\n\\n- ONNX: opset 17, dynamic batch and image size, onnxslim simplified, exported on the CPU (FP32). Opset 17 rather\\n  than Ultralytics\' default 18+: ONNX Runtime CUDA lacks some opset 19+ kernels (Ultralytics\' own note).\\n- TensorRT: one engine per static batch size (FP16 builder flag; inputs and outputs stay FP32), built on the GPU\\n  it will run on. Static shapes so each batch size gets tactics tuned for it. T4: FP16 only (no FP8, no bf16).\\n  Engine builds are not bit-reproducible (tactic timing), so each sha256 is recorded, not pinned.\\n\\npython export.py yolov8n.pt OUT_DIR --batches 1 8 32\\n\\"\\"\\"\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport shutil\\nfrom pathlib import Path\\n\\nOPSET = 17\\nIMGSZ = 640\\n\\n\\ndef sha256(path):\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\"weights\\")\\n    p.add_argument(\\"out_dir\\")\\n    p.add_argument(\\"--batches\\", type=int, nargs=\\"+\\", default=[1, 8, 32])\\n    args = p.parse_args()\\n\\n    import onnx\\n    import tensorrt\\n    import torch\\n    import ultralytics\\n    from ultralytics import YOLO\\n\\n    out = Path(args.out_dir)\\n    work = (\\n        out / \\"work\\"\\n    )  # Ultralytics writes exports next to the weights; the engine export reuses yolov8n.onnx\\n    work.mkdir(parents=True, exist_ok=True)\\n    src = work / \\"yolov8n.pt\\"\\n    shutil.copy2(args.weights, src)\\n    files = {}\\n\\n    f = YOLO(src).export(\\n        format=\\"onnx\\",\\n        opset=OPSET,\\n        dynamic=True,\\n        simplify=True,\\n        imgsz=IMGSZ,\\n        device=\\"cpu\\",\\n    )\\n    dst = out / f\\"yolov8n-fp32-opset{OPSET}.onnx\\"\\n    shutil.move(f, dst)\\n    files[dst.name] = {\\"sha256\\": sha256(dst), \\"precision\\": \\"fp32\\", \\"batch\\": \\"dynamic\\"}\\n\\n    for b in args.batches:\\n        f = YOLO(src).export(\\n            format=\\"engine\\",\\n            quantize=16,\\n            batch=b,\\n            dynamic=False,\\n            opset=OPSET,\\n            simplify=True,\\n            imgsz=IMGSZ,\\n            device=0,\\n        )\\n        dst = out / f\\"yolov8n-fp16-b{b}.engine\\"\\n        shutil.move(f, dst)\\n        (work / \\"yolov8n.onnx\\").unlink(\\n            missing_ok=True\\n        )  # FP32 intermediate of the engine build\\n        files[dst.name] = {\\"sha256\\": sha256(dst), \\"precision\\": \\"fp16\\", \\"batch\\": b}\\n\\n    (out / \\"exports.json\\").write_text(\\n        json.dumps(\\n            {\\n                \\"weights\\": Path(args.weights).name,\\n                \\"weights_sha256\\": sha256(args.weights),\\n                \\"opset\\": OPSET,\\n                \\"imgsz\\": IMGSZ,\\n                \\"versions\\": {\\n                    \\"ultralytics\\": ultralytics.__version__,\\n                    \\"torch\\": torch.__version__,\\n                    \\"onnx\\": onnx.__version__,\\n                    \\"tensorrt\\": tensorrt.__version__,\\n                },\\n                \\"files\\": files,\\n            },\\n            indent=1,\\n        )\\n        + \\"\\\\n\\"\\n    )\\n    shutil.rmtree(work)\\n    print(json.dumps(files, indent=1))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"], "inference/vision/triton_serve.py": ["348da9778e3d6f1f743d2386116e9790862c4960fec40db9a4a495130a74cf30", "\\"\\"\\"Serve YOLOv8n TensorRT FP16 engines through Triton Inference Server on Kaggle, without Docker.\\n\\nThe NGC tritonserver image cannot run on Kaggle (no Docker), and the PyPI `tritonserver` wheel is only the\\nin-process Python API: it needs libtritonserver.so from that image. PyTriton (nvidia-pytriton, Apache-2.0) ships a\\ntritonserver binary with the Python backend only, so Triton here serves a Python callable that runs the TensorRT\\nengine (TrtModel from benchmark/vision/run_vision.py); Triton\'s own TensorRT and ONNX Runtime backends are not\\nused. Clients get Triton\'s HTTP/gRPC KServe v2 API and Prometheus metrics on port 8002.\\n\\nOne model per static-batch engine: yolov8n_b<N>, max_batch_size N, so a request of N images runs the engine built\\nfor N (dynamic batching cannot form larger batches than N).\\n\\npython triton_serve.py yolov8n-fp16-b1.engine yolov8n-fp16-b8.engine ... --log-file results/logs/triton.log\\n\\"\\"\\"\\n\\nimport argparse\\nimport os\\nimport re\\nimport sys\\nfrom pathlib import Path\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[2] / \\"benchmark\\" / \\"vision\\"))\\n\\n\\ndef use_own_libpython():\\n    \\"\\"\\"Make PyTriton\'s Python backend stub load this interpreter\'s libpython3.12.so.\\n\\n    The stub links libpython3.12.so.1.0 by name. With a uv-managed (python-build-standalone) Python, the loader\\n    either finds no libpython (error at stub start) or, on Kaggle, the image\'s Ubuntu libpython3.12, whose stdlib\\n    extension layout differs: \\"ModuleNotFoundError: No module named \'_contextvars\'\\" (Kaggle 2026-10-10, v3c). PyTriton\\n    copies os.environ into tritonserver\'s environment and only appends its own libs to LD_LIBRARY_PATH, so putting\\n    sys.base_prefix/lib first makes the stub use the same Python as this process. Reproduced and verified in\\n    ubuntu:24.04 with a CPU model (2026-10-10).\\"\\"\\"\\n    lib = str(Path(sys.base_prefix) / \\"lib\\")\\n    old = os.environ.get(\\"LD_LIBRARY_PATH\\")\\n    os.environ[\\"LD_LIBRARY_PATH\\"] = lib + (f\\":{old}\\" if old else \\"\\")\\n    return lib\\n\\n\\ndef main():\\n    p = argparse.ArgumentParser(\\n        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter\\n    )\\n    p.add_argument(\\n        \\"engines\\", nargs=\\"+\\", help=\\"Ultralytics engine files named ...-b<N>.engine\\"\\n    )\\n    p.add_argument(\\"--log-file\\", default=None)\\n    args = p.parse_args()\\n\\n    import numpy as np\\n    from pytriton.decorators import batch\\n    from pytriton.model_config import ModelConfig, Tensor\\n    from pytriton.triton import Triton, TritonConfig\\n    from run_vision import TrtModel\\n\\n    def make_infer(model):\\n        @batch\\n        def infer(**inputs):\\n            (x,) = inputs.values()\\n            return {\\n                o: y.astype(np.float32, copy=False)\\n                for o, y in zip(model.outs, model(x), strict=True)\\n            }\\n\\n        return infer\\n\\n    print(\\"libpython for the Python backend stub:\\", use_own_libpython(), flush=True)\\n    config = TritonConfig(\\n        http_port=8000, grpc_port=8001, metrics_port=8002, log_file=args.log_file\\n    )\\n    with Triton(config=config) as triton:\\n        for path in args.engines:\\n            n = int(re.search(r\\"-b(\\\\d+)\\\\.engine$\\", path).group(1))\\n            model = TrtModel(path)\\n            shapes = {k: v[1] for k, v in model.info[\\"io\\"].items()}\\n            triton.bind(\\n                model_name=f\\"yolov8n_b{n}\\",\\n                infer_func=make_infer(model),\\n                inputs=[\\n                    Tensor(\\n                        name=model.inp,\\n                        dtype=np.float32,\\n                        shape=tuple(shapes[model.inp][1:]),\\n                    )\\n                ],\\n                outputs=[\\n                    Tensor(name=o, dtype=np.float32, shape=tuple(shapes[o][1:]))\\n                    for o in model.outs\\n                ],\\n                config=ModelConfig(max_batch_size=n),\\n            )\\n        triton.serve()\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"]}'
)
for rel, (sha, text) in FILES.items():
    assert hashlib.sha256(text.encode()).hexdigest() == sha, rel
    p = pathlib.Path(BENCH) / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
    if rel.endswith(".sh"):
        os.chmod(p, 0o755)
print("wrote", len(FILES), "files to", BENCH)

In [ ]:
# Hardware and base image capture, before installing anything.
import json
import os
import pathlib
import shlex
import subprocess
import sys
import time

for d in ("env", "logs", "raw", "server_args"):
    pathlib.Path(RESULTS, d).mkdir(parents=True, exist_ok=True)


def sh(cmd, out=None, check=False, timeout=None):
    """Run a shell command, save stdout+stderr to RESULTS/out, raise on non-zero exit if check. With timeout (s),
    coreutils timeout kills the command's whole process group, so a hang ends in exit 124 instead of the session."""
    if timeout:
        cmd = f"timeout -k 60 {int(timeout)} bash -c {shlex.quote(cmd)}"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)
    text = r.stdout + r.stderr
    if out:
        pathlib.Path(RESULTS, out).write_text(text)
    if check and r.returncode != 0:
        why = (
            f" (timed out after {timeout} s)" if timeout and r.returncode == 124 else ""
        )
        raise RuntimeError(f"exit {r.returncode}{why}: {cmd[:300]}\n{text[-3000:]}")
    return text


sh("nvidia-smi -q", "env/nvidia-smi-q.txt")
print(sh("nvidia-smi", "env/nvidia-smi.txt"))
GPU_COUNT = len(
    [ln for ln in sh("nvidia-smi -L").splitlines() if ln.startswith("GPU ")]
)
sh("lscpu", "env/lscpu.txt")
sh("free -m", "env/free.txt")
sh("cat /etc/os-release", "env/os-release.txt")
sh(f"{sys.executable} -m pip freeze", "env/kaggle-base-pip-freeze.txt")
pathlib.Path(RESULTS, "env/run.json").write_text(
    json.dumps(
        {
            "git_commit": GIT_COMMIT,
            "gpu_count": GPU_COUNT,
            **PINS,
            "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "kaggle_docker_image": os.environ.get("KAGGLE_DOCKER_IMAGE"),
            "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
        },
        indent=1,
    )
)
print("GPU_COUNT =", GPU_COUNT)

In [ ]:
# Preflight: fail fast instead of burning the session. Run v1 (2026-10-07) had no GPU and no internet.
import shutil
import urllib.request

problems = []
if not shutil.which("nvidia-smi"):
    problems.append("nvidia-smi not found")
elif GPU_COUNT == 0:
    problems.append("nvidia-smi reports 0 GPUs")
for url in PREFLIGHT_URLS:
    try:
        urllib.request.urlopen(url, timeout=10)
    except OSError as e:
        problems.append(f"{url} not reachable: {e}")
if problems:
    msg = "no GPU/internet: check Kaggle phone verification and that the accelerator is GPU T4 x2 and Internet is on"
    pathlib.Path(RESULTS, "FAILED-preflight.txt").write_text(
        msg + "\n" + "\n".join(problems) + "\n"
    )
    raise RuntimeError(msg + " (" + "; ".join(problems) + ")")
print(f"preflight OK: {GPU_COUNT} GPU(s), reachable: {', '.join(PREFLIGHT_URLS)}")

In [ ]:
import signal
import traceback
import urllib.request


def wait_ready(proc, timeout=1200):
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            raise RuntimeError(f"server exited with code {proc.returncode}")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
            return time.time() - t
        except OSError:
            time.sleep(5)
    raise TimeoutError("server not ready")


def start_server(cell, model_key, tp=1, extra=()):
    repo, rev = MODELS[model_key]
    args = [
        f"{VENVS}/vllm/bin/vllm",
        "serve",
        repo,
        "--revision",
        rev,
        "--tokenizer-revision",
        rev,
        "--served-model-name",
        repo,
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
        "--dtype",
        "float16",
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEM_UTIL),
        "--attention-backend",
        "TRITON_ATTN",
        "--no-enable-prefix-caching",
        "--seed",
        "0",
        # vLLM defaults: without this, the model's generation_config.json (repetition_penalty 1.1) applies to
        # every request that does not set it (seen in run v2, 2026-10-09).
        "--generation-config",
        "vllm",
        "--tensor-parallel-size",
        str(tp),
        *extra,
    ]
    cmd = shlex.join(args)
    pathlib.Path(RESULTS, "server_args", f"{cell}.txt").write_text(cmd + "\n")
    log = open(pathlib.Path(RESULTS, "logs", f"server-{cell}.log"), "w")  # noqa: SIM115 open for the server's lifetime
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=",".join(str(i) for i in range(tp)))
    proc = subprocess.Popen(
        args, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True
    )
    try:
        print(f"{cell}: server ready in {wait_ready(proc):.0f}s")
    except Exception:
        stop_server(proc, cell)
        raise
    return proc, cmd


def gpu_mem_used():
    out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits")
    return [int(x) for x in out.split() if x.isdigit()]


def stop_server(proc, cell, timeout=180):
    """SIGTERM the server's process group, then SIGKILL it (EngineCore and TP workers live in the same group),
    then wait until every GPU is under 1 GiB used so the next cell starts on an empty GPU."""
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(120)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        pass
    try:
        os.killpg(proc.pid, signal.SIGKILL)  # leftover workers, if any
    except ProcessLookupError:
        pass
    t, used = time.time(), gpu_mem_used()
    while any(u >= 1024 for u in used) and time.time() - t < timeout:
        time.sleep(5)
        used = gpu_mem_used()
    msg = f"{cell}: GPU memory.used MiB after stop {used} ({time.time() - t:.0f}s)"
    pathlib.Path(RESULTS, "logs", f"gpu-drain-{cell}.txt").write_text(msg + "\n")
    print(msg if all(u < 1024 for u in used) else "WARNING " + msg)


def loadgen(
    cell,
    model_key,
    precision,
    concurrency,
    cmd,
    gpus,
    extra="",
    env_prefix="",
    engine="vllm",
    engine_version=None,
):
    repo, rev = MODELS[model_key]
    c = " ".join(str(x) for x in concurrency)
    gpu_ids = ",".join(
        str(i) for i in range(gpus)
    )  # nvidia-smi sampler watches only the server's GPUs
    out = sh(
        f"{env_prefix} CUDA_VISIBLE_DEVICES={gpu_ids} {VENVS}/bench/bin/python {BENCH}/benchmark/loadgen/run.py --base-url http://127.0.0.1:{PORT}/v1 "
        f"--model {repo} --model-revision {rev} --engine {engine} --engine-version {engine_version or VLLM_VERSION} --precision {precision} "
        f"--label {cell} --hardware '{gpus}x Tesla T4 (Kaggle)' --concurrency {c} --repeats {REPEATS} "
        f"--metrics-url http://127.0.0.1:{PORT}/metrics --server-args {shlex.quote(cmd)} --git-commit {GIT_COMMIT} "
        f"--out-dir {RESULTS}/raw/{cell} {extra}",
        f"logs/loadgen-{cell}.log",
        check=True,
        timeout=3600,
    )
    print(out[-3000:])


def gsm8k(label, model_key):
    repo, _ = MODELS[model_key]
    out = sh(
        f"PATH={VENVS}/bench/bin:$PATH bash {BENCH}/benchmark/accuracy/run_gsm8k.sh "
        f"http://127.0.0.1:{PORT}/v1/chat/completions {repo} {RESULTS}/raw/accuracy/{label} {GSM8K_LIMIT} 16",
        check=True,
        timeout=3600,
    )
    print(out[-1500:])


def run_cell(
    cell,
    model_key,
    precision,
    concurrency,
    tp=1,
    accuracy_label=None,
    loadgen_extra="",
    loadgen_env="",
):
    t, proc = time.time(), None
    try:
        proc, cmd = start_server(cell, model_key, tp)
        loadgen(
            cell,
            model_key,
            precision,
            concurrency,
            cmd,
            tp,
            extra=loadgen_extra,
            env_prefix=loadgen_env,
        )
        if accuracy_label:
            gsm8k(accuracy_label, model_key)
    except Exception:  # noqa: BLE001 record and move on to the next cell
        pathlib.Path(RESULTS, f"FAILED-{cell}.txt").write_text(traceback.format_exc())
        print(f"{cell} FAILED, see results/FAILED-{cell}.txt")
    finally:
        if proc:
            stop_server(proc, cell)
    pathlib.Path(RESULTS, "env", f"timing-{cell}.txt").write_text(
        f"{time.time() - t:.0f} s wall clock\n"
    )
    print(f"{cell}: {time.time() - t:.0f}s")

In [ ]:
# Step runner. Every unit of work is one step(): it is skipped with NOT_RUN-<name>.txt when the remaining session
# budget cannot cover its estimate, records FAILED-<name>.txt with the traceback on any exception (the notebook
# moves on), and writes env/timing-<name>.txt. Returns fn's value (True for None) on success, None otherwise.
import hashlib
import urllib.request

T0 = time.time()


def time_left():
    return BUDGET_S - (time.time() - T0)


def not_run(name, reason):
    pathlib.Path(RESULTS, f"NOT_RUN-{name}.txt").write_text(f"not run: {reason}\n")
    print(name, "not run:", reason)


def step(name, est_s, fn, *args, **kw):
    if time_left() < est_s:
        not_run(
            name,
            f"time budget: {time_left():.0f} s left of {BUDGET_S} s, step estimate {est_s} s",
        )
        return None
    t = time.time()
    try:
        res = fn(*args, **kw)
        return True if res is None else res
    except Exception:  # noqa: BLE001 record and move on
        pathlib.Path(RESULTS, f"FAILED-{name}.txt").write_text(traceback.format_exc())
        print(f"{name} FAILED, see results/FAILED-{name}.txt")
        return None
    finally:
        pathlib.Path(RESULTS, "env", f"timing-{name}.txt").write_text(
            f"{time.time() - t:.0f} s wall clock\n"
        )
        print(f"{name}: {time.time() - t:.0f}s")


def fetch(url, sha, dest):
    """Download url to dest and check its sha256 (raises on mismatch)."""
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url, timeout=300) as r:
        data = r.read()
    got = hashlib.sha256(data).hexdigest()
    if got != sha:
        raise ValueError(f"{url}: sha256 {got} != pinned {sha}")
    dest.write_bytes(data)
    return dest


def install_vllm():
    """vLLM 0.31.0 venv from the run v2 lock (notebooks/vllm-requirements.txt)."""
    log = sh(
        f"uv venv -q --python {PYTHON_VERSION} {VENVS}/vllm && uv pip install -q --python {VENVS}/vllm/bin/python "
        f"--torch-backend cu129 -r {BENCH}/notebooks/vllm-requirements.txt",
        "logs/install-vllm.log",
        check=True,
        timeout=2400,
    )
    print(log[-1500:])
    sh(f"uv pip freeze --python {VENVS}/vllm/bin/python", "env/pip-freeze-vllm.txt")


def start_proc(cell, args, env, ready_url, timeout=900):
    """Start a server in its own process group, log to logs/server-<cell>.log, wait until ready_url answers 200."""
    cmd = shlex.join(args)
    pathlib.Path(RESULTS, "server_args", f"{cell}.txt").write_text(cmd + "\n")
    log = open(pathlib.Path(RESULTS, "logs", f"server-{cell}.log"), "w")  # noqa: SIM115 open for the server's lifetime
    proc = subprocess.Popen(
        args, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True
    )
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            raise RuntimeError(
                f"{cell}: server exited with code {proc.returncode}, see logs/server-{cell}.log"
            )
        try:
            urllib.request.urlopen(ready_url, timeout=5)
            print(f"{cell}: ready in {time.time() - t:.0f}s")
            return proc, cmd
        except OSError:
            time.sleep(5)
    stop_server(proc, cell)
    raise TimeoutError(f"{cell}: not ready after {timeout}s")

## Install the client venv (estimate 2 min)

In [ ]:
# uv and the client venv (loadgen, COCO fetch, lm-eval). Other venvs are installed by the steps that need them.
def install_base():
    print(
        sh(
            f"pip install -q uv=={UV_VERSION} && uv --version",
            "logs/install-uv.log",
            check=True,
            timeout=600,
        )
    )
    sh(
        f"uv venv -q --python {PYTHON_VERSION} {VENVS}/bench && "
        f"uv pip install -q --python {VENVS}/bench/bin/python -r {BENCH}/benchmark/requirements.txt",
        "logs/install-bench.log",
        check=True,
        timeout=1200,
    )
    sh(f"uv pip freeze --python {VENVS}/bench/bin/python", "env/pip-freeze-bench.txt")


BASE_OK = step("install-base", 600, install_base)

## A. DCGM: not run again (settled by the 2026-10-10 session)

In [ ]:
# DCGM is not run again: profiling needs root over the host engine, which Kaggle does not give.
reason = (
    "DCGM profiling fails on Kaggle for every field with 'DCGMError_RequiresRoot: This operation is not supported "
    f"when the host engine is running as non root'; basic fields worked. Evidence: {DCGM_EVIDENCE}"
)
not_run("dcgm", reason)
not_run("vllm-t4-1.5b-fp16-dcgm", reason)

## B. Vision: YOLOv8n ONNX Runtime CUDA, TensorRT FP16, Triton, mAP (estimate 45 min)

In [ ]:
# Vision: venv, data, exports, then the benchmarks and mAP. One step each, so one failure does not stop the rest.
VPY = f"{VISION_ENV} {VENVS}/vision/bin/python"
WEIGHTS = "/tmp/data/yolov8n.pt"
ONNX = f"{MODELS_DIR}/yolov8n-fp32-opset17.onnx"
ENGINE = {b: f"{MODELS_DIR}/yolov8n-fp16-b{b}.engine" for b in VISION_BATCHES}
HW = "'1x Tesla T4 (Kaggle)'"


def vision_install():
    print(
        sh(
            f"uv venv -q --python {PYTHON_VERSION} {VENVS}/vision && uv pip install -q --python {VENVS}/vision/bin/python "
            "--torch-backend cu129 --extra-index-url https://pypi.nvidia.com --index-strategy unsafe-best-match "
            f"-r {BENCH}/notebooks/vision-requirements.txt",
            "logs/install-vision.log",
            check=True,
            timeout=1800,
        )[-1500:]
    )
    sh(f"uv pip freeze --python {VENVS}/vision/bin/python", "env/pip-freeze-vision.txt")
    print(
        sh(
            f"{VPY} -c 'import torch, tensorrt, onnxruntime, ultralytics; print(torch.__version__, torch.version.cuda, "
            "torch.cuda.get_device_capability(0), tensorrt.__version__, onnxruntime.__version__, "
            "onnxruntime.get_available_providers(), ultralytics.__version__)'",
            "env/vision-versions.txt",
            check=True,
        )
    )


def vision_data():
    print(
        sh(
            f"{VENVS}/bench/bin/python {BENCH}/benchmark/vision/fetch_coco_subset.py --out {DATA}",
            "logs/fetch-coco.log",
            check=True,
            timeout=1200,
        )
    )
    fetch(YOLO_URL, YOLO_SHA256, WEIGHTS)


def vision_export():
    out = sh(
        f"{VPY} {BENCH}/inference/vision/export.py {WEIGHTS} {MODELS_DIR} --batches {' '.join(map(str, VISION_BATCHES))}",
        "logs/vision-export.log",
        check=True,
        timeout=3000,
    )
    print(out[-3000:])


def vision_bench(label, backend, model, precision, batches, extra="", log=None):
    b = " ".join(map(str, batches))
    out = sh(
        f"{VPY} {BENCH}/benchmark/vision/run_vision.py --backend {backend} --model {model} --images {DATA}/images/val2017 "
        f"--batch {b} --repeats {REPEATS} --label {label} --precision {precision} --hardware {HW} "
        f"--git-commit {GIT_COMMIT} --out-dir {RESULTS}/raw/{label} {extra}",
        log or f"logs/{label}.log",
        check=True,
        timeout=1200,
    )
    print(out[-2000:])


def vision_trt():
    for b in VISION_BATCHES:
        vision_bench(
            "vision-t4-yolov8n-trt-fp16",
            "trt",
            ENGINE[b],
            "fp16",
            [b],
            log=f"logs/vision-t4-yolov8n-trt-fp16-b{b}.log",
        )


def vision_triton():
    args = [
        f"{VENVS}/vision/bin/python",
        f"{BENCH}/inference/vision/triton_serve.py",
        *ENGINE.values(),
        "--log-file",
        f"{RESULTS}/logs/tritonserver.log",
    ]
    proc, cmd = start_proc(
        "vision-triton",
        args,
        dict(os.environ, CUDA_VISIBLE_DEVICES="0"),
        "http://127.0.0.1:8000/v2/health/ready",
        timeout=600,
    )
    try:
        for b in VISION_BATCHES:
            vision_bench(
                "vision-t4-yolov8n-trt-fp16-triton",
                "triton",
                f"yolov8n_b{b}",
                "fp16",
                [b],
                extra=f"--model-file {ENGINE[b]} --triton-url 127.0.0.1:8000 "
                f"--metrics-url http://127.0.0.1:8002/metrics --server-args {shlex.quote(cmd)}",
                log=f"logs/vision-t4-yolov8n-trt-fp16-triton-b{b}.log",
            )
    finally:
        stop_server(proc, "vision-triton")


def vision_map(label, model, precision):
    out = sh(
        f"{VPY} {BENCH}/benchmark/vision/eval_map.py {model} {DATA} --label {label} --precision {precision} "
        f"--device 0 --git-commit {GIT_COMMIT} --hardware {HW} --out {RESULTS}/raw/vision-accuracy",
        f"logs/map-{label}.log",
        check=True,
        timeout=800,
    )
    print(out[-1500:])


if (
    BASE_OK
    and step("vision-install", 900, vision_install)
    and step("vision-data", 600, vision_data)
    and step("vision-export", 1500, vision_export)
):
    if RUN_SPEED:
        step(
            "vision-t4-yolov8n-onnx-fp32-ortcuda",
            600,
            vision_bench,
            "vision-t4-yolov8n-onnx-fp32-ortcuda",
            "ort-cuda",
            ONNX,
            "fp32",
            VISION_BATCHES,
        )
        step("vision-t4-yolov8n-trt-fp16", 600, vision_trt)
    else:
        for name in (
            "vision-t4-yolov8n-onnx-fp32-ortcuda",
            "vision-t4-yolov8n-trt-fp16",
        ):
            not_run(name, f"RUN_SPEED is False: already measured, see {SPEED_EVIDENCE}")
    if RUN_TRITON:
        step(
            "vision-triton", 900, vision_triton
        )  # FAILED-vision-triton.txt + logs/server-vision-triton.log if Triton cannot run
    else:
        not_run(
            "vision-triton",
            f"RUN_TRITON is False: already measured, see {TRITON_EVIDENCE}",
        )
    for label, model, precision in (
        ("pt-fp32", WEIGHTS, "fp32"),
        ("onnx-fp32", ONNX, "fp32"),
        ("trt-fp16", ENGINE[1], "fp16"),
    ):
        step(f"map-{label}", 400, vision_map, label, model, precision)

## Summary

In [ ]:
# Quick look. The real report is benchmark/report.py, run offline after downloading the results.
# Wrapped like a step: nothing in the summary may raise, so the end-of-session files are always written.
import glob


def summary():
    for f in sorted(glob.glob(f"{RESULTS}/raw/**/*.json", recursive=True)):
        try:
            d = json.loads(pathlib.Path(f).read_text())
            print(
                os.path.relpath(f, RESULTS),
                json.dumps(d.get("summary") or d.get("metrics"))[:300],
            )
        except (OSError, ValueError, AttributeError) as e:
            print(f, "unreadable:", e)
    print(
        sh(f"ls {RESULTS}/FAILED-* {RESULTS}/NOT_RUN-* 2>/dev/null")
        or "no failed or skipped steps"
    )


step("summary", 0, summary)
sh(
    "nvidia-smi -q", "env/nvidia-smi-q-end.txt"
)  # cumulative clock event reason counters over the session
pathlib.Path(RESULTS, "env/finished_utc.txt").write_text(
    time.strftime("%Y-%m-%dT%H:%M:%SZ\n", time.gmtime())
)
print(f"session used {time.time() - T0:.0f} s of the {BUDGET_S} s budget")